# MASLD research workflow — cleaned, annotated research archive

**Status: structurally reviewed; NOT executed or end-to-end validated.** This is a preserved, chronological research record, not a single runnable pipeline. It is not appropriate to claim complete reproducibility from this notebook alone.

## How to use this notebook

- **Do not select Run all.** Cells reflect multiple development iterations, hard-coded Colab/Drive locations, downloads, environment mutations, and exploratory data repairs. Execute only the relevant stage after reviewing its prerequisites.
- The original code and cell order are preserved. Original cell numbers are recorded in cell metadata (`masld_original_cell_index`), and each stage is introduced by a heading.
- Two incomplete Python cells were quarantined into Markdown code blocks to keep this notebook syntactically sound without inventing missing research logic.
- Several steps load or write external scripts, data, and `.joblib` models. Those are **not bundled here**. Never unpickle an untrusted `.joblib` file.
- The reconstructed V4 candidate is **not** the original frozen V4 model, and no independent validation is established by this cleaned copy.

## Reproducibility gate

To claim an end-to-end reproducible workflow, provide versioned input data or a documented lawful acquisition procedure, the pipeline scripts and pinned environment, clear split/label/feature procedures, a clean-session execution log, saved model hash and provenance, and independent evaluation artifacts. See the accompanying validation report.


## 01 — Google Drive and original project setup


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

drive_path = "/content/drive/MyDrive"

print("Files and folders in Google Drive:\n")

for item in os.listdir(drive_path):
    full_path = os.path.join(drive_path, item)

    if os.path.isdir(full_path):
        print("FOLDER:", item)
    else:
        print("FILE:", item)

In [ ]:
import os

drive_path = "/content/drive/MyDrive"

print("Searching for ZIP files...\n")

zip_files = []

for root, dirs, files in os.walk(drive_path):
    for file in files:
        if file.lower().endswith(".zip"):
            full_path = os.path.join(root, file)
            zip_files.append(full_path)
            print(full_path)

print("\nTotal ZIP files found:", len(zip_files))

In [ ]:
import os
import zipfile

zip_path = "/content/drive/MyDrive/ML_Project-main (1)/ML_Project-main(1).zip"

print("File exists:", os.path.isfile(zip_path))

In [ ]:
file_size = os.path.getsize(zip_path)

print("File size in bytes:", file_size)
print("File size in MB:", round(file_size / (1024 * 1024), 3))
print("Is valid ZIP:", zipfile.is_zipfile(zip_path))

In [ ]:
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    files = zip_ref.namelist()

    print("Total entries:", len(files))
    print("\nFiles inside ZIP:\n")

    for file in files[:50]:
        print(file)

    python_files = [
        file for file in files
        if file.endswith(".py")
    ]

    print("\nTotal Python files:", len(python_files))

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/ML_Project-main (1)/ML_Project-main(1).zip"

extract_path = "/content/ML_Project"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    bad_file = zip_ref.testzip()
    if bad_file is not None:
        raise ValueError(f"Corrupted ZIP member: {bad_file}")

    zip_ref.extractall(extract_path)

print("Project extracted successfully!")
print("Extracted folders:", os.listdir(extract_path))

In [ ]:
project_path = "/content/ML_Project/ML_Project-main"

os.chdir(project_path)

print("Current directory:", os.getcwd())
print("\nProject contents:")

for item in os.listdir(project_path):
    print(item)

In [ ]:
models_path = os.path.join(project_path, "data", "models")

print("Trained models:\n")

for model in os.listdir(models_path):
    model_path = os.path.join(models_path, model)

    print(
        model,
        "| Size:",
        round(os.path.getsize(model_path) / 1024, 2),
        "KB"
    )

In [ ]:
print("PROJECT STRUCTURE\n")

for root, dirs, files in os.walk(project_path):
    level = root.replace(project_path, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}  {file}")

In [ ]:
print("MAIN PROJECT FILES:\n")

for file in os.listdir(project_path):
    if file.endswith((".py", ".txt", ".yaml", ".yml", ".toml")):
        print(file)

print("\nRequirements file exists:",
      os.path.exists(os.path.join(project_path, "requirements.txt")))

print("Pipeline exists:",
      os.path.exists(os.path.join(project_path, "run_pipeline.py")))

In [ ]:
import os

models_path = os.path.join(project_path, "data", "models")

if not os.path.isdir(models_path):
    raise FileNotFoundError("Models folder not found!")

print("TRAINED MODELS:\n")

for filename in sorted(os.listdir(models_path)):
    file_path = os.path.join(models_path, filename)

    if os.path.isfile(file_path):
        print(
            filename,
            "| Size:",
            round(os.path.getsize(file_path) / 1024, 2),
            "KB"
        )

In [ ]:
requirements_path = os.path.join(
    project_path, "requirements.txt"
)

with open(requirements_path, "r") as f:
    print(f.read())

In [ ]:
pipeline_path = os.path.join(
    project_path, "run_pipeline.py"
)

with open(pipeline_path, "r", encoding="utf-8") as f:
    pipeline_code = f.read()

print(pipeline_code[:12000])

In [ ]:
%cd /content/ML_Project/ML_Project-main

%pip install -r requirements.txt

In [ ]:
import numpy
import pandas
import scipy
import sklearn
import xgboost
import shap
import imblearn
import GEOparse

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)
print("SHAP:", shap.__version__)

print("\nAll required ML libraries imported successfully!")

In [ ]:
import os

project_path = "/content/ML_Project/ML_Project-main"
models_path = os.path.join(project_path, "data", "models")

for filename in sorted(os.listdir(models_path)):
    filepath = os.path.join(models_path, filename)

    if filename.endswith(".pkl"):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"{filename}: {size_mb:.3f} MB")

## 02 — External-validation pipeline setup


In [ ]:
import os

drive_path = "/content/drive/MyDrive"

print("Searching for ZIP files in Google Drive...\n")

for root, dirs, files in os.walk(drive_path):
    for file in files:
        if file.lower().endswith(".zip"):
            full_path = os.path.join(root, file)

            print("ZIP FILE:", file)
            print("PATH:", full_path)
            print("SIZE (MB):",
                  round(os.path.getsize(full_path) / (1024 * 1024), 3))
            print("-" * 60)

In [ ]:
import os
import zipfile

zip_path = "/content/drive/MyDrive/ML_Project-main (1)/MASLD_external_validation_pipeline (1).zip"

print("File exists:", os.path.isfile(zip_path))
print("Is valid ZIP:", zipfile.is_zipfile(zip_path))

In [ ]:
import os
import zipfile

extract_path = "/content/MASLD_New_Pipeline"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    bad_file = zip_ref.testzip()

    if bad_file is not None:
        raise ValueError(f"Corrupted file: {bad_file}")

    destination = os.path.realpath(extract_path)

    for member in zip_ref.infolist():
        member_path = os.path.realpath(
            os.path.join(destination, member.filename)
        )

        if os.path.commonpath([destination, member_path]) != destination:
            raise ValueError(f"Unsafe ZIP path: {member.filename}")

    zip_ref.extractall(extract_path)

print("SUCCESS: New pipeline extracted!")

In [ ]:
import os

print("NEW PIPELINE FILES:\n")

for root, dirs, files in os.walk(extract_path):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
models_path = "/content/ML_Project/ML_Project-main/data/models"

if os.path.isdir(models_path):
    print("Original trained models:\n")

    for file in sorted(os.listdir(models_path)):
        if file.endswith(".pkl"):
            print(file)
else:
    print("Original project not found in this runtime.")

In [ ]:
readme_path = "/content/MASLD_New_Pipeline/README_EXTERNAL_VALIDATION.md"

with open(readme_path, "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
requirements_path = "/content/MASLD_New_Pipeline/requirements_external_validation.txt"

with open(requirements_path, "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
pipeline_path = "/content/MASLD_New_Pipeline/external_validation_pipeline.py"

with open(pipeline_path, "r", encoding="utf-8") as f:
    pipeline_code = f.read()

print("Pipeline length:", len(pipeline_code), "characters")
print("\n===== PIPELINE CODE =====\n")
print(pipeline_code)

In [ ]:
import os

models_path = "/content/ML_Project/ML_Project-main/data/models"

if not os.path.isdir(models_path):
    print("ERROR: Original trained models folder not found.")
else:
    models = sorted(
        f for f in os.listdir(models_path)
        if f.endswith(".pkl")
    )

    print("Total trained models:", len(models))

    for model in models:
        print(model)

    if len(models) == 8:
        print("\nSUCCESS: All eight trained model files found!")

In [ ]:
import os
import shutil

original_project = "/content/ML_Project/ML_Project-main"
new_pipeline = "/content/MASLD_New_Pipeline"

files_to_copy = [
    "external_validation_pipeline.py",
    "requirements_external_validation.txt",
    "README_EXTERNAL_VALIDATION.md"
]

for filename in files_to_copy:
    source = os.path.join(new_pipeline, filename)
    destination = os.path.join(original_project, filename)

    if not os.path.isfile(source):
        raise FileNotFoundError(f"Missing: {source}")

    shutil.copy2(source, destination)
    print("Copied:", filename)

print("\nNew pipeline successfully added to project!")

In [ ]:
%cd /content/ML_Project/ML_Project-main

In [ ]:
import os

required_files = [
    "external_validation_pipeline.py",
    "requirements_external_validation.txt",
    "data/raw/GSE48452/GSE48452_expression.csv",
    "data/raw/GSE48452/GSE48452_metadata.csv"
]

for file in required_files:
    exists = os.path.isfile(file)
    print(("FOUND" if exists else "MISSING"), file)

    if not exists:
        raise FileNotFoundError(file)

In [ ]:
import sys

print("Python version:", sys.version)

In [ ]:
%pip install -r requirements_external_validation.txt

In [ ]:
import os

project_path = "/content/ML_Project/ML_Project-main"

print("Project exists:", os.path.isdir(project_path))

print("New pipeline exists:", os.path.isfile(
    project_path + "/external_validation_pipeline.py"
))

print("Original data exists:", os.path.isdir(
    project_path + "/data/raw/GSE48452"
))

In [ ]:
import sys
import sklearn
import numpy
import pandas
import xgboost
import imblearn

print("Python:", sys.version.split()[0])
print("Scikit-learn:", sklearn.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("XGBoost:", xgboost.__version__)
print("Imbalanced-learn:", imblearn.__version__)

In [ ]:
%cd /content/ML_Project/ML_Project-main

## 03 — Acquisition, preparation and first training


In [ ]:
!python external_validation_pipeline.py --stage preflight

In [ ]:
%cd /content/ML_Project/ML_Project-main

In [ ]:
!python external_validation_pipeline.py --stage acquire

In [ ]:
import os

raw_path = "/content/ML_Project/ML_Project-main/external_validation_v2/raw"

for root, dirs, files in os.walk(raw_path):
    for file in files:
        path = os.path.join(root, file)
        size_mb = os.path.getsize(path) / (1024 * 1024)

        print(f"{file}: {size_mb:.2f} MB")

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount("/content/drive")

source = "/content/ML_Project/ML_Project-main/external_validation_v2/raw"

backup = "/content/drive/MyDrive/MASLD_Validation_Backup/raw"

if not os.path.isdir(source):
    raise FileNotFoundError("Acquisition folder not found.")

shutil.copytree(source, backup, dirs_exist_ok=True)

print("Raw data backup completed!")

In [ ]:
!python external_validation_pipeline.py --stage acquire

In [ ]:
%cd /content/ML_Project/ML_Project-main

In [ ]:
!python external_validation_pipeline.py --stage prepare

In [ ]:
import pandas as pd
import gzip
import os

project_path = "/content/ML_Project/ML_Project-main"

counts_path = (
    project_path +
    "/external_validation_v2/raw/GSE126848/"
    "GSE126848_Gene_counts_raw.txt.gz"
)

metadata_path = (
    project_path +
    "/data/raw/GSE126848/GSE126848_metadata.csv"
)

print("Count file exists:", os.path.isfile(counts_path))
print("Metadata exists:", os.path.isfile(metadata_path))

# Read just the count-matrix header, not the entire dataset.
with gzip.open(counts_path, "rt") as f:
    for i in range(3):
        line = f.readline()
        print(f"\nRAW LINE {i + 1}:")
        print(repr(line[:1200]))

# Inspect columns.
counts_header = pd.read_csv(
    counts_path,
    sep="\t",
    nrows=3
)

print("\nCOUNT MATRIX SHAPE (FIRST 3 ROWS):")
print(counts_header.shape)

print("\nCOUNT MATRIX COLUMNS:")
for i, col in enumerate(counts_header.columns):
    print(i, repr(col))

# Read metadata.
meta = pd.read_csv(metadata_path, index_col=0)

print("\nMETADATA SAMPLE IDS (FIRST 15):")
print(meta.index[:15].tolist())

print("\nMETADATA COLUMNS:")
print(meta.columns.tolist())

print("\nMETADATA SAMPLE TITLES (FIRST 15):")
if "Sample_title" in meta.columns:
    print(meta["Sample_title"].head(15).to_string())
else:
    print("Sample_title column not found")

In [ ]:
count_columns = set(map(str, counts_header.columns[1:]))
metadata_ids = set(map(str, meta.index))

print("Count matrix sample columns:", len(count_columns))
print("Metadata sample IDs:", len(metadata_ids))

print("\nExact matching sample IDs:")
print(sorted(count_columns & metadata_ids))

if "Sample_title" in meta.columns:
    metadata_titles = set(
        meta["Sample_title"].astype(str).str.strip()
    )

    print("\nMatching sample titles:")
    print(sorted(count_columns & metadata_titles))

In [ ]:
import pandas as pd

project_path = "/content/ML_Project/ML_Project-main"

meta = pd.read_csv(
    project_path + "/data/raw/GSE126848/GSE126848_metadata.csv",
    index_col=0
)

counts_path = (
    project_path +
    "/external_validation_v2/raw/GSE126848/"
    "GSE126848_Gene_counts_raw.txt.gz"
)

count_columns = pd.read_csv(
    counts_path,
    sep="\t",
    nrows=0
).columns[1:].astype(str).tolist()

mapping = meta[[
    "Sample_title",
    "Sample_description"
]].copy()

mapping["patient_id"] = (
    mapping["Sample_description"]
    .astype(str)
    .str.strip()
)

print(mapping.head(15).to_string())

print("\nMetadata patients:", len(mapping))
print("Count matrix patients:", len(count_columns))

unique = mapping["patient_id"].is_unique
missing = set(count_columns) - set(mapping["patient_id"])
extra = set(mapping["patient_id"]) - set(count_columns)

print("Unique patient identifiers:", unique)
print("Unmatched count columns:", sorted(missing))
print("Unmatched metadata identifiers:", sorted(extra))

assert unique, "Duplicate patient IDs detected"
assert not missing, "Some count columns are not mapped"
assert not extra, "Some metadata samples are not mapped"
assert len(mapping) == len(count_columns) == 57

print("\nSUCCESS: All 57 samples match!")

In [ ]:
import pandas as pd

# Create a fresh mapping from the metadata
mapping = meta[[
    "Sample_title",
    "Sample_description"
]].copy()

# Convert numeric patient IDs to four-character strings
mapping["patient_id"] = (
    mapping["Sample_description"]
    .astype(str)
    .str.strip()
    .str.zfill(4)
)

print(mapping.head(15).to_string())

In [ ]:
count_ids = set(count_columns)
metadata_ids = set(mapping["patient_id"])

missing = count_ids - metadata_ids
extra = metadata_ids - count_ids

print("Count matrix patients:", len(count_ids))
print("Metadata patients:", len(metadata_ids))
print("Matching patients:", len(count_ids & metadata_ids))

print("\nUnmatched count IDs:", sorted(missing))
print("Unmatched metadata IDs:", sorted(extra))

assert mapping["patient_id"].is_unique
assert len(count_ids) == 57
assert len(metadata_ids) == 57
assert not missing
assert not extra

print("\nSUCCESS: All 57 patient IDs match correctly!")

In [ ]:
from pathlib import Path
import shutil

script = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_pipeline.py"
)

# Create another backup
shutil.copy2(
    script,
    script.with_name(
        "external_validation_pipeline_before_zero_fix.py.bak"
    )
)

code = script.read_text(encoding="utf-8")

old = 'description = str(row.get("Sample_description", "")).strip()'

new = (
    'description = str(row.get("Sample_description", "")).strip().zfill(4)'
)

if code.count(old) != 1:
    raise RuntimeError(
        "Expected code not found exactly once. "
        "Check whether the previous modification was applied."
    )

code = code.replace(old, new)

script.write_text(code, encoding="utf-8")

print("SUCCESS: Leading-zero fix applied to pipeline!")

In [ ]:
from pathlib import Path
import shutil
import ast

script = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_pipeline.py"
)

# Create a backup before modifying anything
backup = script.with_name(
    "external_validation_pipeline_backup.py"
)

shutil.copy2(script, backup)

code = script.read_text(encoding="utf-8")

start_marker = "def standardize_count_matrix_samples("
end_marker = "\ndef prepare_gse126848("

if code.count(start_marker) != 1 or code.count(end_marker) != 1:
    raise RuntimeError(
        "Could not uniquely locate the sample-matching function. "
        "No changes made."
    )

start = code.index(start_marker)
end = code.index(end_marker, start)

new_function = '''
def standardize_count_matrix_samples(
    df: pd.DataFrame, meta: pd.DataFrame, gse_id: str
) -> pd.DataFrame:
    """Map count-matrix columns to GEO GSM IDs."""

    gene_col = identify_gene_id_column(df)
    df = df.copy()
    df[gene_col] = df[gene_col].astype(str)
    df = df.set_index(gene_col)

    title_to_gsm = {}
    patient_to_gsm = {}

    for gsm, row in meta.iterrows():

        title = str(row.get("Sample_title", "")).strip()

        if title:
            if title in title_to_gsm:
                raise PipelineError(
                    f"{gse_id}: duplicate sample title {title}"
                )
            title_to_gsm[title] = gsm

        if gse_id == "GSE126848":

            patient_id = str(
                row.get("Sample_description", "")
            ).strip()

            if not patient_id or patient_id.lower() == "nan":
                raise PipelineError(
                    f"Missing patient ID for {gsm}"
                )

            # Restore leading zeros, e.g. 869 -> 0869
            patient_id = patient_id.zfill(4)

            if patient_id in patient_to_gsm:
                raise PipelineError(
                    f"Duplicate patient ID: {patient_id}"
                )

            patient_to_gsm[patient_id] = gsm

    rename = {}

    for col in df.columns:

        cs = str(col).strip()

        if gse_id == "GSE126848" and cs in patient_to_gsm:
            rename[col] = patient_to_gsm[cs]

        elif cs in meta.index:
            rename[col] = cs

        elif cs in title_to_gsm:
            rename[col] = title_to_gsm[cs]

        else:
            norm = re.sub(
                r"[^A-Za-z0-9]+", "_", cs
            ).strip("_").lower()

            for title, gsm in title_to_gsm.items():
                tnorm = re.sub(
                    r"[^A-Za-z0-9]+", "_", title
                ).strip("_").lower()

                if norm == tnorm:
                    rename[col] = gsm
                    break

    sample_cols = [c for c in df.columns if c in rename]

    require(
        len(sample_cols) >= int(0.90 * len(meta)),
        f"{gse_id}: matched only {len(sample_cols)}/{len(meta)} "
        "count-matrix columns to GEO sample metadata."
    )

    counts = df[sample_cols].rename(columns=rename)

    require(
        counts.columns.is_unique,
        f"{gse_id}: duplicate GSM mappings detected."
    )

    counts = counts.apply(
        pd.to_numeric, errors="coerce"
    )

    counts = counts.dropna(how="all")

    require(
        not counts.isna().any().any(),
        f"{gse_id}: non-numeric or missing values."
    )

    shared = [
        s for s in meta.index
        if s in counts.columns
    ]

    return counts[shared]


'''

updated_code = code[:start] + new_function + code[end:]

# Check syntax before writing to disk
ast.parse(updated_code)

script.write_text(updated_code, encoding="utf-8")

print("SUCCESS: Sample-matching function replaced!")
print("Original backup:", backup)

In [ ]:
code = script.read_text(encoding="utf-8")

print(
    "Corrected function:",
    "def standardize_count_matrix_samples(" in code
)

print(
    "Leading-zero handling:",
    "patient_id.zfill(4)" in code
)

print(
    "Description mapping:",
    'row.get("Sample_description", "")' in code
)

In [ ]:
import importlib.util
import pandas as pd

spec = importlib.util.spec_from_file_location(
    "external_validation_pipeline",
    str(script)
)

pipeline = importlib.util.module_from_spec(spec)
import sys
sys.modules[spec.name] = pipeline
spec.loader.exec_module(pipeline)

counts_path = (
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v2/raw/GSE126848/"
    "GSE126848_Gene_counts_raw.txt.gz"
)

meta_path = (
    "/content/ML_Project/ML_Project-main/"
    "data/raw/GSE126848/GSE126848_metadata.csv"
)

df = pd.read_csv(counts_path, sep="\t")
meta = pd.read_csv(meta_path, index_col=0)

matched = pipeline.standardize_count_matrix_samples(
    df,
    meta,
    "GSE126848"
)

print("Matched count matrix shape:", matched.shape)
print("Total matched samples:", matched.shape[1])
print("Unique sample IDs:", matched.columns.is_unique)

assert matched.shape[1] == 57
assert matched.columns.is_unique

print("\nSUCCESS: All 57 samples matched correctly!")

In [ ]:
%cd /content/ML_Project/ML_Project-main

In [ ]:
!python external_validation_pipeline.py --stage prepare

In [ ]:
%cd /content/ML_Project/ML_Project-main

In [ ]:
!python external_validation_pipeline.py --stage train --quick

In [ ]:
import os

models_dir = (
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v2/models"
)

for filename in sorted(os.listdir(models_dir)):
    filepath = os.path.join(models_dir, filename)
    print(filename, round(os.path.getsize(filepath)/1024, 2), "KB")

In [ ]:
%cd /content/ML_Project/ML_Project-main

In [ ]:
!python external_validation_pipeline.py --stage validate

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

results_dir = (
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v2/results"
)

for dataset in ["GSE126848", "GSE135251"]:
    for task in ["binary", "multiclass"]:

        path = (
            f"{results_dir}/"
            f"external_predictions_{dataset}_{task}.csv"
        )

        df = pd.read_csv(path)

        print("\n" + "=" * 65)
        print(dataset, task.upper())
        print("=" * 65)

        print("\nActual classes:")
        print(df["y_true"].value_counts().sort_index())

        print("\nPredicted classes:")
        print(df["y_pred"].value_counts().sort_index())

        labels = [0, 1] if task == "binary" else [0, 1, 2]

        cm = confusion_matrix(
            df["y_true"],
            df["y_pred"],
            labels=labels
        )

        print("\nConfusion matrix:")
        print(cm)

        print("\nClassification report:")
        print(
            classification_report(
                df["y_true"],
                df["y_pred"],
                labels=labels,
                zero_division=0
            )
        )

In [ ]:
import pandas as pd
from pathlib import Path

results_dir = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v2/results"
)

for dataset in ["GSE126848", "GSE135251"]:
    path = results_dir / f"external_predictions_{dataset}_binary.csv"
    df = pd.read_csv(path)

    print("\n" + "=" * 65)
    print(dataset, "BINARY PROBABILITIES")
    print("=" * 65)

    print("\nProbability columns:")
    print([c for c in df.columns if c.startswith("prob_")])

    print("\nMean predicted probabilities by actual class:")
    print(
        df.groupby("y_true")[
            [c for c in df.columns if c.startswith("prob_")]
        ].mean().round(3)
    )

    print("\nPrediction probability summary:")
    print(
        df[
            [c for c in df.columns if c.startswith("prob_")]
        ].describe().round(3)
    )

In [ ]:
for dataset in ["GSE126848", "GSE135251"]:
    path = results_dir / f"external_predictions_{dataset}_multiclass.csv"
    df = pd.read_csv(path)

    probability_columns = [
        c for c in df.columns
        if c.startswith("prob_")
    ]

    print("\n" + "=" * 65)
    print(dataset, "MULTICLASS PROBABILITIES")
    print("=" * 65)

    print("\nAverage probabilities by actual class:")
    print(
        df.groupby("known_label")[probability_columns]
        .mean()
        .round(3)
    )

    print("\nPredicted class distribution:")
    print(df["predicted_label"].value_counts())

In [ ]:
from pathlib import Path

results_dir = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v2/results"
)

gene_lists = {}

for task in ["binary", "multiclass"]:

    gene_file = results_dir / f"selected_genes_{task}.txt"

    with open(gene_file, "r") as f:
        genes = [
            line.strip()
            for line in f
            if line.strip()
        ]

    gene_lists[task] = genes

    print(f"\n{task.upper()} MODEL")
    print("Selected genes:", len(genes))
    print("First 30 genes:", genes[:30])

In [ ]:
binary_genes = set(gene_lists["binary"])
multiclass_genes = set(gene_lists["multiclass"])

common_selected = sorted(
    binary_genes.intersection(multiclass_genes)
)

print("Binary selected genes:", len(binary_genes))
print("Multiclass selected genes:", len(multiclass_genes))
print("Shared selected genes:", len(common_selected))

print("\nFirst 30 shared genes:")
for gene in common_selected[:30]:
    print(gene)

In [ ]:
import json

for task in ["binary", "multiclass"]:

    summary_file = (
        results_dir / f"training_summary_{task}.json"
    )

    with open(summary_file, "r") as f:
        summary = json.load(f)

    print("\n" + "=" * 55)
    print(task.upper(), "TRAINING SUMMARY")
    print("=" * 55)

    print(json.dumps(summary, indent=4))

In [ ]:
import pandas as pd
from pathlib import Path

base = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v2/gene_level"
)

datasets = {}

for gse in ["GSE48452", "GSE126848", "GSE135251"]:
    path = base / f"{gse}_common_rank.csv.gz"

    df = pd.read_csv(path, index_col=0)

    datasets[gse] = df

    print(f"{gse}: {df.shape[0]} genes x {df.shape[1]} samples")

assert all(
    df.index.equals(datasets["GSE48452"].index)
    for df in datasets.values()
), "Gene order differs across cohorts!"

print("\nSUCCESS: Gene order is identical across all datasets.")

In [ ]:
import numpy as np
import pandas as pd

selected_genes = sorted(
    set(gene_lists["binary"]) |
    set(gene_lists["multiclass"])
)

rows = []

for gse, df in datasets.items():
    x = df.loc[selected_genes].to_numpy(dtype=float)

    rows.append({
        "Dataset": gse,
        "Samples": x.shape[1],
        "Selected genes": x.shape[0],
        "Mean rank": np.mean(x),
        "Median rank": np.median(x),
        "Rank SD": np.std(x),
        "5th percentile": np.percentile(x, 5),
        "95th percentile": np.percentile(x, 95)
    })

summary = pd.DataFrame(rows)

display(summary.round(4))

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, Image
import numpy as np
from pathlib import Path

# Output location
output_dir = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v2/figures"
)

output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "selected_gene_distribution.png"

# Load selected genes from your previous steps
selected_genes = sorted(
    set(gene_lists["binary"]) |
    set(gene_lists["multiclass"])
)

fig, ax = plt.subplots(figsize=(10, 6))

for gse, df in datasets.items():

    values = df.loc[selected_genes].to_numpy(
        dtype=float
    ).ravel()

    values = values[np.isfinite(values)]

    ax.hist(
        values,
        bins=50,
        density=True,
        histtype="step",
        linewidth=2,
        label=gse
    )

ax.set_xlabel("Within-sample gene percentile rank")
ax.set_ylabel("Density")
ax.set_title("Selected Gene Distributions Across Cohorts")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(output_file, dpi=200, bbox_inches="tight")

plt.close(fig)

print("Graph saved successfully!")
print("Location:", output_file)

display(Image(filename=str(output_file)))

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot([1, 2, 3, 4], [2, 4, 6, 8])
plt.title("Test Graph")
plt.show()

print("Graph test completed!")

## 04 — Gene matrices and experimental re-ranking


In [ ]:
import pandas as pd
from pathlib import Path

gene_dir = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v2/gene_level"
)

full_data = {}

for gse in ["GSE48452", "GSE126848", "GSE135251"]:
    path = gene_dir / f"{gse}_gene_rank.csv.gz"

    full_data[gse] = pd.read_csv(path, index_col=0)

    print(gse, full_data[gse].shape)

In [ ]:
common_genes = [
    gene.strip()
    for gene in (
        gene_dir / "common_genes.txt"
    ).read_text().splitlines()
    if gene.strip()
]

reranked = {}

for gse, df in full_data.items():

    x = df.loc[common_genes]

    reranked[gse] = x.rank(
        axis=0,
        method="average",
        pct=True
    )

    print(gse, reranked[gse].shape)

In [ ]:
import numpy as np
import pandas as pd

rows = []

for gse in reranked:

    old_values = datasets[gse].loc[selected_genes].values
    new_values = reranked[gse].loc[selected_genes].values

    rows.append({
        "Dataset": gse,
        "Old mean rank": old_values.mean(),
        "New mean rank": new_values.mean(),
        "Old rank SD": old_values.std(),
        "New rank SD": new_values.std(),
        "Mean absolute change": np.abs(
            old_values - new_values
        ).mean()
    })

comparison = pd.DataFrame(rows)

display(comparison.round(4))

In [ ]:
from pathlib import Path
import pandas as pd

project = Path("/content/ML_Project/ML_Project-main")

experiment = project / "external_validation_v2" / "experiments" / "common_gene_ranking"

experiment.mkdir(parents=True, exist_ok=True)

print("Experiment folder:", experiment)

In [ ]:
for gse, df in reranked.items():
    output = experiment / f"{gse}_common_rank.csv.gz"

    df.to_csv(output, compression="gzip")

    print(
        gse,
        "saved:",
        df.shape,
        "->",
        output
    )

In [ ]:
import numpy as np

expected_genes = list(common_genes)

for gse in ["GSE48452", "GSE126848", "GSE135251"]:
    path = experiment / f"{gse}_common_rank.csv.gz"
    df = pd.read_csv(path, index_col=0)

    assert df.index.tolist() == expected_genes
    assert df.index.is_unique
    assert df.columns.is_unique
    assert np.isfinite(df.to_numpy()).all()

    print(f"{gse}: {df.shape[0]} genes x {df.shape[1]} samples")

print("\nSUCCESS: All experimental matrices verified.")

In [ ]:
from pathlib import Path

project = Path("/content/ML_Project/ML_Project-main")

experiment = (
    project / "external_validation_v2" /
    "experiments" / "common_gene_ranking"
)

gene_dir = project / "external_validation_v2" / "gene_level"

experiment.mkdir(parents=True, exist_ok=True)

print("Experiment directory:", experiment)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    f1_score
)

# Load training labels
meta = pd.read_csv(
    gene_dir / "GSE48452_metadata_harmonized.csv",
    index_col=0
)

# Load both feature representations
original = pd.read_csv(
    gene_dir / "GSE48452_common_rank.csv.gz",
    index_col=0
)

reranked_data = pd.read_csv(
    experiment / "GSE48452_common_rank.csv.gz",
    index_col=0
)

assert original.index.equals(reranked_data.index)

samples = meta.index.tolist()

assert all(s in original.columns for s in samples)
assert all(s in reranked_data.columns for s in samples)

X_original = original[samples].T.to_numpy()
X_reranked = reranked_data[samples].T.to_numpy()

y = meta.loc[samples, "label_binary"].astype(int).to_numpy()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for representation, X in [
    ("Original ranking", X_original),
    ("Common-gene ranking", X_reranked)
]:

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), 1):

        model = Pipeline([
            ("select", SelectKBest(f_classif, k=200)),
            ("clf", RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            ))
        ])

        model.fit(X[train_idx], y[train_idx])

        predictions = model.predict(X[test_idx])
        probabilities = model.predict_proba(X[test_idx])[:, 1]

        results.append({
            "Representation": representation,
            "Fold": fold,
            "ROC_AUC": roc_auc_score(
                y[test_idx], probabilities
            ),
            "Balanced_Accuracy": balanced_accuracy_score(
                y[test_idx], predictions
            ),
            "Macro_F1": f1_score(
                y[test_idx], predictions,
                average="macro"
            )
        })

results_df = pd.DataFrame(results)

print("FOLD-BY-FOLD RESULTS")
display(results_df.round(4))

print("\nMEAN PERFORMANCE")
summary = results_df.groupby("Representation")[
    ["ROC_AUC", "Balanced_Accuracy", "Macro_F1"]
].agg(["mean", "std"])

display(summary.round(4))

results_df.to_csv(
    experiment / "ranking_comparison_binary.csv",
    index=False
)

print("\nComparison results saved.")

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")

backup_path = Path(
    "/content/drive/MyDrive/"
    "MASLD_Validation_Backup/common_gene_ranking"
)

shutil.copytree(
    experiment,
    backup_path,
    dirs_exist_ok=True
)

print("Experiment backed up successfully!")

In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    f1_score
)

seeds = [11, 21, 31, 42, 52]

results = []

for seed in seeds:

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed
    )

    for representation, X in [
        ("Original ranking", X_original),
        ("Common-gene ranking", X_reranked)
    ]:

        for fold, (train_idx, test_idx) in enumerate(
            cv.split(X, y), 1
        ):

            model = Pipeline([
                ("select", SelectKBest(f_classif, k=200)),
                ("clf", RandomForestClassifier(
                    n_estimators=200,
                    random_state=42,
                    n_jobs=-1
                ))
            ])

            model.fit(X[train_idx], y[train_idx])

            pred = model.predict(X[test_idx])
            prob = model.predict_proba(X[test_idx])[:, 1]

            results.append({
                "seed": seed,
                "representation": representation,
                "fold": fold,
                "auc": roc_auc_score(y[test_idx], prob),
                "balanced_accuracy": balanced_accuracy_score(
                    y[test_idx], pred
                ),
                "macro_f1": f1_score(
                    y[test_idx], pred,
                    average="macro"
                )
            })

repeated_results = pd.DataFrame(results)

summary = repeated_results.groupby("representation")[
    ["auc", "balanced_accuracy", "macro_f1"]
].agg(["mean", "std"])

print("REPEATED CROSS-VALIDATION RESULTS")
display(summary.round(4))

repeated_results.to_csv(
    experiment / "repeated_cv_comparison.csv",
    index=False
)

print("Results saved successfully!")

In [ ]:
seed_summary = repeated_results.groupby(
    ["seed", "representation"]
)[
    ["auc", "balanced_accuracy", "macro_f1"]
].mean()

display(seed_summary.round(4))

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

source = Path(
    "/content/ML_Project/ML_Project-main/external_validation_v2"
)

destination = Path(
    "/content/drive/MyDrive/MASLD_Validation_Backup/baseline_v2"
)

if not source.is_dir():
    raise FileNotFoundError(source)

shutil.copytree(
    source,
    destination,
    dirs_exist_ok=True
)

print("Baseline V2 backup saved successfully!")
print("Location:", destination)

In [ ]:
from pathlib import Path

project = Path("/content/ML_Project/ML_Project-main")

v3_dir = project / "external_validation_v3"
v3_dir.mkdir(parents=True, exist_ok=True)

for folder in [
    "data",
    "models",
    "results",
    "figures",
    "logs"
]:
    (v3_dir / folder).mkdir(exist_ok=True)

print("Version 3 workspace created successfully!")
print("Location:", v3_dir)

In [ ]:
import shutil

experiment = (
    project / "external_validation_v2" /
    "experiments" / "common_gene_ranking"
)

for gse in ["GSE48452", "GSE126848", "GSE135251"]:
    filename = f"{gse}_common_rank.csv.gz"

    source = experiment / filename
    destination = v3_dir / "data" / filename

    if not source.is_file():
        raise FileNotFoundError(source)

    shutil.copy2(source, destination)
    print("Copied:", filename)

print("\nVersion 3 datasets ready!")

In [ ]:
gene_dir = project / "external_validation_v2" / "gene_level"

for gse in ["GSE48452", "GSE126848", "GSE135251"]:
    filename = f"{gse}_metadata_harmonized.csv"

    source = gene_dir / filename
    destination = v3_dir / "data" / filename

    if not source.is_file():
        raise FileNotFoundError(source)

    shutil.copy2(source, destination)
    print("Copied:", filename)

print("\nClinical metadata copied successfully!")

In [ ]:
import pandas as pd

for gse, expected_samples in [
    ("GSE48452", 73),
    ("GSE126848", 57),
    ("GSE135251", 216)
]:
    expression = pd.read_csv(
        v3_dir / "data" / f"{gse}_common_rank.csv.gz",
        index_col=0
    )

    metadata = pd.read_csv(
        v3_dir / "data" / f"{gse}_metadata_harmonized.csv",
        index_col=0
    )

    assert expression.shape == (10152, expected_samples)
    assert len(metadata) == expected_samples
    assert set(expression.columns) == set(metadata.index)

    print(
        f"{gse}: {expression.shape[0]} genes, "
        f"{expression.shape[1]} samples — VERIFIED"
    )

print("\nSUCCESS: Version 3 input validation completed!")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

v3_dir = Path(
    "/content/ML_Project/ML_Project-main/external_validation_v3"
)

data_dir = v3_dir / "data"
models_dir = v3_dir / "models"
results_dir = v3_dir / "results"

models_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

expression = pd.read_csv(
    data_dir / "GSE48452_common_rank.csv.gz",
    index_col=0
)

metadata = pd.read_csv(
    data_dir / "GSE48452_metadata_harmonized.csv",
    index_col=0
)

samples = metadata.index.tolist()

assert expression.index.is_unique
assert expression.columns.is_unique
assert set(samples) == set(expression.columns)

genes = expression.index.tolist()

X = expression[samples].T.to_numpy(dtype=np.float32)

y_binary = metadata.loc[
    samples, "label_binary"
].astype(int).to_numpy()

y_multiclass = metadata.loc[
    samples, "label_3class"
].astype(int).to_numpy()

assert np.isfinite(X).all()

print("Training data loaded successfully!")
print("Samples:", X.shape[0])
print("Genes:", X.shape[1])

print("\nBinary labels:")
print(pd.Series(y_binary).value_counts().sort_index())

print("\nMulticlass labels:")
print(pd.Series(y_multiclass).value_counts().sort_index())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV
)

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

from sklearn.base import clone
import joblib
import json

RANDOM_STATE = 42

def make_models(task):

    xgb_kwargs = {
        "n_estimators": 200,
        "max_depth": 3,
        "learning_rate": 0.05,
        "random_state": 42,
        "n_jobs": 1,
        "eval_metric": "logloss" if task == "binary" else "mlogloss"
    }

    if task == "multiclass":
        xgb_kwargs["objective"] = "multi:softprob"
        xgb_kwargs["num_class"] = 3
    else:
        xgb_kwargs["objective"] = "binary:logistic"

    return {
        "LogisticRegression": LogisticRegression(
            max_iter=4000, random_state=42
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=200, random_state=42, n_jobs=1
        ),
        "SVM": SVC(
            probability=True, random_state=42
        ),
        "XGBoost": XGBClassifier(**xgb_kwargs)
    }

def make_pipeline(model):
    return Pipeline([
        ("select", SelectKBest(f_classif, k=200)),
        ("scale", StandardScaler()),
        ("smote", SMOTE(
            random_state=42,
            k_neighbors=3
        )),
        ("clf", clone(model))
    ])

print("Version 3 model definitions ready!")

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    f1_score
)
from sklearn.pipeline import Pipeline

def run_v3_training_simple(X, y, task="binary"):

    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=2000,
            random_state=42
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=1
        )
    }

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    results = []

    for model_name, estimator in models.items():

        print(f"\nTraining {model_name}...", flush=True)

        for fold, (train_idx, test_idx) in enumerate(
            cv.split(X, y), start=1
        ):

            pipeline = Pipeline([
                ("select", SelectKBest(f_classif, k=200)),
                ("scale", StandardScaler()),
                ("clf", estimator)
            ])

            pipeline.fit(X[train_idx], y[train_idx])

            pred = pipeline.predict(X[test_idx])
            proba = pipeline.predict_proba(X[test_idx])

            if task == "binary":
                auc = roc_auc_score(
                    y[test_idx],
                    proba[:, 1]
                )
            else:
                auc = roc_auc_score(
                    y[test_idx],
                    proba,
                    multi_class="ovr",
                    average="macro",
                    labels=[0, 1, 2]
                )

            results.append({
                "model": model_name,
                "fold": fold,
                "auc": auc,
                "balanced_accuracy": balanced_accuracy_score(
                    y[test_idx], pred
                ),
                "macro_f1": f1_score(
                    y[test_idx],
                    pred,
                    average="macro",
                    zero_division=0
                )
            })

            print(
                f"Fold {fold} completed | "
                f"AUC: {auc:.3f}",
                flush=True
            )

    df = pd.DataFrame(results)

    print("\nFINAL RESULTS:")
    display(
        df.groupby("model")[
            ["auc", "balanced_accuracy", "macro_f1"]
        ].mean().round(4)
    )

    return df

In [ ]:
binary_results = run_v3_training(X, y_binary, "binary")

In [ ]:
print("Checking required variables...\n")

required = [
    "X",
    "y_binary",
    "y_multiclass",
    "make_models",
    "make_pipeline",
    "results_dir"
]

for name in required:
    print(
        name,
        "AVAILABLE" if name in globals() else "MISSING"
    )

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_binary,
    test_size=0.2,
    stratify=y_binary,
    random_state=42
)

test_model = Pipeline([
    ("select", SelectKBest(f_classif, k=200)),
    ("clf", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=1
    ))
])

print("Starting Random Forest test...", flush=True)

test_model.fit(X_train, y_train)

predictions = test_model.predict(X_test)
probabilities = test_model.predict_proba(X_test)[:, 1]

print("\nTEST COMPLETED SUCCESSFULLY!")
print("Accuracy:", accuracy_score(y_test, predictions))
print("ROC-AUC:", roc_auc_score(y_test, probabilities))

In [ ]:
print("TEST 1: Google Colab is working", flush=True)

a = 10
b = 20

print("Result:", a + b, flush=True)

In [ ]:
required = [
    "X",
    "y_binary",
    "y_multiclass",
    "make_models",
    "make_pipeline",
    "results_dir",
    "run_v3_training_simple"
]

for name in required:
    print(
        name,
        "AVAILABLE" if name in globals() else "MISSING",
        flush=True
    )

In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

project = Path("/content/ML_Project/ML_Project-main")

data_dir = project / "external_validation_v3" / "data"

print("Loading training data...", flush=True)

expression = pd.read_csv(
    data_dir / "GSE48452_common_rank.csv.gz",
    index_col=0
)

metadata = pd.read_csv(
    data_dir / "GSE48452_metadata_harmonized.csv",
    index_col=0
)

samples = metadata.index.tolist()

X = expression[samples].T.to_numpy(dtype=np.float32)

y_binary = metadata.loc[
    samples, "label_binary"
].astype(int).to_numpy()

print("Data loaded:", X.shape, flush=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

model = Pipeline([
    ("select", SelectKBest(f_classif, k=50)),
    ("clf", LogisticRegression(max_iter=1000))
])

print("Starting model training...", flush=True)

model.fit(X_train, y_train)

print("Training completed!", flush=True)

predictions = model.predict(X_test)

print(
    "Diagnostic accuracy:",
    accuracy_score(y_test, predictions)
)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

from pathlib import Path

results_dir = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3/results"
)

results_dir.mkdir(parents=True, exist_ok=True)

models = {
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=1
    )
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

print("Starting V3 binary cross-validation...", flush=True)

for model_name, estimator in models.items():

    print(f"\nMODEL: {model_name}", flush=True)

    for fold, (train_idx, test_idx) in enumerate(
        cv.split(X, y_binary), start=1
    ):

        model = Pipeline([
            ("select", SelectKBest(
                score_func=f_classif,
                k=200
            )),
            ("scale", StandardScaler()),
            ("clf", estimator)
        ])

        model.fit(X[train_idx], y_binary[train_idx])

        pred = model.predict(X[test_idx])
        prob = model.predict_proba(X[test_idx])[:, 1]

        results.append({
            "Model": model_name,
            "Fold": fold,
            "Accuracy": accuracy_score(
                y_binary[test_idx], pred
            ),
            "Balanced_Accuracy": balanced_accuracy_score(
                y_binary[test_idx], pred
            ),
            "Macro_F1": f1_score(
                y_binary[test_idx],
                pred,
                average="macro"
            ),
            "ROC_AUC": roc_auc_score(
                y_binary[test_idx], prob
            )
        })

        print(
            f"Fold {fold}/5 completed",
            flush=True
        )

results_df = pd.DataFrame(results)

print("\nFINAL CROSS-VALIDATION RESULTS")

summary = (
    results_df.groupby("Model")[
        ["Accuracy", "Balanced_Accuracy", "Macro_F1", "ROC_AUC"]
    ].agg(["mean", "std"])
)

display(summary.round(4))

results_df.to_csv(
    results_dir / "v3_binary_cv_test.csv",
    index=False
)

print("\nSUCCESS: V3 binary comparison completed!")

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

models = {
    "LogisticRegression": LogisticRegression(
        max_iter=3000,
        random_state=42
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=1
    )
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

multiclass_results = []

print("Starting V3 multiclass cross-validation...", flush=True)

for model_name, estimator in models.items():

    print(f"\nMODEL: {model_name}", flush=True)

    for fold, (train_idx, test_idx) in enumerate(
        cv.split(X, y_multiclass), start=1
    ):

        model = Pipeline([
            ("select", SelectKBest(
                score_func=f_classif,
                k=200
            )),
            ("scale", StandardScaler()),
            ("clf", estimator)
        ])

        model.fit(X[train_idx], y_multiclass[train_idx])

        predictions = model.predict(X[test_idx])
        probabilities = model.predict_proba(X[test_idx])

        auc = roc_auc_score(
            y_multiclass[test_idx],
            probabilities,
            multi_class="ovr",
            average="macro",
            labels=[0, 1, 2]
        )

        multiclass_results.append({
            "Model": model_name,
            "Fold": fold,
            "Accuracy": accuracy_score(
                y_multiclass[test_idx], predictions
            ),
            "Balanced_Accuracy": balanced_accuracy_score(
                y_multiclass[test_idx], predictions
            ),
            "Macro_F1": f1_score(
                y_multiclass[test_idx],
                predictions,
                average="macro",
                zero_division=0
            ),
            "ROC_AUC": auc
        })

        print(
            f"Fold {fold}/5 completed | AUC={auc:.3f}",
            flush=True
        )

multiclass_results_df = pd.DataFrame(multiclass_results)

summary = multiclass_results_df.groupby("Model")[
    ["Accuracy", "Balanced_Accuracy", "Macro_F1", "ROC_AUC"]
].agg(["mean", "std"])

print("\nFINAL MULTICLASS RESULTS")
display(summary.round(4))

multiclass_results_df.to_csv(
    results_dir / "v3_multiclass_cv_test.csv",
    index=False
)

print("\nSUCCESS: V3 multiclass comparison completed!")

In [ ]:
from sklearn.svm import SVC
from xgboost import XGBClassifier

extra_models = {
    "SVM": SVC(
        C=1.0,
        kernel="rbf",
        gamma="scale",
        probability=True,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        random_state=42,
        n_jobs=1,
        eval_metric="mlogloss"
    )
}

print("SVM and XGBoost ready!")

In [ ]:
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

extra_results = []

for model_name, estimator in extra_models.items():

    print(f"\nTraining {model_name}...", flush=True)

    for fold, (tr, te) in enumerate(
        cv.split(X, y_multiclass), 1
    ):

        model = Pipeline([
            ("select", SelectKBest(f_classif, k=200)),
            ("scale", StandardScaler()),
            ("clf", clone(estimator))
        ])

        model.fit(X[tr], y_multiclass[tr])

        pred = model.predict(X[te])
        proba = model.predict_proba(X[te])

        scores = {
            "Model": model_name,
            "Fold": fold,
            "Accuracy": accuracy_score(y_multiclass[te], pred),
            "Balanced_Accuracy": balanced_accuracy_score(
                y_multiclass[te], pred
            ),
            "Macro_F1": f1_score(
                y_multiclass[te], pred,
                average="macro",
                zero_division=0
            ),
            "ROC_AUC": roc_auc_score(
                y_multiclass[te],
                proba,
                multi_class="ovr",
                average="macro",
                labels=[0, 1, 2]
            )
        }

        extra_results.append(scores)

        print(f"Fold {fold}/5 completed", flush=True)

extra_df = pd.DataFrame(extra_results)

print("\nSVM AND XGBOOST RESULTS")

display(
    extra_df.groupby("Model")[
        ["Accuracy", "Balanced_Accuracy", "Macro_F1", "ROC_AUC"]
    ].mean().round(4)
)

extra_df.to_csv(
    results_dir / "v3_extra_multiclass_cv.csv",
    index=False
)

print("\nSUCCESS: Additional model comparison completed!")

In [ ]:
import pandas as pd
from pathlib import Path

from sklearn.base import clone
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)
from xgboost import XGBClassifier

results_dir = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3/results"
)
results_dir.mkdir(parents=True, exist_ok=True)

binary_models = {
    "SVM": SVC(
        C=1.0,
        kernel="rbf",
        probability=True,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=1
    )
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

binary_extra_results = []

print("Starting additional V3 binary models...", flush=True)

for model_name, estimator in binary_models.items():

    print(f"\nTraining {model_name}", flush=True)

    for fold, (tr, te) in enumerate(
        cv.split(X, y_binary), start=1
    ):

        pipeline = Pipeline([
            ("select", SelectKBest(f_classif, k=200)),
            ("scale", StandardScaler()),
            ("clf", clone(estimator))
        ])

        pipeline.fit(X[tr], y_binary[tr])

        pred = pipeline.predict(X[te])
        proba = pipeline.predict_proba(X[te])[:, 1]

        binary_extra_results.append({
            "Model": model_name,
            "Fold": fold,
            "Accuracy": accuracy_score(y_binary[te], pred),
            "Balanced_Accuracy": balanced_accuracy_score(
                y_binary[te], pred
            ),
            "Macro_F1": f1_score(
                y_binary[te], pred,
                average="macro",
                zero_division=0
            ),
            "ROC_AUC": roc_auc_score(
                y_binary[te], proba
            )
        })

        print(f"Fold {fold}/5 completed", flush=True)

binary_extra_df = pd.DataFrame(binary_extra_results)

print("\nSVM AND XGBOOST BINARY RESULTS")

display(
    binary_extra_df.groupby("Model")[
        ["Accuracy", "Balanced_Accuracy", "Macro_F1", "ROC_AUC"]
    ].mean().round(4)
)

binary_extra_df.to_csv(
    results_dir / "v3_extra_binary_cv.csv",
    index=False
)

print("\nSUCCESS: Additional binary comparison completed!")

In [ ]:
original_binary = pd.read_csv(
    results_dir / "v3_binary_cv_test.csv"
)

extra_binary = pd.read_csv(
    results_dir / "v3_extra_binary_cv.csv"
)

all_binary = pd.concat(
    [original_binary, extra_binary],
    ignore_index=True
)

summary = (
    all_binary.groupby("Model")[
        ["Accuracy", "Balanced_Accuracy", "Macro_F1", "ROC_AUC"]
    ]
    .mean()
    .sort_values("Macro_F1", ascending=False)
)

print("COMPLETE V3 BINARY MODEL COMPARISON")
display(summary.round(4))

summary.to_csv(
    results_dir / "v3_all_binary_models_summary.csv"
)

In [ ]:
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    f1_score
)
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

nested_results = []

for fold, (train_idx, test_idx) in enumerate(
    outer_cv.split(X, y_binary), start=1
):

    print(f"\nStarting outer fold {fold}/5", flush=True)

    inner_cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42 + fold
    )

    pipeline = Pipeline([
        ("select", SelectKBest(f_classif)),
        ("scale", StandardScaler()),
        ("smote", SMOTE(random_state=42, k_neighbors=2)),
        ("clf", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=1
        ))
    ])

    grid = {
        "select__k": [100, 200],
        "clf__n_estimators": [100, 200],
        "clf__max_depth": [2, 3]
    }

    search = GridSearchCV(
        pipeline,
        param_grid=grid,
        scoring="f1_macro",
        cv=inner_cv,
        n_jobs=1,
        refit=True,
        error_score="raise"
    )

    search.fit(
        X[train_idx],
        y_binary[train_idx]
    )

    pred = search.predict(X[test_idx])
    prob = search.predict_proba(X[test_idx])[:, 1]

    result = {
        "fold": fold,
        "roc_auc": roc_auc_score(
            y_binary[test_idx], prob
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_binary[test_idx], pred
        ),
        "macro_f1": f1_score(
            y_binary[test_idx],
            pred,
            average="macro"
        ),
        "best_params": str(search.best_params_)
    }

    nested_results.append(result)

    print(
        f"Fold {fold} completed | "
        f"AUC={result['roc_auc']:.3f} | "
        f"F1={result['macro_f1']:.3f}",
        flush=True
    )

nested_df = pd.DataFrame(nested_results)

print("\nNESTED CROSS-VALIDATION RESULTS")
display(nested_df)

print("\nMEAN PERFORMANCE")
display(
    nested_df[
        ["roc_auc", "balanced_accuracy", "macro_f1"]
    ].agg(["mean", "std"]).round(4)
)

nested_df.to_csv(
    results_dir / "v3_xgboost_binary_nested_cv.csv",
    index=False
)

print("\nSUCCESS: Binary nested CV completed!")

## 05 — Model development and exploratory evaluation


In [ ]:
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier

def make_corrected_binary_pipeline():

    return Pipeline([
        ("variance", VarianceThreshold(threshold=0.0)),

        ("select", SelectKBest(
            score_func=f_classif,
            k=200
        )),

        ("scale", StandardScaler()),

        ("smote", SMOTE(
            random_state=42,
            k_neighbors=2
        )),

        ("clf", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=1
        ))
    ])

print("Corrected binary pipeline created!")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_binary,
    test_size=0.20,
    stratify=y_binary,
    random_state=42
)

model = make_corrected_binary_pipeline()

print("Testing corrected pipeline...", flush=True)

model.fit(X_train, y_train)

pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

print("Corrected pipeline trained successfully!")
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 4))
print("Balanced accuracy:", round(balanced_accuracy_score(y_test, pred), 4))
print("Macro F1:", round(f1_score(y_test, pred, average="macro"), 4))

print(
    "Genes remaining after variance filtering:",
    model.named_steps["variance"].get_support().sum()
)

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import VarianceThreshold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("CHECKING CONSTANT GENES IN TRAINING FOLDS\n")

for fold, (train_idx, test_idx) in enumerate(
    cv.split(X, y_binary), start=1
):

    X_train_fold = X[train_idx]

    selector = VarianceThreshold(threshold=0.0)
    selector.fit(X_train_fold)

    retained = selector.get_support().sum()
    removed = X_train_fold.shape[1] - retained

    print(
        f"Fold {fold}: "
        f"Retained {retained} genes | "
        f"Removed {removed} constant genes"
    )

print("\nVariance filtering verification completed.")

In [ ]:
print("Genes after variance filtering:",
      model.named_steps["variance"].get_support().sum())

print("Genes selected for classification:",
      model.named_steps["select"].get_support().sum())

print("Final classifier:",
      type(model.named_steps["clf"]).__name__)

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import VarianceThreshold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("CHECKING ALL FIVE TRAINING FOLDS\n")

for fold, (train_idx, test_idx) in enumerate(
    cv.split(X, y_binary), start=1
):
    X_train_fold = X[train_idx]

    selector = VarianceThreshold(threshold=0.0)
    selector.fit(X_train_fold)

    remaining = selector.get_support().sum()
    removed = X_train_fold.shape[1] - remaining

    print(
        f"Fold {fold}: "
        f"{remaining} genes retained, "
        f"{removed} constant genes removed"
    )

print("\nVerification completed!")

In [ ]:
from pathlib import Path
import shutil

source = Path(
    "/content/ML_Project/ML_Project-main/external_validation_v3"
)

destination = Path(
    "/content/drive/MyDrive/MASLD_Validation_Backup/v3_experiments"
)

assert source.is_dir(), "V3 directory not found"
assert Path("/content/drive/MyDrive").is_dir(), "Mount Google Drive first"

shutil.copytree(source, destination, dirs_exist_ok=True)

print("V3 experiment backup saved successfully!")

In [ ]:
from pathlib import Path

v3_dir = Path(
    "/content/ML_Project/ML_Project-main/external_validation_v3"
)

final_dir = v3_dir / "v3_final_run"

for folder in ["models", "results", "figures", "logs"]:
    (final_dir / folder).mkdir(parents=True, exist_ok=True)

print("Final-run workspace created successfully!")
print("Location:", final_dir)

In [ ]:
from pathlib import Path

data_dir = v3_dir / "data"

for gse in ["GSE48452", "GSE126848", "GSE135251"]:
    expression = data_dir / f"{gse}_common_rank.csv.gz"
    metadata = data_dir / f"{gse}_metadata_harmonized.csv"

    assert expression.is_file(), f"Missing: {expression}"
    assert metadata.is_file(), f"Missing: {metadata}"

    print(f"{gse}: expression and metadata FOUND")

print("\nSUCCESS: All V3 inputs are available.")

In [ ]:

%%writefile /content/ML_Project/ML_Project-main/external_validation_v3/run_v3_pipeline.py

import argparse
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, roc_auc_score
)

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

ROOT = Path(__file__).resolve().parent
DATA = ROOT / "data"
OUTPUT = ROOT / "v3_final_run"
MODELS = OUTPUT / "models"
RESULTS = OUTPUT / "results"

RANDOM_STATE = 42

for folder in [MODELS, RESULTS]:
    folder.mkdir(parents=True, exist_ok=True)


def load_data(task):
    expression = pd.read_csv(
        DATA / "GSE48452_common_rank.csv.gz",
        index_col=0
    )
    metadata = pd.read_csv(
        DATA / "GSE48452_metadata_harmonized.csv",
        index_col=0
    )

    genes = expression.index.tolist()
    samples = metadata.index.tolist()

    assert expression.index.is_unique
    assert expression.columns.is_unique
    assert set(samples) == set(expression.columns)
    assert len(samples) == 73
    assert len(genes) == 10152

    X = expression[samples].T.to_numpy(dtype=np.float32)

    column = "label_binary" if task == "binary" else "label_3class"
    y = metadata.loc[samples, column].astype(int).to_numpy()

    expected_classes = [0, 1] if task == "binary" else [0, 1, 2]

    assert sorted(np.unique(y).tolist()) == expected_classes
    assert np.isfinite(X).all()

    return X, y, genes, samples


def model_definitions(task, quick):
    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=4000, random_state=RANDOM_STATE
        ),
        "RandomForest": RandomForestClassifier(
            random_state=RANDOM_STATE, n_jobs=1
        ),
        "SVM": SVC(
            probability=True, random_state=RANDOM_STATE
        ),
        "XGBoost": XGBClassifier(
            objective="binary:logistic" if task == "binary"
            else "multi:softprob",
            eval_metric="logloss" if task == "binary" else "mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=1
        )
    }

    if quick:
        grids = {
            "LogisticRegression": {"clf__C": [1.0]},
            "RandomForest": {
                "clf__n_estimators": [100],
                "clf__max_depth": [None]
            },
            "SVM": {"clf__C": [1.0], "clf__gamma": ["scale"]},
            "XGBoost": {
                "clf__n_estimators": [100],
                "clf__max_depth": [2]
            }
        }
        k_values = [200]
    else:
        grids = {
            "LogisticRegression": {"clf__C": [0.1, 1.0, 10.0]},
            "RandomForest": {
                "clf__n_estimators": [100, 200],
                "clf__max_depth": [None, 5, 10]
            },
            "SVM": {
                "clf__C": [0.1, 1.0, 10.0],
                "clf__gamma": ["scale", "auto"]
            },
            "XGBoost": {
                "clf__n_estimators": [100, 200],
                "clf__max_depth": [2, 3],
                "clf__learning_rate": [0.03, 0.1]
            }
        }
        k_values = [100, 200, 400]

    return models, grids, k_values


def create_pipeline(estimator):
    return Pipeline([
        ("variance", VarianceThreshold(threshold=0.0)),
        ("select", SelectKBest(f_classif, k=200)),
        ("scale", StandardScaler()),
        ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=2)),
        ("clf", clone(estimator))
    ])


def make_search(name, models, grids, k_values, cv):
    params = dict(grids[name])
    params["select__k"] = k_values

    return GridSearchCV(
        estimator=create_pipeline(models[name]),
        param_grid=params,
        scoring="f1_macro",
        cv=cv,
        n_jobs=1,
        refit=True,
        error_score="raise"
    )


def calculate_metrics(y_true, predictions, probabilities, task):
    scores = {
        "accuracy": accuracy_score(y_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_true, predictions),
        "macro_f1": f1_score(
            y_true, predictions, average="macro", zero_division=0
        )
    }

    if task == "binary":
        scores["roc_auc"] = roc_auc_score(
            y_true, probabilities[:, 1]
        )
    else:
        scores["roc_auc"] = roc_auc_score(
            y_true, probabilities,
            multi_class="ovr", average="macro",
            labels=[0, 1, 2]
        )

    return scores


def train_task(task, quick=False):
    print(f"\n{'=' * 65}", flush=True)
    print(f"V3 TRAINING: {task.upper()}", flush=True)
    print(f"{'=' * 65}", flush=True)

    X, y, genes, samples = load_data(task)
    models, grids, k_values = model_definitions(task, quick)

    outer = StratifiedKFold(
        n_splits=5, shuffle=True, random_state=RANDOM_STATE
    )

    rows = []
    model_rows = []

    # Outer CV estimates the entire algorithm-selection procedure.
    for fold, (train_idx, test_idx) in enumerate(outer.split(X, y), 1):
        print(f"\nOuter fold {fold}/5", flush=True)

        inner = StratifiedKFold(
            n_splits=3, shuffle=True,
            random_state=RANDOM_STATE + fold
        )

        candidates = []

        for name in models:
            print(f"  Tuning {name}...", flush=True)

            search = make_search(
                name, models, grids, k_values, inner
            )
            search.fit(X[train_idx], y[train_idx])

            candidates.append((name, search))

            print(
                f"  {name}: inner F1={search.best_score_:.4f}",
                flush=True
            )

        # Selection uses inner training CV only.
        best_name, best_search = max(
            candidates, key=lambda item: item[1].best_score_
        )

        fitted = best_search.best_estimator_
        predictions = fitted.predict(X[test_idx])
        probabilities = fitted.predict_proba(X[test_idx])

        scores = calculate_metrics(
            y[test_idx], predictions, probabilities, task
        )

        rows.append({
            "fold": fold,
            "selected_model": best_name,
            **scores,
            "best_params": json.dumps(
                best_search.best_params_, sort_keys=True
            )
        })

        for name, search in candidates:
            model_rows.append({
                "fold": fold,
                "model": name,
                "inner_cv_macro_f1": search.best_score_
            })

        print(
            f"  Selected: {best_name} | "
            f"Outer F1={scores['macro_f1']:.4f} | "
            f"AUC={scores['roc_auc']:.4f}",
            flush=True
        )

    results = pd.DataFrame(rows)
    results.to_csv(
        RESULTS / f"{task}_nested_cv.csv", index=False
    )

    pd.DataFrame(model_rows).to_csv(
        RESULTS / f"{task}_inner_model_scores.csv",
        index=False
    )

    print("\nNESTED CV SUMMARY", flush=True)
    summary = results[
        ["accuracy", "balanced_accuracy", "macro_f1", "roc_auc"]
    ].agg(["mean", "std"])

    print(summary.round(4).to_string(), flush=True)

    summary.to_csv(
        RESULTS / f"{task}_nested_cv_summary.csv"
    )

    # Final selection and fitting use the full training cohort only.
    print("\nSelecting final model using GSE48452...", flush=True)

    final_inner = StratifiedKFold(
        n_splits=5, shuffle=True,
        random_state=RANDOM_STATE
    )

    final_candidates = []

    for name in models:
        search = make_search(
            name, models, grids, k_values, final_inner
        )
        search.fit(X, y)
        final_candidates.append((name, search))

        print(
            f"  {name}: full-training CV F1={search.best_score_:.4f}",
            flush=True
        )

    winner, winner_search = max(
        final_candidates, key=lambda item: item[1].best_score_
    )

    final_model = winner_search.best_estimator_

    variance_mask = final_model.named_steps[
        "variance"
    ].get_support()

    retained_genes = np.asarray(genes)[variance_mask]

    selected_mask = final_model.named_steps[
        "select"
    ].get_support()

    selected_genes = retained_genes[selected_mask].tolist()

    artifact = {
        "task": task,
        "model_name": winner,
        "model": final_model,
        "common_genes": genes,
        "selected_genes": selected_genes,
        "training_dataset": "GSE48452",
        "best_params": winner_search.best_params_,
        "quick_run": quick,
        "label_mapping": (
            {0: "Healthy", 1: "Disease"}
            if task == "binary"
            else {0: "Healthy", 1: "MASLD", 2: "MASH"}
        )
    }

    suffix = "quick" if quick else "full"

    output_model = MODELS / f"final_{task}_{suffix}.joblib"
    joblib.dump(artifact, output_model)

    (RESULTS / f"selected_genes_{task}_{suffix}.txt").write_text(
        "\n".join(selected_genes), encoding="utf-8"
    )

    print(f"\nFinal selected algorithm: {winner}", flush=True)
    print(f"Selected genes: {len(selected_genes)}", flush=True)
    print(f"Saved: {output_model}", flush=True)

    return results


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--task",
        choices=["binary", "multiclass", "both"],
        default="both"
    )
    parser.add_argument("--quick", action="store_true")
    args = parser.parse_args()

    tasks = (
        ["binary", "multiclass"]
        if args.task == "both"
        else [args.task]
    )

    for task in tasks:
        train_task(task, quick=args.quick)

    print("\nV3 TRAINING COMPLETED SUCCESSFULLY!", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
import py_compile

script_path = (
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3/run_v3_pipeline.py"
)

py_compile.compile(script_path, doraise=True)

print("SUCCESS: Version 3 script passed the syntax check!")

In [ ]:
%cd /content/ML_Project/ML_Project-main/external_validation_v3

In [ ]:
!python -u run_v3_pipeline.py --task binary --quick

In [ ]:
from pathlib import Path
import joblib

model_path = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3/v3_final_run/"
    "models/final_binary_quick.joblib"
)

assert model_path.is_file(), "Model not saved!"

artifact = joblib.load(model_path)

print("Final algorithm:", artifact["model_name"])
print("Training cohort:", artifact["training_dataset"])
print("Input genes:", len(artifact["common_genes"]))
print("Selected genes:", len(artifact["selected_genes"]))
print("Quick experiment:", artifact["quick_run"])

print("\nSUCCESS: Final binary model saved and loaded!")

In [ ]:
!python -u run_v3_pipeline.py --task multiclass --quick

In [ ]:
import joblib
from pathlib import Path

model_path = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3/v3_final_run/"
    "models/final_binary_quick.joblib"
)

assert model_path.exists(), "Model file missing!"

artifact = joblib.load(model_path)

print("Selected model:", artifact["model_name"])
print("Training dataset:", artifact["training_dataset"])
print("Total input genes:", len(artifact["common_genes"]))
print("Selected genes:", len(artifact["selected_genes"]))
print("Quick mode:", artifact["quick_run"])

print("\nModel successfully loaded!")

In [ ]:
%cd /content/ML_Project/ML_Project-main/external_validation_v3

In [ ]:
!python -u run_v3_pipeline.py --task multiclass --quick

In [ ]:
import joblib
from pathlib import Path

models_dir = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3/v3_final_run/models"
)

for task in ["binary", "multiclass"]:

    model_path = models_dir / f"final_{task}_quick.joblib"

    assert model_path.is_file(), f"Missing: {model_path}"

    artifact = joblib.load(model_path)

    print("\n" + "=" * 60)
    print(task.upper(), "MODEL")
    print("=" * 60)

    print("Selected algorithm:", artifact["model_name"])
    print("Training cohort:", artifact["training_dataset"])
    print("Input genes:", len(artifact["common_genes"]))
    print("Selected genes:", len(artifact["selected_genes"]))
    print("Training mode:", "Quick" if artifact["quick_run"] else "Full")

    print("\nPipeline steps:")
    print(list(artifact["model"].named_steps.keys()))

print("\nSUCCESS: Both V3 models verified!")

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

source = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3/v3_final_run"
)

destination = Path(
    "/content/drive/MyDrive/"
    "MASLD_Validation_Backup/v3_quick_final"
)

assert source.is_dir()

shutil.copytree(
    source,
    destination,
    dirs_exist_ok=True
)

print("SUCCESS: V3 quick models and results backed up!")

In [ ]:
!python -u run_v3_pipeline.py --task binary

In [ ]:
!python -u run_v3_pipeline.py --task multiclass

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

v3 = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3"
)

model_dir = v3 / "v3_final_run/models"
output_dir = v3 / "v3_final_run/external_evaluation"
output_dir.mkdir(parents=True, exist_ok=True)

results = []

for task in ["binary", "multiclass"]:

    artifact = joblib.load(
        model_dir / f"final_{task}_full.joblib"
    )

    pipeline = artifact["model"]
    expected_genes = artifact["common_genes"]

    label_column = (
        "label_binary" if task == "binary"
        else "label_3class"
    )

    print(f"\n{'=' * 55}")
    print(f"TASK: {task.upper()}")
    print(f"MODEL: {artifact['model_name']}")
    print(f"{'=' * 55}")

    for dataset in ["GSE126848", "GSE135251"]:

        expression = pd.read_csv(
            v3 / "data" / f"{dataset}_common_rank.csv.gz",
            index_col=0
        )

        metadata = pd.read_csv(
            v3 / "data" / f"{dataset}_metadata_harmonized.csv",
            index_col=0
        )

        assert expression.index.is_unique
        assert expression.columns.is_unique
        assert metadata.index.is_unique

        missing_genes = set(expected_genes) - set(expression.index)

        assert not missing_genes, (
            f"{dataset}: missing {len(missing_genes)} genes"
        )

        samples = metadata.index.tolist()

        assert set(samples) == set(expression.columns), (
            f"{dataset}: sample mismatch"
        )

        # Use the exact gene order saved with the trained model.
        X_external = expression.loc[
            expected_genes, samples
        ].T.to_numpy(dtype=np.float32)

        y_true = metadata.loc[
            samples, label_column
        ].astype(int).to_numpy()

        assert np.isfinite(X_external).all()

        y_pred = pipeline.predict(X_external)
        probabilities = pipeline.predict_proba(X_external)

        labels = [0, 1] if task == "binary" else [0, 1, 2]

        metrics = {
            "task": task,
            "dataset": dataset,
            "model": artifact["model_name"],
            "samples": len(samples),
            "accuracy": accuracy_score(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(
                y_true, y_pred
            ),
            "macro_f1": f1_score(
                y_true, y_pred,
                average="macro",
                zero_division=0
            ),
            "majority_class_baseline_accuracy": (
                pd.Series(y_true).value_counts().max()
                / len(y_true)
            )
        }

        if task == "binary":
            metrics["roc_auc"] = roc_auc_score(
                y_true, probabilities[:, 1]
            )
        else:
            metrics["roc_auc"] = roc_auc_score(
                y_true,
                probabilities,
                multi_class="ovr",
                average="macro",
                labels=labels
            )

        results.append(metrics)

        print(f"\nDATASET: {dataset}")
        print("Samples:", len(samples))
        print("Accuracy:", round(metrics["accuracy"], 4))
        print(
            "Balanced accuracy:",
            round(metrics["balanced_accuracy"], 4)
        )
        print("Macro F1:", round(metrics["macro_f1"], 4))
        print("ROC-AUC:", round(metrics["roc_auc"], 4))
        print(
            "Majority baseline accuracy:",
            round(
                metrics["majority_class_baseline_accuracy"], 4
            )
        )

        print("\nConfusion matrix:")
        print(
            confusion_matrix(
                y_true, y_pred, labels=labels
            )
        )

        print("\nClassification report:")
        print(
            classification_report(
                y_true, y_pred,
                labels=labels,
                zero_division=0
            )
        )

        prediction_df = pd.DataFrame({
            "sample_id": samples,
            "true_label": y_true,
            "predicted_label": y_pred
        })

        for idx, class_label in enumerate(
            pipeline.classes_
        ):
            prediction_df[
                f"probability_class_{class_label}"
            ] = probabilities[:, idx]

        prediction_df.to_csv(
            output_dir / f"{dataset}_{task}_predictions.csv",
            index=False
        )

summary = pd.DataFrame(results)

summary.to_csv(
    output_dir / "v3_external_summary.csv",
    index=False
)

print("\n" + "=" * 65)
print("COMPLETE EXPLORATORY EXTERNAL EVALUATION")
print("=" * 65)

display(summary.round(4))

print("\nSUCCESS: Evaluation results saved!")

In [ ]:
import shutil
from pathlib import Path

source = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3/v3_final_run"
)

destination = Path(
    "/content/drive/MyDrive/"
    "MASLD_Validation_Backup/v3_full_results"
)

assert source.is_dir()
assert Path("/content/drive/MyDrive").is_dir()

shutil.copytree(
    source,
    destination,
    dirs_exist_ok=True
)

print("SUCCESS: Full V3 models and results backed up!")

In [ ]:
import pandas as pd
from pathlib import Path

evaluation_dir = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3/v3_final_run/external_evaluation"
)

for task in ["binary", "multiclass"]:
    for dataset in ["GSE126848", "GSE135251"]:

        file_path = evaluation_dir / f"{dataset}_{task}_predictions.csv"
        df = pd.read_csv(file_path)

        labels = [0, 1] if task == "binary" else [0, 1, 2]

        print("\n" + "=" * 60)
        print(f"{dataset} — {task.upper()}")
        print("=" * 60)

        print("\nActual label counts:")
        print(df["true_label"].value_counts().sort_index())

        print("\nPredicted label counts:")
        print(df["predicted_label"].value_counts().sort_index())

        probability_columns = [
            f"probability_class_{label}" for label in labels
        ]

        print("\nAverage predicted probabilities by actual class:")
        display(
            df.groupby("true_label")[probability_columns]
              .mean()
              .round(4)
        )

        print("\nPrediction agreement:")
        display(
            pd.crosstab(
                df["true_label"],
                df["predicted_label"],
                rownames=["Actual"],
                colnames=["Predicted"]
            )
        )

print("\nSUCCESS: Prediction diagnostics completed.")

In [ ]:
import joblib
from pathlib import Path

model_dir = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3/v3_final_run/models"
)

binary = joblib.load(model_dir / "final_binary_full.joblib")
multiclass = joblib.load(model_dir / "final_multiclass_full.joblib")

binary_genes = set(binary["selected_genes"])
multiclass_genes = set(multiclass["selected_genes"])

common = binary_genes & multiclass_genes
union = binary_genes | multiclass_genes

print("Binary selected genes:", len(binary_genes))
print("Multiclass selected genes:", len(multiclass_genes))
print("Genes shared by both models:", len(common))
print("Genes unique to binary:", len(binary_genes - common))
print("Genes unique to multiclass:", len(multiclass_genes - common))
print("Jaccard similarity:", round(len(common) / len(union), 4))

print("\nFirst 20 shared genes:")
print(sorted(common)[:20])

In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.feature_selection import VarianceThreshold

v3_dir = Path(
    "/content/ML_Project/ML_Project-main/external_validation_v3"
)

expression = pd.read_csv(
    v3_dir / "data/GSE48452_common_rank.csv.gz",
    index_col=0
)

metadata = pd.read_csv(
    v3_dir / "data/GSE48452_metadata_harmonized.csv",
    index_col=0
)

sample_ids = metadata.index.tolist()
gene_names = np.asarray(expression.index)

X_stability = expression[sample_ids].T.to_numpy(
    dtype=np.float32
)

splitter = StratifiedShuffleSplit(
    n_splits=30,
    test_size=0.20,
    random_state=42
)

results_dir = v3_dir / "v3_final_run/results"

for task, label_column in [
    ("binary", "label_binary"),
    ("multiclass", "label_3class")
]:

    y = metadata.loc[
        sample_ids, label_column
    ].astype(int).to_numpy()

    selection_counts = np.zeros(
        len(gene_names), dtype=int
    )

    for train_idx, _ in splitter.split(
        X_stability, y
    ):

        X_train = X_stability[train_idx]
        y_train = y[train_idx]

        variance = VarianceThreshold()
        X_filtered = variance.fit_transform(X_train)

        with np.errstate(divide="ignore", invalid="ignore"):
            selector = SelectKBest(
                score_func=f_classif,
                k=min(400, X_filtered.shape[1])
            )
            selector.fit(X_filtered, y_train)

        retained_indices = np.flatnonzero(
            variance.get_support()
        )

        selected_indices = retained_indices[
            selector.get_support()
        ]

        selection_counts[selected_indices] += 1

    stability = pd.DataFrame({
        "gene": gene_names,
        "selection_count": selection_counts,
        "selection_frequency": (
            selection_counts / 30
        )
    }).sort_values(
        ["selection_frequency", "gene"],
        ascending=[False, True]
    )

    output = results_dir / f"{task}_gene_stability.csv"

    stability.to_csv(output, index=False)

    print(f"\n{task.upper()} GENE STABILITY")
    print("Genes selected in every split:",
          (stability["selection_count"] == 30).sum())
    print("Genes selected in >= 80% of splits:",
          (stability["selection_frequency"] >= 0.8).sum())
    print("Genes selected in >= 50% of splits:",
          (stability["selection_frequency"] >= 0.5).sum())

    print("\nTop 10 most consistently selected genes:")
    display(stability.head(10))

print("\nSUCCESS: Gene-selection stability analysis completed!")

In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

v3 = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3"
)

model_dir = v3 / "v3_final_run/models"
output_dir = v3 / "v3_final_run/results"
output_dir.mkdir(parents=True, exist_ok=True)

datasets = {}

for cohort in ["GSE48452", "GSE126848", "GSE135251"]:
    datasets[cohort] = pd.read_csv(
        v3 / "data" / f"{cohort}_common_rank.csv.gz",
        index_col=0
    )

for task in ["binary", "multiclass"]:

    artifact = joblib.load(
        model_dir / f"final_{task}_full.joblib"
    )

    genes = artifact["selected_genes"]

    reference = datasets["GSE48452"].loc[genes]

    ref_mean = reference.mean(axis=1)

    rows = []

    for cohort in ["GSE126848", "GSE135251"]:

        external = datasets[cohort].loc[genes]

        external_mean = external.mean(axis=1)

        differences = (
            external_mean - ref_mean
        ).abs()

        correlation = ref_mean.corr(
            external_mean,
            method="spearman"
        )

        rows.append({
            "task": task,
            "external_dataset": cohort,
            "genes": len(genes),
            "mean_absolute_rank_difference": differences.mean(),
            "median_absolute_rank_difference": differences.median(),
            "fraction_genes_difference_gt_0.1": (
                differences > 0.1
            ).mean(),
            "spearman_mean_rank_correlation": correlation
        })

    result = pd.DataFrame(rows)

    print(f"\n{task.upper()} GENE-DISTRIBUTION COMPARISON")
    display(result.round(4))

    result.to_csv(
        output_dir / f"{task}_gene_distribution_shift.csv",
        index=False
    )

print("\nSUCCESS: Gene-distribution diagnostics completed!")

In [ ]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

v3 = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3"
)

output_dir = v3 / "v3_final_run/results"
output_dir.mkdir(parents=True, exist_ok=True)

cohorts = {}

for name in ["GSE48452", "GSE126848", "GSE135251"]:

    expr = pd.read_csv(
        v3 / "data" / f"{name}_common_rank.csv.gz",
        index_col=0
    )

    meta = pd.read_csv(
        v3 / "data" / f"{name}_metadata_harmonized.csv",
        index_col=0
    )

    assert expr.index.is_unique
    assert meta.index.is_unique
    assert set(meta.index) == set(expr.columns)

    cohorts[name] = (expr, meta)

class_names = {
    0: "Healthy",
    1: "MASLD",
    2: "MASH"
}

rows = []

for task in ["binary", "multiclass"]:

    artifact = joblib.load(
        v3 / "v3_final_run/models" /
        f"final_{task}_full.joblib"
    )

    genes = artifact["selected_genes"]

    train_expr, train_meta = cohorts["GSE48452"]

    for class_id, class_name in class_names.items():

        train_samples = train_meta.index[
            train_meta["label_3class"].astype(int) == class_id
        ].tolist()

        train_mean = train_expr.loc[
            genes, train_samples
        ].mean(axis=1)

        for external_name in ["GSE126848", "GSE135251"]:

            ext_expr, ext_meta = cohorts[external_name]

            ext_samples = ext_meta.index[
                ext_meta["label_3class"].astype(int) == class_id
            ].tolist()

            ext_mean = ext_expr.loc[
                genes, ext_samples
            ].mean(axis=1)

            difference = (ext_mean - train_mean).abs()

            rows.append({
                "task": task,
                "class": class_name,
                "external_dataset": external_name,
                "training_samples": len(train_samples),
                "external_samples": len(ext_samples),
                "mean_absolute_rank_difference": difference.mean(),
                "median_absolute_rank_difference": difference.median(),
                "fraction_shift_gt_0.1": (difference > 0.1).mean(),
                "spearman_correlation": train_mean.corr(
                    ext_mean, method="spearman"
                )
            })

results = pd.DataFrame(rows)

display(results.round(4))

results.to_csv(
    output_dir / "class_specific_rank_shift.csv",
    index=False
)

print("SUCCESS: Class-specific rank comparison completed!")

In [ ]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

v3 = Path(
    "/content/ML_Project/ML_Project-main/external_validation_v3"
)

output_dir = v3 / "v3_final_run/results"
output_dir.mkdir(parents=True, exist_ok=True)

cohorts = {}

for cohort in ["GSE48452", "GSE126848", "GSE135251"]:

    expr = pd.read_csv(
        v3 / "data" / f"{cohort}_common_rank.csv.gz",
        index_col=0
    )

    meta = pd.read_csv(
        v3 / "data" / f"{cohort}_metadata_harmonized.csv",
        index_col=0
    )

    assert set(meta.index) == set(expr.columns)

    cohorts[cohort] = (expr, meta)


def disease_contrast(cohort, genes, disease_class):

    expr, meta = cohorts[cohort]

    healthy_ids = meta.index[
        meta["label_3class"].astype(int) == 0
    ].tolist()

    disease_ids = meta.index[
        meta["label_3class"].astype(int) == disease_class
    ].tolist()

    healthy_mean = expr.loc[
        genes, healthy_ids
    ].mean(axis=1)

    disease_mean = expr.loc[
        genes, disease_ids
    ].mean(axis=1)

    return disease_mean - healthy_mean


rows = []

for task in ["binary", "multiclass"]:

    artifact = joblib.load(
        v3 / "v3_final_run/models" /
        f"final_{task}_full.joblib"
    )

    genes = artifact["selected_genes"]

    for disease_class, disease_name in [
        (1, "MASLD_vs_Healthy"),
        (2, "MASH_vs_Healthy")
    ]:

        train_effect = disease_contrast(
            "GSE48452", genes, disease_class
        )

        for external in ["GSE126848", "GSE135251"]:

            external_effect = disease_contrast(
                external, genes, disease_class
            )

            # Exclude very small effects from direction testing.
            # This is an exploratory threshold, not a
            # statistical significance cutoff.
            meaningful = (
                (train_effect.abs() >= 0.05) &
                (external_effect.abs() >= 0.05)
            )

            compared = meaningful.sum()

            if compared > 0:
                same_direction = (
                    np.sign(train_effect[meaningful]) ==
                    np.sign(external_effect[meaningful])
                ).mean()
            else:
                same_direction = np.nan

            rows.append({
                "task": task,
                "contrast": disease_name,
                "external_dataset": external,
                "total_selected_genes": len(genes),
                "genes_with_effect_ge_0.05_in_both": int(compared),
                "same_direction_fraction": same_direction,
                "spearman_effect_correlation": train_effect.corr(
                    external_effect,
                    method="spearman"
                ),
                "mean_absolute_effect_difference": (
                    train_effect - external_effect
                ).abs().mean()
            })

results = pd.DataFrame(rows)

print("\nDISEASE-SIGNAL CONSISTENCY RESULTS")
display(results.round(4))

results.to_csv(
    output_dir / "disease_signal_consistency.csv",
    index=False
)

print("\nSUCCESS: Disease-signal analysis completed!")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

v3 = Path(
    "/content/ML_Project/ML_Project-main/external_validation_v3"
)

cohorts = ["GSE48452", "GSE126848", "GSE135251"]

for cohort in cohorts:

    expr = pd.read_csv(
        v3 / "data" / f"{cohort}_common_rank.csv.gz",
        index_col=0
    )

    meta = pd.read_csv(
        v3 / "data" / f"{cohort}_metadata_harmonized.csv",
        index_col=0
    )

    print("\n" + "=" * 60)
    print(cohort)
    print("=" * 60)

    print("Expression shape:", expr.shape)
    print("Metadata shape:", meta.shape)

    print("Duplicate gene names:", expr.index.duplicated().sum())
    print("Duplicate sample IDs:", expr.columns.duplicated().sum())

    print(
        "All sample IDs match:",
        set(expr.columns) == set(meta.index)
    )

    print(
        "Missing expression values:",
        int(expr.isna().sum().sum())
    )

    values = expr.to_numpy(dtype=np.float32)

    print("Minimum rank:", round(float(values.min()), 6))
    print("Maximum rank:", round(float(values.max()), 6))

    print("\nClinical label counts:")
    print(
        meta["label_3class"]
        .value_counts()
        .sort_index()
    )

    print("\nMetadata columns:")
    print(meta.columns.tolist())

    print("\nFirst three sample metadata records:")
    display(meta.head(3))

print("\nSUCCESS: Data integrity audit completed.")

In [ ]:
import pandas as pd
from pathlib import Path

v3 = Path(
    "/content/ML_Project/ML_Project-main/external_validation_v3"
)

output_dir = v3 / "v3_final_run/results"
output_dir.mkdir(parents=True, exist_ok=True)

columns_to_check = [
    "Sample_title",
    "Sample_source_name_ch1",
    "Sample_characteristics_ch1",
    "Sample_description",
    "MASLD_Label",
    "label",
    "label_3class",
    "label_binary"
]

for cohort in ["GSE48452", "GSE126848", "GSE135251"]:

    meta = pd.read_csv(
        v3 / "data" / f"{cohort}_metadata_harmonized.csv",
        index_col=0
    )

    available = [
        c for c in columns_to_check if c in meta.columns
    ]

    audit = meta[available].copy()
    audit.insert(0, "GSM_ID", audit.index)

    path = output_dir / f"{cohort}_label_audit.csv"
    audit.to_csv(path, index=False)

    print("\n" + "=" * 65)
    print(cohort, "— LABEL AUDIT")
    print("=" * 65)

    for label_value, group in audit.groupby("label_3class"):

        print(f"\nCLASS {label_value}: {len(group)} samples")

        display(
            group.head(5).style.set_properties(
                **{"white-space": "pre-wrap"}
            )
        )

    print("Saved:", path)

print("\nSUCCESS: Clinical label audit tables created!")

In [ ]:
import pandas as pd
import re
from pathlib import Path

v3 = Path(
    "/content/ML_Project/ML_Project-main/external_validation_v3"
)

output_dir = v3 / "v3_final_run/results"
output_dir.mkdir(parents=True, exist_ok=True)

def extract_field(text, field):
    match = re.search(
        rf"(?:^|\|)\s*{re.escape(field)}\s*:\s*([^|]+)",
        str(text),
        flags=re.IGNORECASE
    )
    return match.group(1).strip() if match else None


def source_label(cohort, row):
    characteristics = row.get(
        "Sample_characteristics_ch1", ""
    )

    if cohort == "GSE48452":
        group = extract_field(characteristics, "group")

        if group:
            group = group.strip().lower()

            if group == "control":
                return 0, group
            if group == "steatosis":
                return 1, group
            if group == "nash":
                return 2, group

        return None, group

    if cohort == "GSE126848":
        disease = extract_field(characteristics, "disease")

        if disease:
            disease = disease.strip().lower()

            if disease in ["healthy", "control"]:
                return 0, disease
            if disease in ["nafld", "nafl"]:
                return 1, disease
            if disease == "nash":
                return 2, disease

        return None, disease

    if cohort == "GSE135251":
        group = extract_field(
            characteristics, "group in paper"
        )

        if group:
            group = group.strip().lower()

            if group == "control":
                return 0, group
            if group == "nafl":
                return 1, group
            if group.startswith("nash"):
                return 2, group

        return None, group

    return None, None


records = []

for cohort in ["GSE48452", "GSE126848", "GSE135251"]:

    meta = pd.read_csv(
        v3 / "data" / f"{cohort}_metadata_harmonized.csv",
        index_col=0
    )

    for sample_id, row in meta.iterrows():

        expected, original_group = source_label(
            cohort, row
        )

        assigned = int(row["label_3class"])

        if expected is None:
            status = "REVIEW_UNRECOGNIZED"
        elif expected != assigned:
            status = "MISMATCH"
        else:
            status = "MATCH"

        records.append({
            "dataset": cohort,
            "sample_id": sample_id,
            "original_group": original_group,
            "expected_class": expected,
            "assigned_class": assigned,
            "status": status
        })

audit = pd.DataFrame(records)

audit.to_csv(
    output_dir / "all_samples_label_mapping_audit.csv",
    index=False
)

print("TOTAL SAMPLES:", len(audit))
print("\nAUDIT RESULTS:")
print(audit["status"].value_counts())

print("\nRESULTS BY DATASET:")
display(
    pd.crosstab(audit["dataset"], audit["status"])
)

problems = audit[
    audit["status"] != "MATCH"
]

print("\nSAMPLES REQUIRING REVIEW:")
display(problems)

print("\nAudit file saved successfully.")

In [ ]:
import pandas as pd
from pathlib import Path

v3 = Path(
    "/content/ML_Project/ML_Project-main/external_validation_v3"
)

output_dir = v3 / "v3_final_run/results"

audit = pd.read_csv(
    output_dir / "all_samples_label_mapping_audit.csv"
)

unrecognized = audit[
    audit["status"] == "REVIEW_UNRECOGNIZED"
]

records = []

for cohort in ["GSE48452", "GSE126848"]:

    meta = pd.read_csv(
        v3 / "data" / f"{cohort}_metadata_harmonized.csv",
        index_col=0
    )

    ids = unrecognized.loc[
        unrecognized["dataset"] == cohort,
        "sample_id"
    ].tolist()

    for sample_id in ids:

        row = meta.loc[sample_id]

        records.append({
            "dataset": cohort,
            "sample_id": sample_id,
            "sample_title": row.get("Sample_title"),
            "source_name": row.get("Sample_source_name_ch1"),
            "characteristics": row.get(
                "Sample_characteristics_ch1"
            ),
            "assigned_label": row["label_3class"]
        })

review = pd.DataFrame(records)

review.to_csv(
    output_dir / "unrecognized_control_review.csv",
    index=False
)

print("SAMPLES REQUIRING SOURCE VERIFICATION:", len(review))

for cohort, group in review.groupby("dataset"):

    print("\n", cohort)
    print("Number of samples:", len(group))

    display(
        group[
            ["sample_id", "sample_title",
             "characteristics", "assigned_label"]
        ].head(10)
    )

print("\nReview CSV saved successfully.")

In [ ]:
import pandas as pd
from pathlib import Path

results_dir = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3/v3_final_run/results"
)

audit = pd.read_csv(
    results_dir / "all_samples_label_mapping_audit.csv"
)

# Recognize study-defined non-NAFLD control groups.
corrections = {
    ("GSE48452", "healthy obese"): 0,
    ("GSE126848", "obese"): 0
}

for (dataset, group), expected_class in corrections.items():

    mask = (
        (audit["dataset"] == dataset) &
        (audit["original_group"].str.lower().str.strip() == group) &
        (audit["status"] == "REVIEW_UNRECOGNIZED")
    )

    audit.loc[mask, "expected_class"] = expected_class

audit["status"] = "MISMATCH"

audit.loc[
    audit["expected_class"].isna(), "status"
] = "REVIEW_UNRECOGNIZED"

audit.loc[
    audit["expected_class"].notna() &
    (audit["expected_class"] == audit["assigned_class"]),
    "status"
] = "MATCH"

print("FINAL SOURCE-GROUP MAPPING AUDIT")
print(audit["status"].value_counts())

print("\nBY DATASET")
display(pd.crosstab(audit["dataset"], audit["status"]))

audit.to_csv(
    results_dir / "all_samples_label_mapping_audit_verified.csv",
    index=False
)

assert (audit["status"] == "MATCH").all(), (
    "Some records still require review."
)

print("\nSUCCESS: All source-group mappings verified.")

In [ ]:
import pandas as pd
from pathlib import Path

v3 = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3"
)

meta = pd.read_csv(
    v3 / "data/GSE126848_metadata_harmonized.csv",
    index_col=0,
    dtype={"Sample_description": str}
)

mapping = pd.DataFrame({
    "GSM_ID": meta.index,
    "Sample_title": meta["Sample_title"].values,
    "Original_patient_ID": meta["Sample_description"].values,
})

mapping["Normalized_patient_ID"] = (
    mapping["Original_patient_ID"]
    .str.strip()
    .str.zfill(4)
)

print("TOTAL SAMPLES:", len(mapping))
print(
    "UNIQUE NORMALIZED PATIENT IDS:",
    mapping["Normalized_patient_ID"].nunique()
)
print(
    "MISSING PATIENT IDS:",
    mapping["Original_patient_ID"].isna().sum()
)
print(
    "DUPLICATED NORMALIZED IDS:",
    mapping["Normalized_patient_ID"].duplicated().sum()
)

print("\nFIRST 15 PATIENT MAPPINGS")
display(mapping.head(15))

output = (
    v3 / "v3_final_run/results/"
    "GSE126848_patient_ID_mapping_audit.csv"
)

mapping.to_csv(output, index=False)

print("\nMapping audit saved.")

In [ ]:
from pathlib import Path
import pandas as pd
import gzip

project = Path(
    "/content/ML_Project/ML_Project-main"
)

# Locate the original raw count matrix.
matches = list(
    project.rglob("GSE126848_Gene_counts_raw.txt.gz")
)

if not matches:
    print("ERROR: Original GSE126848 count matrix not found.")
else:
    raw_path = matches[0]

    print("Raw count matrix:", raw_path)

    # Read only the header to avoid loading the full matrix.
    with gzip.open(raw_path, "rt") as f:
        header = f.readline().strip().split("\t")

    print("\nFirst 10 raw matrix columns:")
    print(header[:10])

    # First column usually contains gene identifiers.
    raw_ids = [
        str(x).strip().zfill(4)
        for x in header[1:]
    ]

    mapping_ids = (
        mapping["Normalized_patient_ID"]
        .astype(str)
        .str.strip()
        .str.zfill(4)
        .tolist()
    )

    raw_set = set(raw_ids)
    mapping_set = set(mapping_ids)

    print("\nRAW MATRIX MATCHING RESULTS")
    print("Raw sample columns:", len(raw_ids))
    print("Unique raw IDs:", len(raw_set))
    print("Metadata patient IDs:", len(mapping_ids))

    print(
        "Matched metadata IDs:",
        len(mapping_set & raw_set)
    )

    print(
        "Missing from raw matrix:",
        sorted(mapping_set - raw_set)
    )

    print(
        "Extra raw matrix IDs:",
        sorted(raw_set - mapping_set)
    )

    print(
        "Duplicate raw IDs:",
        len(raw_ids) - len(raw_set)
    )

    if (
        len(raw_ids) == 57
        and len(raw_set) == 57
        and mapping_set == raw_set
    ):
        print(
            "\nSUCCESS: All 57 patient IDs match "
            "the original count-matrix headers!"
        )
    else:
        print(
            "\nREVIEW REQUIRED: The raw matrix and "
            "metadata IDs do not fully match."
        )

In [ ]:
from pathlib import Path

project = Path("/content/ML_Project/ML_Project-main")

print("SEARCHING FOR GSE126848 INTERMEDIATE FILES\n")

for path in sorted(project.rglob("*GSE126848*")):

    if path.is_file():
        print(
            path.relative_to(project),
            "| Size:",
            round(path.stat().st_size / (1024**2), 2),
            "MB"
        )

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

project = Path("/content/ML_Project/ML_Project-main")

v2 = project / "external_validation_v2"
v3 = project / "external_validation_v3"

experimental_path = (
    v2 / "experiments/common_gene_ranking/"
    "GSE126848_common_rank.csv.gz"
)

intermediate_path = (
    v2 / "gene_level/GSE126848_gene_rank.csv.gz"
)

final_path = (
    v3 / "data/GSE126848_common_rank.csv.gz"
)

print("Loading matrices...", flush=True)

experimental = pd.read_csv(
    experimental_path, index_col=0
)

intermediate = pd.read_csv(
    intermediate_path, index_col=0
)

final = pd.read_csv(
    final_path, index_col=0
)

print("\nMATRIX SHAPES")
print("Experimental:", experimental.shape)
print("Intermediate:", intermediate.shape)
print("V3 final:", final.shape)

# Check 1: V2 experiment versus V3.
same_genes = experimental.index.equals(final.index)
same_samples = experimental.columns.equals(final.columns)

print("\nCHECK 1 — V2 EXPERIMENT VS V3")
print("Same gene order:", same_genes)
print("Same sample order:", same_samples)

if same_genes and same_samples:
    difference = np.abs(
        experimental.to_numpy() - final.to_numpy()
    )

    print("Maximum absolute difference:", difference.max())
    print(
        "Identical within numerical tolerance:",
        np.allclose(
            experimental.to_numpy(),
            final.to_numpy(),
            rtol=1e-7,
            atol=1e-9
        )
    )

# Check 2: Reproduce the common-gene ranking.
print("\nCHECK 2 — RECOMPUTE COMMON-GENE RANKS")

assert final.index.isin(intermediate.index).all(), (
    "Some V3 genes are missing from the intermediate matrix."
)

assert final.columns.isin(intermediate.columns).all(), (
    "Some V3 samples are missing from the intermediate matrix."
)

aligned_intermediate = intermediate.loc[
    final.index, final.columns
]

recomputed = aligned_intermediate.rank(
    axis=0,
    method="average",
    pct=True
)

rank_difference = np.abs(
    recomputed.to_numpy() - final.to_numpy()
)

print(
    "Maximum absolute rank difference:",
    rank_difference.max()
)

print(
    "Mean absolute rank difference:",
    rank_difference.mean()
)

print(
    "Recomputed ranks match V3:",
    np.allclose(
        recomputed.to_numpy(),
        final.to_numpy(),
        rtol=1e-7,
        atol=1e-9
    )
)

print("\nCHECK COMPLETED.")

In [ ]:
import pandas as pd
from pathlib import Path

project = Path("/content/ML_Project/ML_Project-main")

mapping_path = (
    project /
    "external_validation_v2/cache/"
    "GSE126848_ensembl_to_symbol.csv"
)

assert mapping_path.is_file(), "Gene mapping file missing!"

gene_map = pd.read_csv(mapping_path)

print("GENE MAPPING FILE AUDIT")
print("=" * 55)

print("Shape:", gene_map.shape)
print("\nColumns:")
print(gene_map.columns.tolist())

print("\nFirst 10 records:")
display(gene_map.head(10))

print("\nMissing values by column:")
print(gene_map.isna().sum())

print("\nData types:")
print(gene_map.dtypes)

In [ ]:
import gzip
from pathlib import Path

raw_path = (
    project /
    "external_validation_v2/raw/GSE126848/"
    "GSE126848_Gene_counts_raw.txt.gz"
)

assert raw_path.is_file()

print("ORIGINAL RAW COUNT MATRIX")
print("=" * 55)

with gzip.open(raw_path, "rt") as f:
    for _ in range(8):
        print(f.readline().rstrip()[:200])

In [ ]:
import pandas as pd
from pathlib import Path

project = Path("/content/ML_Project/ML_Project-main")

mapping_path = (
    project /
    "external_validation_v2/cache/"
    "GSE126848_ensembl_to_symbol.csv"
)

raw_path = (
    project /
    "external_validation_v2/raw/GSE126848/"
    "GSE126848_Gene_counts_raw.txt.gz"
)

gene_map = pd.read_csv(mapping_path, dtype=str)

raw_gene_ids = pd.read_csv(
    raw_path,
    sep="\t",
    usecols=[0],
    dtype=str
).iloc[:, 0]

raw_gene_ids = raw_gene_ids.str.replace(
    r"\.\d+$", "", regex=True
)

mapping_ids = gene_map["source_id"].str.replace(
    r"\.\d+$", "", regex=True
)

print("GENE MAPPING QUALITY AUDIT")
print("=" * 60)

print("Raw gene rows:", len(raw_gene_ids))
print("Unique raw gene IDs:", raw_gene_ids.nunique())

print("Mapping entries:", len(gene_map))
print("Unique mapping IDs:", mapping_ids.nunique())
print("Unique mapped symbols:", gene_map["gene_symbol"].nunique())

print(
    "Duplicate source IDs:",
    mapping_ids.duplicated().sum()
)

print(
    "Duplicate gene symbol entries:",
    gene_map["gene_symbol"].duplicated().sum()
)

print(
    "Raw gene IDs with a mapping:",
    raw_gene_ids.isin(set(mapping_ids)).sum()
)

print(
    "Raw gene IDs without a mapping:",
    (~raw_gene_ids.isin(set(mapping_ids))).sum()
)

duplicates = gene_map[
    gene_map["gene_symbol"].duplicated(keep=False)
].sort_values("gene_symbol")

print("\nEXAMPLES OF DUPLICATE GENE SYMBOLS")
display(duplicates.head(20))

print("\nSUCCESS: Mapping coverage audit completed!")

In [ ]:
from pathlib import Path

script = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_pipeline.py"
)

assert script.is_file(), "External validation script not found"

lines = script.read_text(
    encoding="utf-8"
).splitlines()

keywords = [
    "groupby(",
    "gene_symbol",
    "ensembl",
    "normalize",
    "log1p",
    "rank(",
    "standardize_count"
]

print("GENE-PROCESSING CODE LOCATIONS")
print("=" * 60)

found = set()

for index, line in enumerate(lines):
    if any(word in line.lower() for word in keywords):
        found.add(index)

for index in sorted(found):
    start = max(0, index - 2)
    end = min(len(lines), index + 3)

    print(f"\n--- Line {index + 1} ---")

    for j in range(start, end):
        print(f"{j + 1}: {lines[j]}")

In [ ]:
from pathlib import Path

project = Path("/content/ML_Project/ML_Project-main")

script = project / "external_validation_pipeline.py"

lines = script.read_text(encoding="utf-8").splitlines()

sections = [
    (656, 710, "GENE MAPPING AND NORMALIZATION"),
    (751, 830, "SAMPLE-ID STANDARDIZATION"),
    (860, 900, "GSE126848 PROCESSING")
]

for start, end, heading in sections:

    print("\n" + "=" * 70)
    print(heading)
    print("=" * 70)

    for number in range(start, min(end, len(lines)) + 1):
        print(f"{number:4}: {lines[number - 1]}")

In [ ]:
import pandas as pd
from pathlib import Path

project = Path("/content/ML_Project/ML_Project-main")

mapping = pd.read_csv(
    project /
    "external_validation_v2/cache/"
    "GSE126848_ensembl_to_symbol.csv"
)

v3 = pd.read_csv(
    project /
    "external_validation_v3/data/"
    "GSE126848_common_rank.csv.gz",
    index_col=0
)

mapped_symbols = set(mapping["gene_symbol"])
v3_genes = set(v3.index.astype(str))

print("GENE COVERAGE VERIFICATION")
print("=" * 55)

print("Unique mapped symbols:", len(mapped_symbols))
print("V3 genes:", len(v3_genes))

missing = sorted(v3_genes - mapped_symbols)

print("V3 genes missing from mapping:", len(missing))
print("First 20 missing genes:", missing[:20])

if len(missing) == 0:
    print("\nPASS: All V3 genes exist in the mapping reference.")
else:
    print("\nREVIEW: Some V3 genes are absent from the mapping.")

In [ ]:
import ast
import numpy as np
import pandas as pd
from pathlib import Path

root = Path("/content/ML_Project/ML_Project-main")
v2 = root / "external_validation_v2"
v3 = root / "external_validation_v3"

script_path = root / "external_validation_pipeline.py"

raw_path = (
    v2 / "raw/GSE126848/"
    "GSE126848_Gene_counts_raw.txt.gz"
)

mapping_path = (
    v2 / "cache/GSE126848_ensembl_to_symbol.csv"
)

meta_path = (
    v3 / "data/GSE126848_metadata_harmonized.csv"
)

intermediate_path = (
    v2 / "gene_level/GSE126848_gene_rank.csv.gz"
)

final_path = (
    v3 / "data/GSE126848_common_rank.csv.gz"
)

# -------------------------------------------------------
# STEP 1: Read normalization constants from source code.
# -------------------------------------------------------

tree = ast.parse(script_path.read_text(encoding="utf-8"))

required = {
    "RNA_CPM_MIN",
    "RNA_MIN_SAMPLE_FRACTION"
}

constants = {}

for node in tree.body:
    if isinstance(node, ast.Assign):
        for target in node.targets:
            if isinstance(target, ast.Name):
                if target.id in required:
                    constants[target.id] = ast.literal_eval(
                        node.value
                    )

    elif isinstance(node, ast.AnnAssign):
        if isinstance(node.target, ast.Name):
            if node.target.id in required:
                constants[node.target.id] = ast.literal_eval(
                    node.value
                )

missing_constants = required - set(constants)

if missing_constants:
    raise ValueError(
        f"Unable to read source-code constants: "
        f"{missing_constants}"
    )

RNA_CPM_MIN = constants["RNA_CPM_MIN"]
RNA_MIN_SAMPLE_FRACTION = constants[
    "RNA_MIN_SAMPLE_FRACTION"
]

print("STEP 1 — ORIGINAL NORMALIZATION PARAMETERS")
print("RNA_CPM_MIN:", RNA_CPM_MIN)
print("RNA_MIN_SAMPLE_FRACTION:", RNA_MIN_SAMPLE_FRACTION)

# -------------------------------------------------------
# STEP 2: Load original counts and mapping.
# -------------------------------------------------------

raw = pd.read_csv(
    raw_path,
    sep="\t",
    index_col=0
)

gene_map = pd.read_csv(
    mapping_path,
    dtype=str
)

meta = pd.read_csv(
    meta_path,
    index_col=0,
    dtype={"Sample_description": str}
)

print("\nSTEP 2 — ORIGINAL INPUTS")
print("Raw counts:", raw.shape)
print("Mapping:", gene_map.shape)
print("Metadata:", meta.shape)

# -------------------------------------------------------
# STEP 3: Reproduce patient ID -> GSM conversion.
# -------------------------------------------------------

patient_ids = (
    meta["Sample_description"]
    .astype(str)
    .str.strip()
    .str.zfill(4)
)

assert patient_ids.is_unique
assert not patient_ids.isna().any()

patient_to_gsm = dict(zip(patient_ids, meta.index))

assert set(raw.columns) == set(patient_to_gsm)

raw = raw.rename(columns=patient_to_gsm)

# Preserve metadata sample order.
raw = raw.loc[:, meta.index]

assert raw.columns.equals(meta.index)

print("\nSTEP 3 — SAMPLE MAPPING")
print("Matched samples:", raw.shape[1])
print("Unique GSM columns:", raw.columns.nunique())

# -------------------------------------------------------
# STEP 4: Reproduce Ensembl -> gene symbol mapping.
# -------------------------------------------------------

raw.index = (
    raw.index.astype(str)
    .str.strip()
    .str.split(".")
    .str[0]
)

gene_map["source_id"] = (
    gene_map["source_id"]
    .astype(str)
    .str.strip()
    .str.split(".")
    .str[0]
)

symbol_lookup = (
    gene_map
    .set_index("source_id")["gene_symbol"]
)

symbols = raw.index.to_series().map(symbol_lookup)

keep = symbols.notna() & symbols.astype(str).ne("")

mapped = raw.loc[keep].copy()

mapped["gene_symbol"] = symbols.loc[keep].to_numpy()

# Exact aggregation rule from your original pipeline.
gene_counts = (
    mapped.groupby("gene_symbol")
    .sum(numeric_only=True)
    .clip(lower=0)
)

print("\nSTEP 4 — GENE AGGREGATION")
print("Mapped gene-symbol rows:", gene_counts.shape[0])
print("Samples:", gene_counts.shape[1])

# -------------------------------------------------------
# STEP 5: Reproduce original CPM normalization/filtering.
# -------------------------------------------------------

x = (
    gene_counts
    .astype(float)
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

assert (x.to_numpy() >= 0).all()

library_sizes = x.sum(axis=0)

assert (library_sizes > 0).all()

cpm = x.div(library_sizes, axis=1) * 1_000_000.0

min_n = max(
    2,
    int(np.ceil(
        RNA_MIN_SAMPLE_FRACTION * cpm.shape[1]
    ))
)

expressed = (
    (cpm >= RNA_CPM_MIN).sum(axis=1) >= min_n
)

cpm = cpm.loc[expressed]

logcpm = np.log2(cpm + 1.0)

print("\nSTEP 5 — CPM FILTERING")
print("Minimum required samples:", min_n)
print("Genes after filtering:", len(cpm))

# -------------------------------------------------------
# STEP 6: Reproduce intermediate gene ranks.
# -------------------------------------------------------

reconstructed_intermediate = (
    logcpm.rank(
        axis=0,
        method="average",
        pct=True
    ).astype(np.float32)
)

saved_intermediate = pd.read_csv(
    intermediate_path,
    index_col=0
)

same_genes = (
    set(reconstructed_intermediate.index)
    == set(saved_intermediate.index)
)

same_samples = (
    set(reconstructed_intermediate.columns)
    == set(saved_intermediate.columns)
)

print("\nSTEP 6 — INTERMEDIATE MATRIX AUDIT")
print("Same gene set:", same_genes)
print("Same sample set:", same_samples)

if not same_genes or not same_samples:
    print("REVIEW: Intermediate matrix structure differs.")
else:
    aligned = reconstructed_intermediate.loc[
        saved_intermediate.index,
        saved_intermediate.columns
    ]

    diff = np.abs(
        aligned.to_numpy(dtype=np.float64)
        - saved_intermediate.to_numpy(dtype=np.float64)
    )

    print("Maximum absolute difference:", diff.max())
    print("Mean absolute difference:", diff.mean())

    print(
        "Intermediate reconstruction matches:",
        np.allclose(
            aligned.to_numpy(),
            saved_intermediate.to_numpy(),
            rtol=1e-6,
            atol=1e-7
        )
    )

# -------------------------------------------------------
# STEP 7: Reconstruct final common-gene V3 ranks
# directly from the filtered logCPM values.
# -------------------------------------------------------

saved_v3 = pd.read_csv(
    final_path,
    index_col=0
)

assert saved_v3.index.isin(logcpm.index).all()
assert saved_v3.columns.isin(logcpm.columns).all()

common_logcpm = logcpm.loc[
    saved_v3.index,
    saved_v3.columns
]

reconstructed_v3 = (
    common_logcpm.rank(
        axis=0,
        method="average",
        pct=True
    ).astype(np.float32)
)

final_diff = np.abs(
    reconstructed_v3.to_numpy(dtype=np.float64)
    - saved_v3.to_numpy(dtype=np.float64)
)

final_match = np.allclose(
    reconstructed_v3.to_numpy(),
    saved_v3.to_numpy(),
    rtol=1e-6,
    atol=1e-7
)

print("\nSTEP 7 — FINAL V3 RECONSTRUCTION")
print("Reconstructed shape:", reconstructed_v3.shape)
print("Saved V3 shape:", saved_v3.shape)
print("Maximum difference:", final_diff.max())
print("Mean difference:", final_diff.mean())
print("Raw-to-V3 reconstruction matches:", final_match)

print("\nAUDIT COMPLETED — ORIGINAL FILES UNCHANGED")

In [ ]:
from pathlib import Path

project = Path("/content/ML_Project/ML_Project-main")

print("GSE135251 SOURCE AND INTERMEDIATE FILES")
print("=" * 65)

for path in sorted(project.rglob("*GSE135251*")):

    if path.is_file():
        print(
            path.relative_to(project),
            "|",
            round(path.stat().st_size / (1024**2), 2),
            "MB"
        )

In [ ]:
import tarfile
from pathlib import Path
from collections import Counter

project = Path("/content/ML_Project/ML_Project-main")

tar_path = (
    project /
    "external_validation_v2/raw/GSE135251/"
    "GSE135251_RAW.tar"
)

assert tar_path.is_file(), "Raw TAR archive not found!"

print("GSE135251 RAW ARCHIVE INSPECTION")
print("=" * 65)

with tarfile.open(tar_path, "r:*") as tar:

    members = [m for m in tar.getmembers() if m.isfile()]

    print("Total files:", len(members))

    print("\nFIRST 20 FILES:")
    for m in members[:20]:
        print(
            m.name,
            "|",
            round(m.size / (1024**2), 3),
            "MB"
        )

    extensions = Counter(
        Path(m.name).suffix.lower()
        for m in members
    )

    print("\nFILE EXTENSIONS:")
    print(dict(extensions))

    print("\nLAST 10 FILES:")
    for m in members[-10:]:
        print(m.name)

    total_mb = sum(m.size for m in members) / (1024**2)
    print("\nTotal uncompressed size (MB):", round(total_mb, 2))

print("\nARCHIVE INSPECTION COMPLETED")

In [ ]:
from pathlib import Path

script_path = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_pipeline.py"
)

lines = script_path.read_text(
    encoding="utf-8"
).splitlines()

print("GSE135251 RAW-DATA PROCESSING CODE")
print("=" * 70)

start_line = 900
end_line = 1030

for number in range(start_line, min(end_line, len(lines)) + 1):
    print(f"{number:4}: {lines[number - 1]}")

In [ ]:
import ast
import gzip
import io
import tarfile
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

root = Path("/content/ML_Project/ML_Project-main")
v2 = root / "external_validation_v2"
v3 = root / "external_validation_v3"

tar_path = v2 / "raw/GSE135251/GSE135251_RAW.tar"
mapping_path = v2 / "cache/GSE135251_ensembl_to_symbol.csv"
meta_path = v3 / "data/GSE135251_metadata_harmonized.csv"
intermediate_path = v2 / "gene_level/GSE135251_gene_rank.csv.gz"
final_path = v3 / "data/GSE135251_common_rank.csv.gz"
script_path = root / "external_validation_pipeline.py"

# ---------------------------------------------------------
# STEP 1 — Read actual filtering parameters.
# ---------------------------------------------------------

tree = ast.parse(script_path.read_text(encoding="utf-8"))

constants = {}

for node in tree.body:
    if isinstance(node, ast.Assign):
        for target in node.targets:
            if isinstance(target, ast.Name):
                if target.id in {
                    "RNA_CPM_MIN",
                    "RNA_MIN_SAMPLE_FRACTION"
                }:
                    constants[target.id] = ast.literal_eval(node.value)

    elif isinstance(node, ast.AnnAssign):
        if isinstance(node.target, ast.Name):
            if node.target.id in {
                "RNA_CPM_MIN",
                "RNA_MIN_SAMPLE_FRACTION"
            }:
                constants[node.target.id] = ast.literal_eval(node.value)

cpm_min = constants["RNA_CPM_MIN"]
sample_fraction = constants["RNA_MIN_SAMPLE_FRACTION"]

print("STEP 1 — NORMALIZATION SETTINGS")
print("Minimum CPM:", cpm_min)
print("Minimum sample fraction:", sample_fraction)

# ---------------------------------------------------------
# STEP 2 — Build the source filename -> GSM lookup.
# ---------------------------------------------------------

meta = pd.read_csv(meta_path, index_col=0)

supp_cols = [
    c for c in meta.columns
    if "supplementary_file" in c.lower()
]

file_to_gsm = {}
ambiguous_names = []

for gsm, row in meta.iterrows():

    for col in supp_cols:

        value = str(row.get(col, ""))

        for token in value.split(" | "):

            filename = Path(token.strip()).name

            if not filename or filename.lower() in {"none", "nan"}:
                continue

            if (
                filename in file_to_gsm
                and file_to_gsm[filename] != gsm
            ):
                ambiguous_names.append(filename)

            file_to_gsm[filename] = gsm

assert not ambiguous_names, (
    f"Ambiguous supplementary filenames: {ambiguous_names[:10]}"
)

print("\nSTEP 2 — METADATA SOURCE MAPPING")
print("Metadata samples:", len(meta))
print("Supplementary filenames:", len(file_to_gsm))
print("Duplicate assignments:", len(ambiguous_names))

# ---------------------------------------------------------
# STEP 3 — Read all count files directly from TAR.
# ---------------------------------------------------------

series = {}
matching_records = []

with tarfile.open(tar_path, "r:*") as tar:

    members = [
        member for member in tar.getmembers()
        if member.isfile()
        and member.name.endswith(".counts.txt.gz")
    ]

    print("\nSTEP 3 — RAW SAMPLE FILES")
    print("Count files in archive:", len(members))

    assert len(members) == 216, (
        f"Expected 216 count files, found {len(members)}"
    )

    for number, member in enumerate(members, start=1):

        filename = Path(member.name).name
        gsm = file_to_gsm.get(filename)

        assert gsm is not None, (
            f"No metadata mapping for {filename}"
        )

        assert gsm not in series, (
            f"Two count files map to the same sample: {gsm}"
        )

        source = tar.extractfile(member)
        assert source is not None

        with source:
            with gzip.GzipFile(fileobj=source, mode="rb") as gz:
                with io.TextIOWrapper(
                    gz, encoding="utf-8", errors="replace"
                ) as stream:
                    df = pd.read_csv(
                        stream,
                        sep=None,
                        engine="python",
                        header=None,
                        comment="#"
                    )

        assert df.shape[1] >= 2, (
            f"Invalid count file: {filename}"
        )

        gene = df.iloc[:, 0].astype(str)
        values = pd.to_numeric(
            df.iloc[:, -1], errors="coerce"
        )

        valid = values.notna()
        gene = gene[valid]
        values = values[valid]

        not_special = ~gene.str.startswith("__")

        sample_counts = pd.Series(
            values[not_special].values,
            index=gene[not_special].values,
            dtype=float
        )

        sample_counts = (
            sample_counts.groupby(level=0).sum()
        )

        series[gsm] = sample_counts

        matching_records.append({
            "GSM_ID": gsm,
            "Raw_filename": filename,
            "Gene_rows": len(sample_counts)
        })

        if number % 40 == 0 or number == len(members):
            print(f"Processed {number}/{len(members)} files")

assert set(series) == set(meta.index), (
    "Raw count samples and metadata GSM IDs differ."
)

print("\nRaw samples matched:", len(series))
print("Every metadata GSM represented:", True)

# ---------------------------------------------------------
# STEP 4 — Combine raw count files into one matrix.
# ---------------------------------------------------------

counts = pd.concat(series, axis=1).fillna(0.0)
counts = counts.loc[:, meta.index]

print("\nSTEP 4 — COMBINED RAW COUNTS")
print("Raw count matrix:", counts.shape)
print("Missing count values:", int(counts.isna().sum().sum()))
print("Duplicate GSM columns:", counts.columns.duplicated().sum())

# ---------------------------------------------------------
# STEP 5 — Map Ensembl IDs and aggregate gene counts.
# ---------------------------------------------------------

gene_map = pd.read_csv(mapping_path, dtype=str)

counts.index = (
    counts.index.astype(str)
    .str.strip()
    .str.split(".").str[0]
)

gene_map["source_id"] = (
    gene_map["source_id"]
    .astype(str)
    .str.strip()
    .str.split(".").str[0]
)

# The original pipeline requires one gene symbol per source ID.
assert not gene_map["source_id"].duplicated().any(), (
    "Duplicate source IDs in cached gene mapping."
)

lookup = gene_map.set_index(
    "source_id"
)["gene_symbol"]

symbols = counts.index.to_series().map(lookup)

valid = symbols.notna() & symbols.astype(str).ne("")

mapped = counts.loc[valid].copy()
mapped["gene_symbol"] = symbols.loc[valid].to_numpy()

gene_counts = (
    mapped.groupby("gene_symbol")
    .sum(numeric_only=True)
    .clip(lower=0)
)

print("\nSTEP 5 — GENE IDENTIFIER MAPPING")
print("Raw gene rows:", len(counts))
print("Mapped gene symbols:", len(gene_counts))
print("Unmapped rows:", int((~valid).sum()))

# ---------------------------------------------------------
# STEP 6 — Reproduce CPM filtering and normalization.
# ---------------------------------------------------------

x = (
    gene_counts.astype(float)
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

assert (x.to_numpy() >= 0).all()

library_sizes = x.sum(axis=0)
assert (library_sizes > 0).all()

cpm = x.div(library_sizes, axis=1) * 1_000_000.0

min_n = max(
    2,
    int(np.ceil(sample_fraction * cpm.shape[1]))
)

keep = (cpm >= cpm_min).sum(axis=1) >= min_n

cpm = cpm.loc[keep]
logcpm = np.log2(cpm + 1.0)

print("\nSTEP 6 — NORMALIZATION")
print("Minimum required samples:", min_n)
print("Genes surviving filter:", len(logcpm))

# ---------------------------------------------------------
# STEP 7 — Verify saved intermediate gene ranks.
# ---------------------------------------------------------

reconstructed_intermediate = (
    logcpm.rank(
        axis=0,
        method="average",
        pct=True
    ).astype(np.float32)
)

saved_intermediate = pd.read_csv(
    intermediate_path,
    index_col=0
)

same_gene_set = (
    set(reconstructed_intermediate.index)
    == set(saved_intermediate.index)
)

same_sample_set = (
    set(reconstructed_intermediate.columns)
    == set(saved_intermediate.columns)
)

print("\nSTEP 7 — INTERMEDIATE RECONSTRUCTION")
print("Reconstructed:", reconstructed_intermediate.shape)
print("Saved:", saved_intermediate.shape)
print("Same gene set:", same_gene_set)
print("Same sample set:", same_sample_set)

intermediate_match = False

if same_gene_set and same_sample_set:

    aligned = reconstructed_intermediate.loc[
        saved_intermediate.index,
        saved_intermediate.columns
    ]

    diff = np.abs(
        aligned.to_numpy(dtype=np.float64)
        - saved_intermediate.to_numpy(dtype=np.float64)
    )

    intermediate_match = np.allclose(
        aligned.to_numpy(),
        saved_intermediate.to_numpy(),
        rtol=1e-6,
        atol=1e-7
    )

    print("Maximum difference:", float(diff.max()))
    print("Mean difference:", float(diff.mean()))

print("Intermediate reconstruction matches:", intermediate_match)

# ---------------------------------------------------------
# STEP 8 — Reconstruct final V3 common-gene ranks.
# ---------------------------------------------------------

saved_v3 = pd.read_csv(final_path, index_col=0)

assert saved_v3.index.isin(logcpm.index).all()
assert saved_v3.columns.isin(logcpm.columns).all()

common_logcpm = logcpm.loc[
    saved_v3.index,
    saved_v3.columns
]

reconstructed_v3 = (
    common_logcpm.rank(
        axis=0,
        method="average",
        pct=True
    ).astype(np.float32)
)

final_diff = np.abs(
    reconstructed_v3.to_numpy(dtype=np.float64)
    - saved_v3.to_numpy(dtype=np.float64)
)

final_match = np.allclose(
    reconstructed_v3.to_numpy(),
    saved_v3.to_numpy(),
    rtol=1e-6,
    atol=1e-7
)

print("\nSTEP 8 — FINAL V3 RECONSTRUCTION")
print("Reconstructed shape:", reconstructed_v3.shape)
print("Saved V3 shape:", saved_v3.shape)
print("Maximum difference:", float(final_diff.max()))
print("Mean difference:", float(final_diff.mean()))
print("Raw-to-V3 reconstruction matches:", final_match)

# ---------------------------------------------------------
# STEP 9 — Save audit evidence (not model data).
# ---------------------------------------------------------

audit_dir = v3 / "v3_final_run/results"
audit_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame(matching_records).to_csv(
    audit_dir / "GSE135251_raw_file_to_GSM_audit.csv",
    index=False
)

print("\nFINAL AUDIT STATUS")

if intermediate_match and final_match:
    print("PASS: GSE135251 raw-to-V3 reconstruction verified.")
else:
    print("REVIEW REQUIRED: A reconstruction discrepancy remains.")

print("Existing V3 models and input matrices unchanged.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

v3 = Path(
    "/content/ML_Project/ML_Project-main/external_validation_v3"
)

datasets = ["GSE48452", "GSE126848", "GSE135251"]

X_parts = []
metadata_parts = []

for cohort in datasets:

    expr = pd.read_csv(
        v3 / "data" / f"{cohort}_common_rank.csv.gz",
        index_col=0
    )

    meta = pd.read_csv(
        v3 / "data" / f"{cohort}_metadata_harmonized.csv",
        index_col=0
    )

    X_cohort = expr.loc[:, meta.index].T

    X_parts.append(X_cohort)

    metadata_parts.append(
        pd.DataFrame({
            "dataset": cohort,
            "class": meta["label_3class"].values
        }, index=meta.index)
    )

X = pd.concat(X_parts, axis=0)
metadata = pd.concat(metadata_parts, axis=0)

assert X.shape == (346, 10152)

# Exploratory visualization only:
# scaling across all cohorts is NOT used for model training.
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2, random_state=42)

coordinates = pca.fit_transform(X_scaled)

plot_df = pd.DataFrame(
    coordinates,
    columns=["PC1", "PC2"],
    index=X.index
).join(metadata)

print("PCA explained variance:")
print("PC1:", round(pca.explained_variance_ratio_[0] * 100, 2), "%")
print("PC2:", round(pca.explained_variance_ratio_[1] * 100, 2), "%")

# PCA by dataset.
plt.figure(figsize=(9, 6))

for cohort in datasets:

    mask = plot_df["dataset"] == cohort

    plt.scatter(
        plot_df.loc[mask, "PC1"],
        plot_df.loc[mask, "PC2"],
        label=cohort,
        alpha=0.7,
        s=25
    )

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA — Samples Colored by Dataset")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

# PCA by disease class.
class_names = {
    0: "Healthy",
    1: "MASLD",
    2: "MASH"
}

plt.figure(figsize=(9, 6))

for label, name in class_names.items():

    mask = plot_df["class"] == label

    plt.scatter(
        plot_df.loc[mask, "PC1"],
        plot_df.loc[mask, "PC2"],
        label=name,
        alpha=0.7,
        s=25
    )

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA — Samples Colored by Clinical Class")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 06 — Additional model experiments


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import files

# Create output folder
output_dir = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3/v3_final_run/figures"
)
output_dir.mkdir(parents=True, exist_ok=True)

# -------- PLOT 1: PCA BY DATASET --------

plt.figure(figsize=(10, 7))

for cohort in ["GSE48452", "GSE126848", "GSE135251"]:
    subset = plot_df[plot_df["dataset"] == cohort]

    plt.scatter(
        subset["PC1"],
        subset["PC2"],
        label=cohort,
        alpha=0.7,
        s=35
    )

plt.xlabel("PC1 (28.63% variance)")
plt.ylabel("PC2 (16.87% variance)")
plt.title("PCA of Gene Expression by Dataset")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()

dataset_file = output_dir / "PCA_by_Dataset.png"
plt.savefig(dataset_file, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# -------- PLOT 2: PCA BY DISEASE CLASS --------

class_names = {
    0: "Healthy",
    1: "MASLD",
    2: "MASH"
}

plt.figure(figsize=(10, 7))

for label, name in class_names.items():
    subset = plot_df[plot_df["class"] == label]

    plt.scatter(
        subset["PC1"],
        subset["PC2"],
        label=name,
        alpha=0.7,
        s=35
    )

plt.xlabel("PC1 (28.63% variance)")
plt.ylabel("PC2 (16.87% variance)")
plt.title("PCA of Gene Expression by Disease Class")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()

class_file = output_dir / "PCA_by_Disease_Class.png"
plt.savefig(class_file, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# -------- DOWNLOAD BOTH FIGURES --------

print("Both PCA figures saved successfully.")

files.download(str(dataset_file))
files.download(str(class_file))

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

v3 = Path(
    "/content/ML_Project/ML_Project-main/external_validation_v3"
)

models_dir = v3 / "v3_final_run/models"

model_files = {
    "Binary": models_dir / "final_binary_full.joblib",
    "Multiclass": models_dir / "final_multiclass_full.joblib"
}

for task_name, model_path in model_files.items():

    artifact = joblib.load(model_path)
    selected_genes = artifact["selected_genes"]

    # X is the 346 x 10152 DataFrame from previous PCA code.
    missing = set(selected_genes) - set(X.columns)

    assert not missing, (
        f"{task_name}: Missing selected genes: {list(missing)[:5]}"
    )

    X_selected = X.loc[:, selected_genes]

    # Exploration only — not a fitted predictive preprocessing step.
    scaled = StandardScaler().fit_transform(X_selected)

    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(scaled)

    result = pd.DataFrame(
        coords,
        columns=["PC1", "PC2"],
        index=X.index
    ).join(metadata)

    print(f"\n{task_name.upper()} MODEL")
    print("Number of selected genes:", len(selected_genes))
    print(
        "PC1 and PC2 explained variance:",
        round(
            sum(pca.explained_variance_ratio_) * 100,
            2
        ),
        "%"
    )

    plt.figure(figsize=(9, 6))

    for cohort in [
        "GSE48452",
        "GSE126848",
        "GSE135251"
    ]:

        subset = result[result["dataset"] == cohort]

        plt.scatter(
            subset["PC1"],
            subset["PC2"],
            label=cohort,
            s=30,
            alpha=0.7
        )

    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title(f"PCA by Dataset — {task_name} Selected Genes")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

In [ ]:
plt.savefig(
    f"/content/PCA_selected_{task_name.lower()}.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from google.colab import files

v3 = Path(
    "/content/ML_Project/ML_Project-main/external_validation_v3"
)

models_dir = v3 / "v3_final_run/models"
output_dir = v3 / "v3_final_run/figures"
output_dir.mkdir(parents=True, exist_ok=True)

model_files = {
    "Binary": models_dir / "final_binary_full.joblib",
    "Multiclass": models_dir / "final_multiclass_full.joblib"
}

# Rebuild combined expression data so the cell works independently.
X_parts = []
metadata_parts = []

for cohort in ["GSE48452", "GSE126848", "GSE135251"]:

    expr = pd.read_csv(
        v3 / "data" / f"{cohort}_common_rank.csv.gz",
        index_col=0
    )

    meta = pd.read_csv(
        v3 / "data" / f"{cohort}_metadata_harmonized.csv",
        index_col=0
    )

    X_parts.append(expr.loc[:, meta.index].T)

    metadata_parts.append(
        pd.DataFrame(
            {
                "dataset": cohort,
                "class": meta["label_3class"].values
            },
            index=meta.index
        )
    )

X = pd.concat(X_parts)
metadata = pd.concat(metadata_parts)

for task_name, model_path in model_files.items():

    print(f"\nProcessing {task_name} model...")

    artifact = joblib.load(model_path)
    selected_genes = list(artifact["selected_genes"])

    assert set(selected_genes).issubset(X.columns), (
        f"Missing selected genes for {task_name}"
    )

    X_selected = X.loc[:, selected_genes]

    X_scaled = StandardScaler().fit_transform(X_selected)

    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(X_scaled)

    result = pd.DataFrame(
        coords,
        columns=["PC1", "PC2"],
        index=X.index
    ).join(metadata)

    explained = pca.explained_variance_ratio_ * 100

    print("Selected genes:", len(selected_genes))
    print("PC1:", round(explained[0], 2), "%")
    print("PC2:", round(explained[1], 2), "%")

    fig, ax = plt.subplots(figsize=(10, 7))

    for cohort in ["GSE48452", "GSE126848", "GSE135251"]:

        subset = result[result["dataset"] == cohort]

        ax.scatter(
            subset["PC1"],
            subset["PC2"],
            label=cohort,
            s=35,
            alpha=0.7
        )

    ax.set_xlabel(f"PC1 ({explained[0]:.2f}%)")
    ax.set_ylabel(f"PC2 ({explained[1]:.2f}%)")
    ax.set_title(
        f"PCA by Dataset — {task_name} Selected Genes"
    )

    ax.legend()
    ax.grid(alpha=0.2)

    fig.tight_layout()

    filename = (
        output_dir /
        f"PCA_selected_{task_name.lower()}.png"
    )

    # Save the actual figure before displaying and closing.
    fig.savefig(filename, dpi=300, bbox_inches="tight")

    print("Saved:", filename)

    plt.show()
    plt.close(fig)

    files.download(str(filename))

print("\nSUCCESS: Both selected-gene PCA plots generated.")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Uses X and metadata from your earlier PCA code.
# Uses model_files from the selected-gene PCA code.

class_names = {
    0: "Healthy",
    1: "MASLD",
    2: "MASH"
}

records = []

for task_name, model_path in model_files.items():

    artifact = joblib.load(model_path)
    genes = list(artifact["selected_genes"])

    # Standardize using the training cohort only.
    train_mask = metadata["dataset"] == "GSE48452"

    scaler = StandardScaler()
    scaler.fit(X.loc[train_mask, genes])

    Z = pd.DataFrame(
        scaler.transform(X.loc[:, genes]),
        index=X.index,
        columns=genes
    )

    # All comparisons use the same scaled gene space.
    for label, class_name in class_names.items():

        source_mask = (
            (metadata["dataset"] == "GSE48452") &
            (metadata["class"] == label)
        )

        source_mean = Z.loc[source_mask].mean(axis=0)

        for target in ["GSE126848", "GSE135251"]:

            target_mask = (
                (metadata["dataset"] == target) &
                (metadata["class"] == label)
            )

            target_mean = Z.loc[target_mask].mean(axis=0)

            distance = np.linalg.norm(
                source_mean.to_numpy()
                - target_mean.to_numpy()
            )

            records.append({
                "Model": task_name,
                "Class": class_name,
                "External_dataset": target,
                "Training_n": int(source_mask.sum()),
                "External_n": int(target_mask.sum()),
                "Mean_distance_per_gene": (
                    distance / np.sqrt(len(genes))
                )
            })

distances = pd.DataFrame(records)

print("WITHIN-CLASS CROSS-DATASET DISTANCES")
display(distances.round(4))

print("\nCLASS-WISE DISTANCE COMPARISON")

for task in ["Binary", "Multiclass"]:

    print(f"\n{task.upper()}")

    display(
        distances[
            distances["Model"] == task
        ].pivot(
            index="Class",
            columns="External_dataset",
            values="Mean_distance_per_gene"
        ).round(4)
    )

In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler

v3 = Path(
    "/content/ML_Project/ML_Project-main/external_validation_v3"
)

model_files = {
    "Binary": v3 / "v3_final_run/models/final_binary_full.joblib",
    "Multiclass": v3 / "v3_final_run/models/final_multiclass_full.joblib"
}

class_names = {0: "Healthy", 1: "MASLD", 2: "MASH"}

records = []

for task, path in model_files.items():

    artifact = joblib.load(path)
    genes = list(artifact["selected_genes"])

    train_mask = metadata["dataset"] == "GSE48452"

    scaler = StandardScaler()
    scaler.fit(X.loc[train_mask, genes])

    Z = pd.DataFrame(
        scaler.transform(X.loc[:, genes]),
        index=X.index,
        columns=genes
    )

    def class_mean(dataset, label):
        mask = (
            (metadata["dataset"] == dataset) &
            (metadata["class"] == label)
        )
        return Z.loc[mask].mean(axis=0).to_numpy()

    def distance(a, b):
        return np.linalg.norm(a - b) / np.sqrt(len(genes))

    # Disease differences within the training cohort.
    for a, b in [(0, 1), (0, 2), (1, 2)]:
        records.append({
            "Model": task,
            "Comparison": "Within training cohort",
            "Groups": f"{class_names[a]} vs {class_names[b]}",
            "Distance_per_gene": distance(
                class_mean("GSE48452", a),
                class_mean("GSE48452", b)
            )
        })

    # Same-class differences between training and external cohorts.
    for external in ["GSE126848", "GSE135251"]:
        for label, name in class_names.items():
            records.append({
                "Model": task,
                "Comparison": f"Training vs {external}",
                "Groups": name,
                "Distance_per_gene": distance(
                    class_mean("GSE48452", label),
                    class_mean(external, label)
                )
            })

comparison = pd.DataFrame(records)

print("DISEASE EFFECT VS DATASET EFFECT")
display(comparison.round(4))

comparison.to_csv(
    v3 / "v3_final_run/results/"
    "disease_vs_dataset_distance.csv",
    index=False
)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!python /content/run_v4_loso.py --quick

In [ ]:
from pathlib import Path

locations = [
    Path("/content/run_v4_loso.py"),
    Path("/content/ML_Project/ML_Project-main/run_v4_loso.py"),
    Path("/content/ML_Project/ML_Project-main/external_validation_v4/run_v4_loso.py")
]

for path in locations:
    print(f"{path}: {path.exists()}")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from pathlib import Path

matches = list(Path("/content").rglob("run_v4_loso.py"))

print("Files found:", len(matches))

for path in matches:
    print(path)

if not matches:
    raise FileNotFoundError(
        "run_v4_loso.py was not uploaded. Complete Step 2 first."
    )

script_path = matches[0]
print("\nScript ready:", script_path)

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

print("\nUploaded files:", list(uploaded.keys()))

for filename in uploaded:
    print("Saved at:", Path("/content") / filename)

In [ ]:
from pathlib import Path
import subprocess
import sys

matches = list(Path("/content").rglob("run_v4_loso.py"))

if not matches:
    print("ERROR: V4 script is still missing.")
else:
    script = matches[0]

    print("Script found:", script)
    print("File size:", script.stat().st_size, "bytes")
    print("\nStarting V4 quick experiment...\n")

    subprocess.run(
        [sys.executable, str(script), "--quick"],
        check=True
    )

In [ ]:
from pathlib import Path

old_file = Path("/content/run_v4_loso (6).py")
new_file = Path("/content/run_v4_loso.py")

if old_file.exists():
    old_file.rename(new_file)
    print("SUCCESS: File renamed!")
else:
    print("Original uploaded file not found.")

print("V4 script exists:", new_file.exists())

In [ ]:
import subprocess
import sys
from pathlib import Path

script = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3/run_v4_loso.py"
)

assert script.is_file(), "V4 script not found."

print("V4 script found successfully.")
print("Starting V4 quick experiment...\n")

subprocess.run(
    [sys.executable, str(script), "--quick"],
    check=True
)

In [ ]:
import pandas as pd
from pathlib import Path

results_path = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v4/loso_baseline/loso_summary.csv"
)

if results_path.exists():
    results = pd.read_csv(results_path)
    display(results.round(4))
else:
    print("Results not found yet. Check the V4 execution output.")

In [ ]:
from pathlib import Path
import pandas as pd

v4_dir = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v4/loso_baseline"
)

prediction_dir = v4_dir / "predictions"

print("V4 PREDICTION FILES")
print("=" * 60)

csv_files = sorted(prediction_dir.glob("*.csv"))

for file in csv_files:
    df = pd.read_csv(file)

    print("\nFile:", file.name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))

print("\nTotal CSV files:", len(csv_files))

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    balanced_accuracy_score
)

v4 = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v4/loso_baseline"
)

prediction_dir = v4 / "predictions"
output_dir = v4 / "diagnostics"
output_dir.mkdir(parents=True, exist_ok=True)

all_reports = []

for file in sorted(prediction_dir.glob("*.csv")):

    df = pd.read_csv(file)

    task = "multiclass" if file.name.startswith("multiclass") else "binary"

    classes = [0, 1, 2] if task == "multiclass" else [0, 1]

    names = (
        ["Healthy", "MASLD", "MASH"]
        if task == "multiclass"
        else ["Healthy", "Disease"]
    )

    y_true = df["true_label"].astype(int)
    y_pred = df["predicted_label"].astype(int)

    print("\n" + "=" * 65)
    print(file.name)
    print("=" * 65)

    print("\nACTUAL CLASS COUNTS")
    print(y_true.value_counts().sort_index())

    print("\nPREDICTED CLASS COUNTS")
    print(y_pred.value_counts().sort_index())

    print("\nCONFUSION MATRIX")
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=classes
    )

    display(
        pd.DataFrame(
            cm,
            index=[f"Actual {n}" for n in names],
            columns=[f"Predicted {n}" for n in names]
        )
    )

    print("\nCLASSIFICATION REPORT")

    report = classification_report(
        y_true,
        y_pred,
        labels=classes,
        target_names=names,
        zero_division=0
    )

    print(report)

    print(
        "Accuracy:",
        round(accuracy_score(y_true, y_pred), 4)
    )

    print(
        "Balanced accuracy:",
        round(balanced_accuracy_score(y_true, y_pred), 4)
    )

    probability_columns = [
        c for c in df.columns
        if c.startswith("prob_class_")
    ]

    print("\nMEAN PREDICTED PROBABILITIES BY ACTUAL CLASS")

    display(
        df.groupby("true_label")[
            probability_columns
        ].mean().round(4)
    )

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=classes,
        target_names=names,
        output_dict=True,
        zero_division=0
    )

    for name in names:

        all_reports.append({
            "file": file.name,
            "task": task,
            "class": name,
            "precision": report_dict[name]["precision"],
            "recall": report_dict[name]["recall"],
            "f1": report_dict[name]["f1-score"],
            "support": report_dict[name]["support"]
        })

    # Save each confusion matrix.
    pd.DataFrame(
        cm,
        index=names,
        columns=names
    ).to_csv(
        output_dir / f"{file.stem}_confusion_matrix.csv"
    )

summary = pd.DataFrame(all_reports)

summary.to_csv(
    output_dir / "v4_per_class_performance.csv",
    index=False
)

print("\n" + "=" * 65)
print("FINAL PER-CLASS PERFORMANCE SUMMARY")
print("=" * 65)

display(summary.round(4))

print("\nSaved diagnostics to:", output_dir)

### Original cell 168 — incomplete, disabled

**Reason:** '(' was never closed (source line 166). This cell is preserved for review, not execution.

```python
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

root = Path("/content/ML_Project/ML_Project-main")
v3 = root / "external_validation_v3"
v4 = root / "external_validation_v4"

out = v4 / "refinement_1"
out.mkdir(parents=True, exist_ok=True)

studies = ["GSE48452", "GSE126848", "GSE135251"]

# Load data.
parts = []
labels = []

for study in studies:
    expr = pd.read_csv(
        v3 / "data" / f"{study}_common_rank.csv.gz",
        index_col=0
    )
    meta = pd.read_csv(
        v3 / "data" / f"{study}_metadata_harmonized.csv",
        index_col=0
    )

    parts.append(expr.loc[:, meta.index].T)

    m = pd.DataFrame(index=meta.index)
    m["study"] = study
    m["binary"] = meta["label_binary"].astype(int)
    m["multiclass"] = meta["label_3class"].astype(int)
    labels.append(m)

X = pd.concat(parts)
M = pd.concat(labels)

assert X.shape == (346, 10152)
assert X.index.equals(M.index)

def choose_consistent_genes(Xtr, ytr, studytr, task, k=50):
    """
    Select genes using training studies only.
    Require agreement in the direction of class mean differences.
    """
    available_studies = sorted(studytr.unique())

    comparisons = (
        [(0, 1)] if task == "binary"
        else [(0, 1), (0, 2), (1, 2)]
    )

    scores = np.zeros(Xtr.shape[1], dtype=float)

    for a, b in comparisons:
        effects = []

        for study in available_studies:
            mask_a = (studytr == study) & (ytr == a)
            mask_b = (studytr == study) & (ytr == b)

            if mask_a.sum() == 0 or mask_b.sum() == 0:
                continue

            mean_a = Xtr.loc[mask_a].mean(axis=0).to_numpy()
            mean_b = Xtr.loc[mask_b].mean(axis=0).to_numpy()

            effects.append(mean_b - mean_a)

        if len(effects) != len(available_studies):
            continue

        effects = np.vstack(effects)

        same_direction = (
            np.all(effects > 0, axis=0)
            | np.all(effects < 0, axis=0)
        )

        # Conservative strength: weakest absolute study effect.
        strength = np.min(np.abs(effects), axis=0)

        scores += np.where(same_direction, strength, 0.0)

    ranking = np.argsort(-scores, kind="stable")
    ranking = ranking[scores[ranking] > 0]

    if len(ranking) == 0:
        raise ValueError(
            "No direction-consistent genes found."
        )

    selected = Xtr.columns[ranking[:k]].tolist()
    return selected

def balanced_study_class_weights(studytr, ytr):
    groups = pd.DataFrame({
        "study": studytr.to_numpy(),
        "class": ytr.to_numpy()
    })

    counts = groups.groupby(["study", "class"]).size()

    weights = np.array([
        1.0 / counts.loc[(s, c)]
        for s, c in zip(groups["study"], groups["class"])
    ])

    return weights / weights.mean()

rows = []

for task in ["binary", "multiclass"]:

    for heldout in studies:

        train_mask = M["study"] != heldout
        test_mask = M["study"] == heldout

        Xtr = X.loc[train_mask]
        Xte = X.loc[test_mask]

        ytr = M.loc[train_mask, task]
        yte = M.loc[test_mask, task]
        studytr = M.loc[train_mask, "study"]

        genes = choose_consistent_genes(
            Xtr, ytr, studytr, task, k=50
        )

        weights = balanced_study_class_weights(
            studytr, ytr
        )

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                C=0.1,
                max_iter=3000,
                solver="lbfgs",
                random_state=42
            ))
        ])

        model.fit(
            Xtr[genes],
            ytr,
            classifier__sample_weight=weights
        )

        pred = model.predict(Xte[genes])
        prob = model.predict_proba(Xte[genes])

        if task == "binary":
            auc = roc_auc_score(
                yte,
                prob[:, list(model.classes_).index(1)]
```


In [ ]:
# Calculate ROC-AUC
if task == "binary":
    auc = roc_auc_score(yte, prob[:, 1])

elif task == "multiclass":
    auc = roc_auc_score(
        yte,
        prob,
        multi_class="ovr",
        average="macro",
        labels=[0, 1, 2]
    )

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

folder = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v4/loso_baseline/predictions"
)

results = []

for file in sorted(folder.glob("*.csv")):

    df = pd.read_csv(file)

    yte = df["true_label"].astype(int)
    pred = df["predicted_label"].astype(int)

    if file.name.startswith("binary"):
        task = "binary"
        auc = roc_auc_score(
            yte,
            df["prob_class_1"]
        )

    else:
        task = "multiclass"

        prob = df[
            ["prob_class_0", "prob_class_1", "prob_class_2"]
        ].to_numpy()

        auc = roc_auc_score(
            yte,
            prob,
            multi_class="ovr",
            average="macro",
            labels=[0, 1, 2]
        )

    results.append({
        "task": task,
        "dataset": file.name,
        "accuracy": accuracy_score(yte, pred),
        "balanced_accuracy": balanced_accuracy_score(yte, pred),
        "macro_f1": f1_score(
            yte, pred, average="macro", zero_division=0
        ),
        "roc_auc": auc
    })

summary = pd.DataFrame(results)

print("V4 MODEL EVALUATION RESULTS")
display(summary.round(4))

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

ROOT = Path("/content/ML_Project/ML_Project-main")
DATA = ROOT / "external_validation_v3/data"
OUT = ROOT / "external_validation_v4/refinement_1"
OUT.mkdir(parents=True, exist_ok=True)

studies = ["GSE48452", "GSE126848", "GSE135251"]

# 1. Load all datasets
X_parts = []
meta_parts = []

for study in studies:
    expr = pd.read_csv(
        DATA / f"{study}_common_rank.csv.gz",
        index_col=0
    )
    meta = pd.read_csv(
        DATA / f"{study}_metadata_harmonized.csv",
        index_col=0
    )

    assert meta.index.isin(expr.columns).all(), study

    X_parts.append(expr.loc[:, meta.index].T)

    temp = pd.DataFrame(index=meta.index)
    temp["study"] = study
    temp["binary"] = meta["label_binary"].astype(int)
    temp["multiclass"] = meta["label_3class"].astype(int)

    meta_parts.append(temp)

X = pd.concat(X_parts)
M = pd.concat(meta_parts)

assert X.shape == (346, 10152)
assert X.index.equals(M.index)

print("Data loaded:", X.shape)


# 2. Select direction-consistent genes using training data only
def select_stable_genes(X_train, y_train, groups, task, k=50):

    comparisons = (
        [(0, 1)] if task == "binary"
        else [(0, 1), (0, 2), (1, 2)]
    )

    unique_studies = sorted(groups.unique())
    score = np.zeros(X_train.shape[1])

    for class_a, class_b in comparisons:
        effects = []

        for study in unique_studies:
            a = (groups == study) & (y_train == class_a)
            b = (groups == study) & (y_train == class_b)

            if a.sum() == 0 or b.sum() == 0:
                raise ValueError("Missing class in training study")

            effect = (
                X_train.loc[b].mean().to_numpy()
                - X_train.loc[a].mean().to_numpy()
            )

            effects.append(effect)

        effects = np.vstack(effects)

        consistent = (
            np.all(effects > 0, axis=0)
            | np.all(effects < 0, axis=0)
        )

        strength = np.min(np.abs(effects), axis=0)

        score += np.where(consistent, strength, 0)

    ranked = np.argsort(-score, kind="stable")
    ranked = ranked[score[ranked] > 0]

    if len(ranked) == 0:
        raise ValueError("No consistent genes found")

    return X_train.columns[ranked[:k]].tolist()


# 3. Balance both study and class contributions
def calculate_weights(groups, labels):

    info = pd.DataFrame({
        "study": groups.to_numpy(),
        "label": labels.to_numpy()
    })

    counts = info.groupby(["study", "label"]).size()

    weights = np.array([
        1.0 / counts.loc[(s, c)]
        for s, c in zip(info["study"], info["label"])
    ])

    return weights / weights.mean()


# 4. Train and evaluate each held-out study
results = []

for task in ["binary", "multiclass"]:

    for heldout in studies:

        train_mask = M["study"] != heldout
        test_mask = M["study"] == heldout

        X_train = X.loc[train_mask]
        X_test = X.loc[test_mask]

        y_train = M.loc[train_mask, task]
        y_test = M.loc[test_mask, task]

        groups = M.loc[train_mask, "study"]

        selected_genes = select_stable_genes(
            X_train, y_train, groups, task, k=50
        )

        weights = calculate_weights(groups, y_train)

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                C=0.1,
                max_iter=3000,
                solver="lbfgs",
                random_state=42
            ))
        ])

        model.fit(
            X_train[selected_genes],
            y_train,
            classifier__sample_weight=weights
        )

        predictions = model.predict(X_test[selected_genes])
        probabilities = model.predict_proba(X_test[selected_genes])

        classes = list(model.classes_)

        if task == "binary":
            auc = roc_auc_score(
                y_test,
                probabilities[:, classes.index(1)]
            )
        else:
            auc = roc_auc_score(
                y_test,
                probabilities,
                multi_class="ovr",
                average="macro",
                labels=classes
            )

        result = {
            "task": task,
            "heldout": heldout,
            "accuracy": accuracy_score(y_test, predictions),
            "balanced_accuracy": balanced_accuracy_score(
                y_test, predictions
            ),
            "f1_macro": f1_score(
                y_test, predictions,
                average="macro",
                zero_division=0
            ),
            "roc_auc": auc,
            "selected_genes": len(selected_genes)
        }

        results.append(result)

        output = pd.DataFrame({
            "GSM_ID": X_test.index,
            "true_label": y_test.to_numpy(),
            "predicted_label": predictions
        })

        for j, label in enumerate(classes):
            output[f"prob_class_{label}"] = probabilities[:, j]

        output.to_csv(
            OUT / f"{task}_{heldout}_predictions.csv",
            index=False
        )

        pd.DataFrame({
            "gene": selected_genes
        }).to_csv(
            OUT / f"{task}_{heldout}_selected_genes.csv",
            index=False
        )

        print(
            f"{task} | {heldout} | "
            f"Balanced accuracy: {result['balanced_accuracy']:.4f} | "
            f"Macro F1: {result['f1_macro']:.4f}"
        )


# 5. Save refinement results
refined = pd.DataFrame(results)

refined.to_csv(
    OUT / "refinement_1_summary.csv",
    index=False
)

print("\nREFINEMENT 1 RESULTS")
display(refined.round(4))


# 6. Compare against original V4 baseline
baseline = pd.read_csv(
    ROOT /
    "external_validation_v4/loso_baseline/loso_summary.csv"
)

compare = baseline[
    ["task", "heldout", "balanced_accuracy", "f1_macro", "roc_auc"]
].merge(
    refined[
        ["task", "heldout", "balanced_accuracy", "f1_macro", "roc_auc"]
    ],
    on=["task", "heldout"],
    suffixes=("_baseline", "_refined")
)

print("\nV4 BASELINE VS REFINEMENT 1")
display(compare.round(4))

print("\nAll results saved to:", OUT)

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

root = Path(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v4/refinement_1"
)

rows = []

for task in ["binary", "multiclass"]:
    classes = [0, 1] if task == "binary" else [0, 1, 2]
    names = (
        ["Healthy", "Disease"]
        if task == "binary"
        else ["Healthy", "MASLD", "MASH"]
    )

    for study in ["GSE48452", "GSE126848", "GSE135251"]:
        path = root / f"{task}_{study}_predictions.csv"
        df = pd.read_csv(path)

        report = classification_report(
            df["true_label"],
            df["predicted_label"],
            labels=classes,
            target_names=names,
            output_dict=True,
            zero_division=0
        )

        cm = confusion_matrix(
            df["true_label"],
            df["predicted_label"],
            labels=classes
        )

        print(f"\n{task.upper()} — {study}")
        display(pd.DataFrame(
            cm,
            index=[f"Actual {n}" for n in names],
            columns=[f"Predicted {n}" for n in names]
        ))

        for name in names:
            rows.append({
                "task": task,
                "heldout": study,
                "class": name,
                "recall": report[name]["recall"],
                "precision": report[name]["precision"],
                "f1": report[name]["f1-score"],
                "support": report[name]["support"]
            })

summary = pd.DataFrame(rows)

print("\nREFINEMENT 1 — PER-CLASS PERFORMANCE")
display(summary.round(4))

summary.to_csv(
    root / "refinement_1_per_class_performance.csv",
    index=False
)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.feature_selection import f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, roc_auc_score
)

ROOT = Path("/content/ML_Project/ML_Project-main")
DATA = ROOT / "external_validation_v3/data"
OUT = ROOT / "external_validation_v4/refinement_2_ablation"
OUT.mkdir(parents=True, exist_ok=True)

studies = ["GSE48452", "GSE126848", "GSE135251"]

X_parts, M_parts = [], []

for study in studies:
    expr = pd.read_csv(
        DATA / f"{study}_common_rank.csv.gz",
        index_col=0
    )
    meta = pd.read_csv(
        DATA / f"{study}_metadata_harmonized.csv",
        index_col=0
    )

    X_parts.append(expr.loc[:, meta.index].T)

    m = pd.DataFrame(index=meta.index)
    m["study"] = study
    m["binary"] = meta["label_binary"].astype(int)
    m["multiclass"] = meta["label_3class"].astype(int)
    M_parts.append(m)

X = pd.concat(X_parts)
M = pd.concat(M_parts)

assert X.shape == (346, 10152)
assert X.index.equals(M.index)

def select_genes(Xtr, ytr, groups, task, method, k=50):

    if method == "pooled":
        with np.errstate(divide="ignore", invalid="ignore"):
            _, pvalues = f_classif(Xtr, ytr)

        # Use smallest ANOVA p-values
        pvalues = np.nan_to_num(
            pvalues, nan=1.0, posinf=1.0, neginf=1.0
        )
        order = np.argsort(pvalues, kind="stable")
        return Xtr.columns[order[:k]].tolist()

    comparisons = (
        [(0, 1)] if task == "binary"
        else [(0, 1), (0, 2), (1, 2)]
    )

    score = np.zeros(Xtr.shape[1])

    for a, b in comparisons:
        effects = []

        for study in sorted(groups.unique()):
            mask_a = (groups == study) & (ytr == a)
            mask_b = (groups == study) & (ytr == b)

            if mask_a.sum() == 0 or mask_b.sum() == 0:
                raise ValueError("Missing class in training study")

            effect = (
                Xtr.loc[mask_b].mean().to_numpy()
                - Xtr.loc[mask_a].mean().to_numpy()
            )
            effects.append(effect)

        effects = np.vstack(effects)
        consistent = (
            np.all(effects > 0, axis=0)
            | np.all(effects < 0, axis=0)
        )

        strength = np.min(np.abs(effects), axis=0)
        score += np.where(consistent, strength, 0)

    order = np.argsort(-score, kind="stable")
    order = order[score[order] > 0]

    if not len(order):
        raise ValueError("No direction-consistent genes")

    return Xtr.columns[order[:k]].tolist()

def make_weights(groups, y, method):
    frame = pd.DataFrame({
        "study": groups.to_numpy(),
        "label": y.to_numpy()
    })

    keys = ["label"] if method == "class" else ["study", "label"]

    counts = frame.groupby(keys).size()

    if method == "class":
        weights = np.array(
            [1.0 / counts.loc[label] for label in frame["label"]]
        )
    else:
        weights = np.array([
            1.0 / counts.loc[(study, label)]
            for study, label in zip(frame["study"], frame["label"])
        ])

    return weights / weights.mean()

experiments = [
    ("A_pooled_class", "pooled", "class"),
    ("B_pooled_studyclass", "pooled", "studyclass"),
    ("C_stable_class", "stable", "class"),
    ("D_stable_studyclass", "stable", "studyclass")
]

results = []

for task in ["binary", "multiclass"]:
    for heldout in studies:

        train = M["study"] != heldout
        test = M["study"] == heldout

        Xtr, Xte = X.loc[train], X.loc[test]
        ytr, yte = M.loc[train, task], M.loc[test, task]
        groups = M.loc[train, "study"]

        for name, feature_method, weight_method in experiments:

            genes = select_genes(
                Xtr, ytr, groups, task, feature_method, 50
            )
            weights = make_weights(groups, ytr, weight_method)

            model = Pipeline([
                ("scaler", StandardScaler()),
                ("classifier", LogisticRegression(
                    C=0.1,
                    solver="lbfgs",
                    max_iter=3000,
                    random_state=42
                ))
            ])

            model.fit(
                Xtr[genes], ytr,
                classifier__sample_weight=weights
            )

            pred = model.predict(Xte[genes])
            prob = model.predict_proba(Xte[genes])
            classes = list(model.classes_)

            if task == "binary":
                auc = roc_auc_score(
                    yte, prob[:, classes.index(1)]
                )
            else:
                auc = roc_auc_score(
                    yte, prob,
                    multi_class="ovr",
                    average="macro",
                    labels=classes
                )

            results.append({
                "task": task,
                "heldout": heldout,
                "experiment": name,
                "balanced_accuracy": balanced_accuracy_score(yte, pred),
                "f1_macro": f1_score(
                    yte, pred, average="macro", zero_division=0
                ),
                "roc_auc": auc
            })

            print(
                f"{task:10} | {heldout} | {name:21} | "
                f"BA={results[-1]['balanced_accuracy']:.4f}"
            )

results = pd.DataFrame(results)
results.to_csv(OUT / "ablation_results.csv", index=False)

print("\nMEAN PERFORMANCE ACROSS THREE HELD-OUT STUDIES")

mean_results = (
    results.groupby(["task", "experiment"])
    [["balanced_accuracy", "f1_macro", "roc_auc"]]
    .mean()
    .round(4)
)

display(mean_results)

print("\nPER-STUDY BALANCED ACCURACY")
display(
    results.pivot_table(
        index=["task", "experiment"],
        columns="heldout",
        values="balanced_accuracy"
    ).round(4)
)

print("\nSaved to:", OUT)

In [ ]:
import numpy as np
import pandas as pd
import joblib
import json

from pathlib import Path
from datetime import datetime, timezone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

ROOT = Path("/content/ML_Project/ML_Project-main")
DATA = ROOT / "external_validation_v3/data"
OUT = ROOT / "external_validation_v4/frozen_candidate_v1"
OUT.mkdir(parents=True, exist_ok=True)

STUDIES = ["GSE48452", "GSE126848", "GSE135251"]

X_parts = []
meta_parts = []

for study in STUDIES:
    expr = pd.read_csv(
        DATA / f"{study}_common_rank.csv.gz",
        index_col=0
    )
    meta = pd.read_csv(
        DATA / f"{study}_metadata_harmonized.csv",
        index_col=0
    )

    assert meta.index.isin(expr.columns).all()

    X_parts.append(expr.loc[:, meta.index].T)

    m = pd.DataFrame(index=meta.index)
    m["study"] = study
    m["label"] = meta["label_binary"].astype(int)
    meta_parts.append(m)

X = pd.concat(X_parts)
M = pd.concat(meta_parts)

assert X.shape == (346, 10152)
assert X.index.equals(M.index)
assert not X.isna().any().any()

# Select genes showing the same disease-effect direction
# across ALL THREE training studies.
effects = []

for study in STUDIES:
    healthy = (M["study"] == study) & (M["label"] == 0)
    disease = (M["study"] == study) & (M["label"] == 1)

    effect = (
        X.loc[disease].mean(axis=0).to_numpy()
        - X.loc[healthy].mean(axis=0).to_numpy()
    )
    effects.append(effect)

effects = np.vstack(effects)

consistent = (
    np.all(effects > 0, axis=0)
    | np.all(effects < 0, axis=0)
)

score = np.where(
    consistent,
    np.min(np.abs(effects), axis=0),
    0.0
)

order = np.argsort(-score, kind="stable")
order = order[score[order] > 0]

print("Direction-consistent genes:", len(order))

if len(order) < 50:
    raise ValueError(
        "Fewer than 50 genes are consistent across all "
        "three studies. Do not change the selection rule "
        "without recording a new model version."
    )

genes = X.columns[order[:50]].tolist()

# Equal contribution from Healthy and Disease classes.
counts = M["label"].value_counts()

weights = M["label"].map(
    lambda label: 1.0 / counts[label]
).to_numpy()

weights = weights / weights.mean()

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        C=0.1,
        solver="lbfgs",
        max_iter=3000,
        random_state=42
    ))
])

model.fit(
    X[genes],
    M["label"],
    classifier__sample_weight=weights
)

artifact = {
    "model": model,
    "selected_genes": genes,
    "common_genes": X.columns.tolist(),
    "task": "binary",
    "classes": model.classes_.tolist(),
    "training_studies": STUDIES,
    "training_samples": len(X),
    "input_representation": (
        "Within-sample percentile ranks computed "
        "over the fixed 10152-gene common panel"
    )
}

model_file = OUT / "binary_candidate_v1.joblib"
joblib.dump(artifact, model_file)

pd.DataFrame({
    "gene": genes,
    "selection_score": score[order[:50]]
}).to_csv(
    OUT / "selected_genes.csv",
    index=False
)

manifest = {
    "version": "candidate_v1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "training_studies": STUDIES,
    "training_n": len(X),
    "common_gene_count": X.shape[1],
    "selected_gene_count": len(genes),
    "gene_selection": (
        "Consistent binary disease-effect direction "
        "in all three training studies; rank by "
        "minimum absolute effect"
    ),
    "sample_weighting": "Inverse class frequency",
    "classifier": "LogisticRegression",
    "C": 0.1,
    "status": "Development candidate; not externally validated"
}

with open(OUT / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

# Verify that the saved artifact can be loaded.
loaded = joblib.load(model_file)

assert loaded["selected_genes"] == genes
assert len(loaded["selected_genes"]) == 50

print("\nMODEL SAVED SUCCESSFULLY")
print("Model:", model_file)
print("Training samples:", len(X))
print("Selected genes:", len(genes))
print("Saved artifact reload: PASSED")
print("\nFirst 10 selected genes:")
print(genes[:10])

## 07 — External validation and dataset investigation


In [ ]:
from pathlib import Path
import requests
import gzip

ROOT = Path("/content/ML_Project/ML_Project-main")
OUT = ROOT / "external_validation_v4/external_GSE89632"
OUT.mkdir(parents=True, exist_ok=True)

url = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/"
    "GSE89nnn/GSE89632/matrix/"
    "GSE89632_series_matrix.txt.gz"
)

destination = OUT / "GSE89632_series_matrix.txt.gz"

if not destination.exists() or destination.stat().st_size == 0:
    print("Downloading GSE89632 series matrix...")

    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()

        with open(destination, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

print("Downloaded file:", destination)
print("Size (MB):", round(destination.stat().st_size / 1e6, 2))

print("\nSAMPLE METADATA")
print("=" * 70)

sample_metadata = {}

with gzip.open(destination, "rt", errors="replace") as f:
    for line in f:
        if line.startswith("!series_matrix_table_begin"):
            break

        if line.startswith("!Sample_"):
            parts = line.strip().split("\t")
            key = parts[0]
            values = [x.strip('"') for x in parts[1:]]
            sample_metadata[key] = values

for key in sample_metadata:
    if any(word in key.lower() for word in [
        "title", "accession", "characteristics", "source"
    ]):
        print("\n", key)
        print(sample_metadata[key][:8])

print("\nNumber of samples:",
      len(sample_metadata.get("!Sample_geo_accession", [])))

print("\nMetadata inspection complete.")

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd
import gzip
import csv
import re

ROOT = Path("/content/ML_Project/ML_Project-main")
DATA = ROOT / "external_validation_v4/external_GSE89632"
matrix_file = DATA / "GSE89632_series_matrix.txt.gz"

assert matrix_file.is_file(), "Series matrix not found"

metadata = {}
table_header = None
first_rows = []

with gzip.open(matrix_file, "rt", errors="replace") as f:
    in_table = False

    for line in f:
        if line.startswith("!Sample_"):
            fields = next(csv.reader([line], delimiter="\t"))
            metadata[fields[0]] = fields[1:]

        if line.startswith("!series_matrix_table_begin"):
            in_table = True
            continue

        if in_table:
            if line.startswith("!series_matrix_table_end"):
                break

            fields = next(csv.reader([line], delimiter="\t"))

            if table_header is None:
                table_header = fields
            elif len(first_rows) < 5:
                first_rows.append(fields)

            if len(first_rows) == 5:
                break

titles = metadata["!Sample_title"]
gsm_ids = metadata["!Sample_geo_accession"]

assert len(titles) == len(gsm_ids) == 63
assert len(set(gsm_ids)) == 63

def classify_title(title):
    t = title.upper()

    if re.search(r"(^|[_\-\s])NASH([_\-\s]|$)", t):
        return "NASH"

    if re.search(r"(^|[_\-\s])SS([_\-\s]|$)", t):
        return "SS"

    if re.search(r"(^|[_\-\s])HC([_\-\s]|$)", t):
        return "Healthy"

    return "UNRECOGNIZED"

groups = [classify_title(t) for t in titles]

sample_table = pd.DataFrame({
    "GSM_ID": gsm_ids,
    "title": titles,
    "title_group": groups
})

print("CLINICAL GROUP COUNTS")
print(sample_table["title_group"].value_counts())

print("\nUNRECOGNIZED TITLES")
print(sample_table.loc[
    sample_table["title_group"] == "UNRECOGNIZED"
].to_string(index=False))

print("\nALL SAMPLE TITLES")
display(sample_table)

print("\nEXPRESSION MATRIX FORMAT")
print("Header length:", len(table_header))
print("First 5 column names:", table_header[:5])
print("First 5 probe IDs:", [r[0] for r in first_rows])
print("Example expression values:", first_rows[0][1:6])

print("\nFROZEN MODEL CHECK")
import joblib

model_path = (
    ROOT / "external_validation_v4/"
    "frozen_candidate_v1/binary_candidate_v1.joblib"
)

artifact = joblib.load(model_path)

print("Selected genes:", len(artifact["selected_genes"]))
print("Common ranking genes:", len(artifact["common_genes"]))
print("Training studies:", artifact["training_studies"])

sample_table.to_csv(
    DATA / "sample_title_audit.csv",
    index=False
)

print("\nInspection complete. Model unchanged.")

In [ ]:
from pathlib import Path
import pandas as pd
import requests
import gzip
import io

ROOT = Path("/content/ML_Project/ML_Project-main")
OUT = ROOT / "external_validation_v4/external_GSE89632"
OUT.mkdir(parents=True, exist_ok=True)

URL = (
    "https://ftp.ncbi.nlm.nih.gov/geo/platforms/"
    "GPL14nnn/GPL14951/annot/GPL14951.annot.gz"
)

annotation_file = OUT / "GPL14951.annot.gz"

if not annotation_file.exists() or annotation_file.stat().st_size == 0:
    print("Downloading GPL14951 annotation...")

    with requests.get(URL, stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(annotation_file, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)

print("Annotation saved:", annotation_file)

# Inspect metadata and identify the platform table.
header = None
example_rows = []

with gzip.open(annotation_file, "rt", errors="replace") as f:
    in_table = False

    for line in f:
        if line.startswith("!platform_table_begin") or \
           line.startswith("!Platform_table_begin"):
            in_table = True
            continue

        if in_table:
            if line.startswith("!platform_table_end") or \
               line.startswith("!Platform_table_end"):
                break

            fields = line.rstrip("\n").split("\t")

            if header is None:
                header = fields
            elif len(example_rows) < 5:
                example_rows.append(fields)

            if len(example_rows) == 5:
                break

print("\nANNOTATION COLUMNS:")
print(header)

print("\nFIRST FIVE ANNOTATION RECORDS:")

if header is not None:
    display(pd.DataFrame(example_rows, columns=header))
else:
    print("Platform table marker not found.")
    print("We need to inspect the annotation's header format.")

In [ ]:
import requests
import pandas as pd
import gzip
import csv
import io
from pathlib import Path

ROOT = Path("/content/ML_Project/ML_Project-main")
OUT = ROOT / "external_validation_v4/external_GSE89632"

OUT.mkdir(parents=True, exist_ok=True)

# Official Illumina HumanHT-12 V4 annotation
url = (
    "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi"
    "?acc=GPL10558&targ=self&form=text&view=full"
)

print("Downloading official Illumina platform annotation...")

r = requests.get(
    url,
    headers={"User-Agent": "MASLD-Research/1.0"},
    timeout=180
)
r.raise_for_status()

lines = r.text.splitlines()

start = next(
    (i for i, line in enumerate(lines)
     if line.strip().lower() == "!platform_table_begin"),
    None
)

end = next(
    (i for i, line in enumerate(lines)
     if line.strip().lower() == "!platform_table_end"),
    None
)

if start is None or end is None or end <= start:
    print("Platform table not found.")
    print("Response status:", r.status_code)
    print("First 500 characters:", r.text[:500])
    raise ValueError("Unable to locate the GPL10558 probe table.")

table_text = "\n".join(lines[start + 1:end])

annotation = pd.read_csv(
    io.StringIO(table_text),
    sep="\t",
    dtype=str,
    low_memory=False
)

print("\nANNOTATION SHAPE:", annotation.shape)

print("\nANNOTATION COLUMNS:")
print(annotation.columns.tolist())

print("\nFIRST FIVE ROWS:")
display(annotation.head())

# Read actual probe identifiers from GSE89632
matrix_path = OUT / "GSE89632_series_matrix.txt.gz"

probe_ids = []

with gzip.open(matrix_path, "rt", errors="replace") as f:
    in_table = False

    for line in f:
        if line.startswith("!series_matrix_table_begin"):
            in_table = True
            continue

        if line.startswith("!series_matrix_table_end"):
            break

        if in_table and line.startswith("ILMN_"):
            probe_ids.append(line.split("\t", 1)[0].strip('"'))

# Match probes using the platform's ID column
annotation_ids = set(annotation["ID"].dropna())
matched = set(probe_ids) & annotation_ids

print("\nPROBE COMPATIBILITY CHECK")
print("GSE89632 probe IDs:", len(set(probe_ids)))
print("Matching platform IDs:", len(matched))
print("Match percentage:",
      round(100 * len(matched) / len(set(probe_ids)), 2), "%")

annotation.to_csv(
    OUT / "GPL10558_annotation.csv",
    index=False
)

print("\nAnnotation saved successfully.")
print("Frozen model remains unchanged.")

In [ ]:
from pathlib import Path
import pandas as pd
import gzip
import csv

ROOT = Path("/content/ML_Project/ML_Project-main")
OUT = ROOT / "external_validation_v4/external_GSE89632"

matrix_file = OUT / "GSE89632_series_matrix.txt.gz"
annotation_file = OUT / "GPL10558_annotation.csv"

assert matrix_file.exists(), "GSE89632 matrix file is missing"

# Use the annotation already loaded in the earlier Colab cell,
# or read it from disk if it was saved.
if annotation_file.exists():
    annotation = pd.read_csv(
        annotation_file,
        dtype=str,
        low_memory=False
    )
elif "annotation" in globals():
    annotation = globals()["annotation"].copy()
else:
    raise FileNotFoundError(
        "GPL10558 annotation is not available. "
        "Please rerun the annotation download cell up to "
        "the point where the annotation DataFrame is created."
    )

annotation["ID"] = annotation["ID"].astype("string").str.strip()
annotation["Symbol"] = annotation["Symbol"].astype("string").str.strip()

probe_ids = []
in_table = False

with gzip.open(matrix_file, "rt", errors="replace") as f:
    for line in f:

        if line.strip() == "!series_matrix_table_begin":
            in_table = True
            continue

        if line.strip() == "!series_matrix_table_end":
            break

        if not in_table:
            continue

        fields = next(csv.reader([line], delimiter="\t"))

        if not fields:
            continue

        probe = fields[0].strip()

        if probe.startswith("ILMN_"):
            probe_ids.append(probe)

unique_probes = set(probe_ids)
annotation_ids = set(annotation["ID"].dropna())

matching_probes = unique_probes & annotation_ids

mapped = annotation.loc[
    annotation["ID"].isin(matching_probes)
].copy()

valid_symbols = (
    mapped["Symbol"].notna()
    & mapped["Symbol"].ne("")
    & mapped["Symbol"].ne("---")
)

mapped_valid = mapped.loc[valid_symbols].copy()

print("=" * 60)
print("GSE89632 PROBE COMPATIBILITY AUDIT")
print("=" * 60)

print("Expression probe IDs:", len(unique_probes))
print("Annotation probe IDs:", len(annotation_ids))
print("Matching probe IDs:", len(matching_probes))

if unique_probes:
    print(
        "Probe ID match:",
        round(
            100 * len(matching_probes) / len(unique_probes),
            2
        ),
        "%"
    )
else:
    print("No expression probe IDs found. Check matrix formatting.")

print("Matching probes with nonempty symbols:", len(mapped_valid))
print("Unique mapped gene symbols:", mapped_valid["Symbol"].nunique())

print("\nEXAMPLE MATCHED ANNOTATION ROWS")
display(mapped_valid[["ID", "Symbol", "ILMN_Gene"]].head(10))

# Check coverage against the frozen model.
import joblib

model_path = (
    ROOT / "external_validation_v4/"
    "frozen_candidate_v1/binary_candidate_v1.joblib"
)

artifact = joblib.load(model_path)

required_genes = set(artifact["selected_genes"])
common_panel = set(artifact["common_genes"])
available_genes = set(mapped_valid["Symbol"])

covered_model_genes = required_genes & available_genes
covered_panel_genes = common_panel & available_genes

print("\nFROZEN MODEL COVERAGE")
print("Required model genes:", len(required_genes))
print("Covered model genes:", len(covered_model_genes))
print("Missing model genes:", len(required_genes - available_genes))
print(
    "Covered common-panel genes:",
    len(covered_panel_genes),
    "/",
    len(common_panel)
)

print("\nMISSING MODEL GENES")
print(sorted(required_genes - available_genes))

print("\nAudit complete. No model predictions generated.")

In [ ]:
import requests
import pandas as pd
import joblib
import io
from pathlib import Path

ROOT = Path("/content/ML_Project/ML_Project-main")
OUT = ROOT / "external_validation_v4/external_GSE89632"

model_file = (
    ROOT / "external_validation_v4/"
    "frozen_candidate_v1/binary_candidate_v1.joblib"
)

artifact = joblib.load(model_file)

url = (
    "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi"
    "?acc=GPL14951&targ=self&form=text&view=full"
)

print("Checking native GPL14951 annotation...")

response = requests.get(
    url,
    headers={"User-Agent": "AcademicResearch/1.0"},
    timeout=180
)
response.raise_for_status()

lines = response.text.splitlines()

start = next(
    (
        i for i, line in enumerate(lines)
        if line.strip().lower() == "!platform_table_begin"
    ),
    None
)

end = next(
    (
        i for i, line in enumerate(lines)
        if line.strip().lower() == "!platform_table_end"
    ),
    None
)

if start is None or end is None:
    print("Native GEO platform table not available in response.")
    print("Response beginning:", response.text[:300])
    print("No model predictions were generated.")
else:
    table_text = "\n".join(lines[start + 1:end])

    platform = pd.read_csv(
        io.StringIO(table_text),
        sep="\t",
        dtype=str,
        low_memory=False
    )

    print("\nGPL14951 ANNOTATION")
    print("Shape:", platform.shape)
    print("Columns:", platform.columns.tolist())

    display(platform.head(5))

    if "ID" in platform.columns:
        previous = pd.read_csv(
            OUT / "GPL10558_annotation.csv",
            dtype=str,
            low_memory=False
        )

        previous_ids = set(previous["ID"].dropna())
        native_ids = set(platform["ID"].dropna())

        print("\nPLATFORM COMPARISON")
        print("GPL14951 IDs:", len(native_ids))
        print("GPL10558 IDs:", len(previous_ids))
        print("Common probe IDs:", len(native_ids & previous_ids))

        if "Symbol" in platform.columns:

            platform["Symbol"] = (
                platform["Symbol"]
                .astype("string")
                .str.strip()
            )

            valid = platform[
                platform["Symbol"].notna()
                & platform["Symbol"].ne("")
                & platform["Symbol"].ne("---")
            ].copy()

            available = set(valid["Symbol"])

            panel = set(artifact["common_genes"])
            selected = set(artifact["selected_genes"])

            print("\nNATIVE ANNOTATION GENE COVERAGE")
            print("Reference genes:", len(panel & available),
                  "/", len(panel))
            print("Model genes:", len(selected & available),
                  "/", len(selected))
            print("Missing model genes:",
                  sorted(selected - available))

            platform.to_csv(
                OUT / "GPL14951_native_annotation.csv",
                index=False
            )

print("\nCompatibility inspection completed.")

In [ ]:
import pandas as pd
import joblib
import gzip
import csv
from pathlib import Path

ROOT = Path("/content/ML_Project/ML_Project-main")
OUT = ROOT / "external_validation_v4/external_GSE89632"

matrix_path = OUT / "GSE89632_series_matrix.txt.gz"

model_path = (
    ROOT / "external_validation_v4/"
    "frozen_candidate_v1/binary_candidate_v1.joblib"
)

# The native GPL14951 annotation was downloaded in the previous cell.
if "platform" not in globals():
    raise RuntimeError(
        "The platform DataFrame is missing. "
        "Please rerun the previous GPL14951 download cell."
    )

native = platform.copy()

native["ID"] = native["ID"].astype("string").str.strip()
native["Symbol"] = native["Symbol"].astype("string").str.strip()

# Read probe IDs from GSE89632
probe_ids = []

with gzip.open(matrix_path, "rt", errors="replace") as f:
    inside_table = False

    for line in f:

        if line.strip() == "!series_matrix_table_begin":
            inside_table = True
            continue

        if line.strip() == "!series_matrix_table_end":
            break

        if not inside_table:
            continue

        fields = next(csv.reader([line], delimiter="\t"))

        if fields and fields[0].startswith("ILMN_"):
            probe_ids.append(fields[0])

expression_probes = set(probe_ids)
native_probes = set(native["ID"].dropna())

matched_probes = expression_probes & native_probes

# Keep nonempty, human gene annotations.
valid = native[
    native["ID"].isin(matched_probes)
    & native["Symbol"].notna()
    & native["Symbol"].ne("")
    & native["Symbol"].ne("---")
    & native["Species"].astype("string").str.contains(
        "Homo sapiens", case=False, na=False
    )
].copy()

artifact = joblib.load(model_path)

model_genes = set(artifact["selected_genes"])
reference_genes = set(artifact["common_genes"])
available_genes = set(valid["Symbol"])

print("=" * 60)
print("NATIVE GPL14951 COMPATIBILITY REPORT")
print("=" * 60)

print("Expression probes:", len(expression_probes))
print("Native platform probes:", len(native_probes))
print("Matching probes:", len(matched_probes))

print(
    "Probe match percentage:",
    round(
        100 * len(matched_probes) / len(expression_probes),
        2
    ) if expression_probes else 0,
    "%"
)

print("\nGENE COVERAGE")
print("Mapped human gene symbols:", len(available_genes))
print("Model genes available:", len(model_genes & available_genes), "/ 50")
print(
    "Reference genes available:",
    len(reference_genes & available_genes),
    "/",
    len(reference_genes)
)

print("\nMISSING MODEL GENES")
print(sorted(model_genes - available_genes))

print("\nEXAMPLE ANNOTATIONS")
display(valid[["ID", "Symbol", "Species"]].head(10))

valid.to_csv(
    OUT / "GPL14951_verified_probe_mapping.csv",
    index=False
)

print("\nNative probe mapping saved successfully.")
print("Frozen model remains unchanged.")

In [ ]:
import pandas as pd
import numpy as np
import joblib
import gzip
import io
import csv
import json

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

ROOT = Path("/content/ML_Project/ML_Project-main")

DATA = ROOT / "external_validation_v4/external_GSE89632"
MODEL = ROOT / "external_validation_v4/frozen_candidate_v1/binary_candidate_v1.joblib"
OUT = DATA / "approximate_input_evaluation"
OUT.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 1. Load frozen model
# --------------------------------------------------

artifact = joblib.load(MODEL)
model = artifact["model"]

selected_genes = list(artifact["selected_genes"])
reference_genes = list(artifact["common_genes"])

assert len(selected_genes) == 50
assert len(reference_genes) == 10152
assert list(model.classes_) == [0, 1]

print("Frozen model loaded successfully.")

# --------------------------------------------------
# 2. Load previously verified native probe mapping
# --------------------------------------------------

mapping = pd.read_csv(
    DATA / "GPL14951_verified_probe_mapping.csv",
    dtype=str,
    low_memory=False
)

mapping = mapping[["ID", "Symbol"]].copy()
mapping = mapping.dropna()

mapping["ID"] = mapping["ID"].str.strip()
mapping["Symbol"] = mapping["Symbol"].str.strip()

# Exclude missing and ambiguous/multiple symbol annotations.
mapping = mapping[
    (mapping["Symbol"] != "")
    & (mapping["Symbol"] != "---")
    & (~mapping["Symbol"].str.contains(
        r"///|//|;|\|", regex=True, na=False
    ))
]

# Avoid accidental duplicate probe joins.
mapping = mapping.drop_duplicates(subset=["ID"])

# --------------------------------------------------
# 3. Read the complete expression matrix
# --------------------------------------------------

matrix_file = DATA / "GSE89632_series_matrix.txt.gz"

table_lines = []
sample_titles = []
sample_ids = []

inside = False

with gzip.open(matrix_file, "rt", errors="replace") as f:

    for line in f:

        if line.startswith("!Sample_title\t"):
            sample_titles = next(
                csv.reader([line], delimiter="\t")
            )[1:]

        if line.startswith("!Sample_geo_accession\t"):
            sample_ids = next(
                csv.reader([line], delimiter="\t")
            )[1:]

        if line.strip() == "!series_matrix_table_begin":
            inside = True
            continue

        if line.strip() == "!series_matrix_table_end":
            break

        if inside:
            table_lines.append(line)

expression = pd.read_csv(
    io.StringIO("".join(table_lines)),
    sep="\t",
    index_col=0
)

assert expression.shape[1] == 63
assert len(sample_ids) == len(sample_titles) == 63
assert list(expression.columns) == sample_ids
assert expression.index.is_unique

expression = expression.apply(
    pd.to_numeric, errors="raise"
)

assert np.isfinite(expression.to_numpy()).all()

print("Expression matrix:", expression.shape)

# --------------------------------------------------
# 4. Convert probes to gene symbols
# --------------------------------------------------

probe_to_gene = mapping.set_index("ID")["Symbol"]

matched_probes = expression.index.intersection(
    probe_to_gene.index
)

gene_expression = expression.loc[matched_probes].copy()

gene_expression["GeneSymbol"] = probe_to_gene.loc[
    matched_probes
].to_numpy()

# Aggregate probes mapped to the same gene.
# Median is a fixed, label-independent microarray rule.
gene_expression = gene_expression.groupby(
    "GeneSymbol", sort=False
).median()

print("Gene-level matrix:", gene_expression.shape)

# --------------------------------------------------
# 5. Check reference-panel overlap
# --------------------------------------------------

available_reference = [
    gene for gene in reference_genes
    if gene in gene_expression.index
]

missing_reference = [
    gene for gene in reference_genes
    if gene not in gene_expression.index
]

missing_selected = [
    gene for gene in selected_genes
    if gene not in gene_expression.index
]

print("Reference genes available:", len(available_reference))
print("Reference genes missing:", len(missing_reference))
print("Selected genes missing:", missing_selected)

assert len(available_reference) == 9752, (
    "Unexpected reference-panel coverage. "
    "Inspect the probe filtering and mapping rules."
)

assert not missing_selected, (
    "A required model gene is missing. "
    "Do not impute or replace it."
)

# --------------------------------------------------
# 6. Calculate approximate within-sample ranks
# --------------------------------------------------

reference_expression = gene_expression.loc[
    available_reference
]

# Each sample is ranked against 9,752 genes,
# NOT the original 10,152-gene training panel.
ranked = reference_expression.rank(
    axis=0,
    method="average",
    pct=True
)

# Samples as rows; 50 model genes as columns.
X_external = ranked.loc[selected_genes].T

assert list(X_external.columns) == selected_genes
assert X_external.shape == (63, 50)
assert not X_external.isna().any().any()

# --------------------------------------------------
# 7. Reconstruct source-group labels
# --------------------------------------------------

def get_group(title):
    t = title.upper()

    if "_NASH_" in t:
        return "NASH"
    if "_SS_" in t:
        return "SS"
    if "_HC_" in t:
        return "Healthy"

    raise ValueError(f"Unrecognized sample title: {title}")

groups = [get_group(title) for title in sample_titles]

# Source group meanings:
# Healthy = 0, simple steatosis/NASH = 1.
y_true = np.array([
    0 if group == "Healthy" else 1
    for group in groups
])

assert dict(pd.Series(groups).value_counts()) == {
    "Healthy": 24,
    "SS": 20,
    "NASH": 19
}

# --------------------------------------------------
# 8. Apply the frozen classifier
# --------------------------------------------------

prob_disease = model.predict_proba(
    X_external
)[:, 1]

# Original classifier decision rule: probability >= 0.5
y_pred = (prob_disease >= 0.5).astype(int)

# --------------------------------------------------
# 9. Calculate evaluation metrics
# --------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

print("\nGSE89632 — APPROXIMATE-INPUT EVALUATION")
print("=" * 55)

print("\nConfusion matrix:")
display(pd.DataFrame(
    cm,
    index=["Actual Healthy", "Actual Disease"],
    columns=["Predicted Healthy", "Predicted Disease"]
))

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Healthy", "Disease"],
    zero_division=0
))

summary = {
    "accuracy": accuracy_score(y_true, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "macro_f1": f1_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    ),
    "roc_auc": roc_auc_score(y_true, prob_disease),
    "healthy_recall_specificity": cm[0, 0] / cm[0].sum(),
    "disease_recall_sensitivity": cm[1, 1] / cm[1].sum(),
    "evaluation_status": (
        "Approximate-input sensitivity analysis; "
        "not exact-preprocessing external validation"
    )
}

display(pd.DataFrame([summary]).T)

# --------------------------------------------------
# 10. Save reproducible results
# --------------------------------------------------

pd.DataFrame({
    "GSM_ID": sample_ids,
    "original_group": groups,
    "true_label": y_true,
    "predicted_label": y_pred,
    "prob_disease": prob_disease
}).to_csv(
    OUT / "GSE89632_predictions.csv",
    index=False
)

pd.DataFrame({
    "gene": missing_reference
}).to_csv(
    OUT / "missing_reference_genes.csv",
    index=False
)

pd.DataFrame(
    [summary]
).to_csv(
    OUT / "GSE89632_metrics.csv",
    index=False
)

X_external.to_csv(
    OUT / "GSE89632_model_input_approximate.csv.gz",
    compression="gzip"
)

with open(OUT / "preprocessing_manifest.json", "w") as f:
    json.dump({
        "source_dataset": "GSE89632",
        "source_platform": "GPL14951",
        "probe_to_gene": "Native GPL14951 Symbol field",
        "duplicate_probe_aggregation": "Median",
        "reference_panel_expected": 10152,
        "reference_panel_available": len(available_reference),
        "ranking": (
            "Within-sample average percentile ranks "
            "over available reference genes"
        ),
        "missing_genes_imputed": False,
        "model_refitted": False,
        "feature_selection_repeated": False,
        "threshold": 0.5,
        "validation_type": "Approximate-input sensitivity analysis"
    }, f, indent=2)

print("\nAll evaluation files saved to:", OUT)
print("Frozen model was not modified.")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    confusion_matrix
)

ROOT = Path("/content/ML_Project/ML_Project-main")

OUT = (
    ROOT / "external_validation_v4/"
    "external_GSE89632/approximate_input_evaluation"
)

path = OUT / "GSE89632_predictions.csv"
df = pd.read_csv(path)

assert len(df) == 63
assert df["GSM_ID"].is_unique
assert set(df["true_label"].unique()) == {0, 1}

# 1. Examine each original disease group.
group_summary = (
    df.groupby("original_group")
    .agg(
        samples=("GSM_ID", "count"),
        predicted_healthy=(
            "predicted_label",
            lambda x: int((x == 0).sum())
        ),
        predicted_disease=(
            "predicted_label",
            lambda x: int((x == 1).sum())
        ),
        mean_disease_probability=("prob_disease", "mean"),
        median_disease_probability=("prob_disease", "median")
    )
    .reset_index()
)

print("PER-GROUP PREDICTION ANALYSIS")
display(group_summary.round(4))

# 2. List the missed disease cases.
missed = df[
    (df["true_label"] == 1) &
    (df["predicted_label"] == 0)
].sort_values("prob_disease")

print("\nMISSED DISEASE CASES")
display(missed)

missed.to_csv(
    OUT / "missed_disease_cases.csv",
    index=False
)

# 3. Stratified bootstrap confidence intervals.
# The fitted model and original 0.5 threshold remain unchanged.
rng = np.random.default_rng(42)

healthy = df[df["true_label"] == 0]
disease = df[df["true_label"] == 1]

records = []

for _ in range(2000):

    h = healthy.iloc[
        rng.integers(0, len(healthy), len(healthy))
    ]

    d = disease.iloc[
        rng.integers(0, len(disease), len(disease))
    ]

    sample = pd.concat([h, d], ignore_index=True)

    y = sample["true_label"].to_numpy()
    p = sample["predicted_label"].to_numpy()
    scores = sample["prob_disease"].to_numpy()

    cm = confusion_matrix(y, p, labels=[0, 1])

    records.append({
        "roc_auc": roc_auc_score(y, scores),
        "balanced_accuracy": balanced_accuracy_score(y, p),
        "specificity": cm[0, 0] / cm[0].sum(),
        "sensitivity": cm[1, 1] / cm[1].sum()
    })

boot = pd.DataFrame(records)

ci = boot.quantile([0.025, 0.975]).T
ci.columns = ["lower_95", "upper_95"]

point = {
    "roc_auc": roc_auc_score(
        df["true_label"], df["prob_disease"]
    ),
    "balanced_accuracy": balanced_accuracy_score(
        df["true_label"], df["predicted_label"]
    ),
    "specificity": (
        (df.query("true_label == 0")["predicted_label"] == 0).mean()
    ),
    "sensitivity": (
        (df.query("true_label == 1")["predicted_label"] == 1).mean()
    )
}

ci.insert(
    0, "estimate",
    pd.Series(point)
)

print("\nPERFORMANCE WITH STRATIFIED BOOTSTRAP 95% INTERVALS")
display(ci.round(4))

group_summary.to_csv(
    OUT / "subgroup_analysis.csv",
    index=False
)

ci.to_csv(
    OUT / "bootstrap_confidence_intervals.csv"
)

print("\nAnalysis saved:", OUT)
print("Frozen model and prediction threshold unchanged.")

In [ ]:
import pandas as pd
from pathlib import Path
from scipy.stats import binomtest
from sklearn.metrics import roc_auc_score

ROOT = Path("/content/ML_Project/ML_Project-main")

OUT = (
    ROOT / "external_validation_v4/"
    "external_GSE89632/approximate_input_evaluation"
)

df = pd.read_csv(OUT / "GSE89632_predictions.csv")

assert len(df) == 63
assert df["GSM_ID"].is_unique

def exact_interval(correct, total):
    interval = binomtest(correct, total).proportion_ci(
        confidence_level=0.95,
        method="exact"
    )
    return interval.low, interval.high

records = []

for group, correct_label in [
    ("Healthy", 0),
    ("SS", 1),
    ("NASH", 1)
]:
    subset = df[df["original_group"] == group]

    correct = int(
        (subset["predicted_label"] == correct_label).sum()
    )
    total = len(subset)

    lower, upper = exact_interval(correct, total)

    records.append({
        "group": group,
        "correct": correct,
        "total": total,
        "recall": correct / total,
        "exact_95_lower": lower,
        "exact_95_upper": upper
    })

group_ci = pd.DataFrame(records)

print("GROUP RECALL WITH EXACT 95% CONFIDENCE INTERVALS")
display(group_ci.round(4))

auc_records = []

for disease_group in ["SS", "NASH"]:
    subset = df[
        df["original_group"].isin(["Healthy", disease_group])
    ]

    auc_records.append({
        "comparison": f"Healthy vs {disease_group}",
        "samples": len(subset),
        "roc_auc": roc_auc_score(
            subset["true_label"],
            subset["prob_disease"]
        )
    })

subgroup_auc = pd.DataFrame(auc_records)

print("\nEXPLORATORY SUBGROUP ROC-AUC")
display(subgroup_auc.round(4))

group_ci.to_csv(
    OUT / "exact_group_confidence_intervals.csv",
    index=False
)

subgroup_auc.to_csv(
    OUT / "subgroup_roc_auc.csv",
    index=False
)

print("\nStatistical report saved.")
print("No retraining or threshold changes performed.")

In [ ]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

ROOT = Path("/content/ML_Project/ML_Project-main")

OUT = (
    ROOT / "external_validation_v4/"
    "external_GSE89632/approximate_input_evaluation"
)

MODEL = (
    ROOT / "external_validation_v4/"
    "frozen_candidate_v1/binary_candidate_v1.joblib"
)

artifact = joblib.load(MODEL)
pipeline = artifact["model"]
genes = list(artifact["selected_genes"])

X = pd.read_csv(
    OUT / "GSE89632_model_input_approximate.csv.gz",
    index_col=0
)

predictions = pd.read_csv(
    OUT / "GSE89632_predictions.csv"
)

X = X.loc[predictions["GSM_ID"], genes]

N_FULL = len(artifact["common_genes"])
N_AVAILABLE = 9752
N_MISSING = N_FULL - N_AVAILABLE

assert N_FULL == 10152
assert N_MISSING == 400
assert X.shape == (63, 50)

# Confirm that saved inputs reproduce the original predictions.
original_probs = pipeline.predict_proba(X)[:, 1]

assert np.allclose(
    original_probs,
    predictions["prob_disease"].to_numpy(),
    atol=1e-8
), "Saved predictions do not match the loaded model."

# Convert approximate ranks to their observed rank positions.
observed_rank = X.to_numpy() * N_AVAILABLE

# A missing gene could rank below or above a selected gene.
# These represent conservative per-feature bounds.
lower_rank = observed_rank / N_FULL
upper_rank = (observed_rank + N_MISSING) / N_FULL

scaler = pipeline.named_steps["scaler"]
classifier = pipeline.named_steps["classifier"]

assert list(classifier.classes_) == [0, 1]

# Convert logistic-regression coefficients into raw input space.
coef = classifier.coef_[0] / scaler.scale_
intercept = (
    classifier.intercept_[0]
    - np.sum(coef * scaler.mean_)
)

# Conservative bounds for each sample's log-odds.
logit_lower = intercept + np.sum(
    np.where(coef >= 0, lower_rank, upper_rank) * coef,
    axis=1
)

logit_upper = intercept + np.sum(
    np.where(coef >= 0, upper_rank, lower_rank) * coef,
    axis=1
)

# Convert to disease probabilities.
from scipy.special import expit

prob_lower = expit(logit_lower)
prob_upper = expit(logit_upper)

# The frozen 0.5 threshold corresponds to log-odds 0.
stable_healthy = logit_upper < 0
stable_disease = logit_lower >= 0
uncertain = ~(stable_healthy | stable_disease)

audit = predictions.copy()
audit["prob_lower_bound"] = prob_lower
audit["prob_upper_bound"] = prob_upper
audit["prediction_stable"] = ~uncertain

print("MISSING-PANEL ROBUSTNESS AUDIT")
print("=" * 55)

print("Total samples:", len(audit))
print("Predictions stable under bounds:", (~uncertain).sum())
print("Predictions potentially changeable:", uncertain.sum())

print("\nSTABILITY BY ORIGINAL GROUP")

summary = (
    audit.groupby("original_group")
    .agg(
        samples=("GSM_ID", "size"),
        stable_predictions=("prediction_stable", "sum")
    )
)

summary["potentially_changeable"] = (
    summary["samples"] - summary["stable_predictions"]
)

display(summary)

print("\nCASES WITH POSSIBLY CHANGEABLE PREDICTIONS")

display(
    audit.loc[
        ~audit["prediction_stable"],
        [
            "GSM_ID",
            "original_group",
            "prob_disease",
            "prob_lower_bound",
            "prob_upper_bound"
        ]
    ].round(4)
)

audit.to_csv(
    OUT / "missing_panel_robustness_audit.csv",
    index=False
)

print("\nAudit saved successfully.")
print("Frozen model and original predictions unchanged.")

In [ ]:
import pandas as pd
from pathlib import Path

root = Path("/content/ML_Project/ML_Project-main")
out = root / "external_validation_v4/external_GSE89632/approximate_input_evaluation"

audit = pd.read_csv(out / "missing_panel_robustness_audit.csv")
metrics = pd.read_csv(out / "GSE89632_metrics.csv")
intervals = pd.read_csv(
    out / "exact_group_confidence_intervals.csv"
)

print("EXTERNAL VALIDATION SUMMARY")
display(metrics.T)

print("\nSTABLE VERSUS POTENTIALLY CHANGEABLE DECISIONS")
stability = pd.crosstab(
    audit["original_group"],
    audit["prediction_stable"],
    rownames=["Clinical group"],
    colnames=["Prediction stable"]
)
display(stability)

print("\nEXACT SUBGROUP CONFIDENCE INTERVALS")
display(intervals.round(4))

# Preserve a simple, auditable summary.
report = pd.DataFrame([{
    "dataset": "GSE89632",
    "platform": "GPL14951",
    "samples": 63,
    "model": "frozen_candidate_v1",
    "model_modified": False,
    "selected_genes_available": 50,
    "reference_genes_available": 9752,
    "reference_genes_expected": 10152,
    "roc_auc": float(metrics.loc[0, "roc_auc"]),
    "balanced_accuracy": float(metrics.loc[0, "balanced_accuracy"]),
    "sensitivity": float(metrics.loc[0, "disease_recall_sensitivity"]),
    "specificity": float(metrics.loc[0, "healthy_recall_specificity"]),
    "stable_decisions": int(audit["prediction_stable"].sum()),
    "potentially_changeable_decisions": int(
        (~audit["prediction_stable"].astype(bool)).sum()
    ),
    "validation_status": "Independent cohort, approximate preprocessing"
}])

report.to_csv(
    out / "FINAL_GSE89632_VALIDATION_SUMMARY.csv",
    index=False
)

print("\nFinal GSE89632 summary saved.")

In [ ]:
from pathlib import Path
import requests
import gzip
import csv
import pandas as pd

ROOT = Path("/content/ML_Project/ML_Project-main")
OUT = ROOT / "external_validation_v4/external_GSE130970"
OUT.mkdir(parents=True, exist_ok=True)

url = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/"
    "GSE130nnn/GSE130970/matrix/"
    "GSE130970_series_matrix.txt.gz"
)

destination = OUT / "GSE130970_series_matrix.txt.gz"

if not destination.exists() or destination.stat().st_size == 0:
    print("Downloading GSE130970 metadata...")
    with requests.get(url, stream=True, timeout=180) as response:
        response.raise_for_status()
        with open(destination, "wb") as f:
            for chunk in response.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)

metadata = {}

with gzip.open(destination, "rt", errors="replace") as f:
    for line in f:
        if line.startswith("!series_matrix_table_begin"):
            break
        if line.startswith("!Sample_"):
            fields = next(csv.reader([line], delimiter="\t"))
            metadata[fields[0]] = fields[1:]

sample_ids = metadata.get("!Sample_geo_accession", [])
print("\nSamples found:", len(sample_ids))
print("Sample IDs unique:", len(sample_ids) == len(set(sample_ids)))

print("\nAVAILABLE METADATA FIELDS")
for key in metadata:
    if any(term in key.lower() for term in [
        "title", "characteristics", "description",
        "source", "relation"
    ]):
        print(key)

print("\nCLINICAL METADATA EXAMPLES")
for key, values in metadata.items():
    if "characteristics" in key.lower():
        print("\n", key)
        print(values[:8])

print("\nSAMPLE TITLES")
print(metadata.get("!Sample_title", [])[:10])

print("\nInspection finished. No predictions generated.")

In [ ]:
import pandas as pd
import gzip
import csv
from pathlib import Path
from collections import Counter

ROOT = Path("/content/ML_Project/ML_Project-main")
OUT = ROOT / "external_validation_v4/external_GSE130970"
FILE = OUT / "GSE130970_series_matrix.txt.gz"

assert FILE.is_file(), "GSE130970 series matrix not found."

# Keep repeated metadata rows instead of overwriting them.
metadata_rows = []

with gzip.open(FILE, "rt", errors="replace") as f:
    for line in f:
        if line.startswith("!series_matrix_table_begin"):
            break

        if line.startswith("!Sample_"):
            fields = next(csv.reader([line], delimiter="\t"))
            metadata_rows.append((fields[0], fields[1:]))

# Build one row per sample.
accession_rows = [
    values for key, values in metadata_rows
    if key == "!Sample_geo_accession"
]

assert len(accession_rows) == 1
gsm_ids = accession_rows[0]
assert len(gsm_ids) == 78
assert len(set(gsm_ids)) == 78

clinical = pd.DataFrame({"GSM_ID": gsm_ids})

field_counts = Counter()

for key, values in metadata_rows:
    if len(values) != 78:
        print("Skipping mismatched field:", key)
        continue

    field_counts[key] += 1

    if key in [
        "!Sample_title",
        "!Sample_source_name_ch1",
        "!Sample_characteristics_ch1",
        "!Sample_description"
    ]:
        column = (
            key.replace("!Sample_", "")
            + "_" + str(field_counts[key])
        )
        clinical[column] = values

print("CLINICAL METADATA COLUMNS")
print(clinical.columns.tolist())

print("\nFIRST 10 SAMPLES")
display(clinical.head(10))

print("\nCHARACTERISTIC VALUES AND COUNTS")

characteristic_columns = [
    c for c in clinical.columns
    if "characteristics" in c
]

for column in characteristic_columns:
    print("\n", column)
    print(clinical[column].value_counts(dropna=False).head(20))

clinical.to_csv(
    OUT / "GSE130970_complete_metadata_audit.csv",
    index=False
)

print("\nComplete metadata saved.")
print("No clinical labels assigned yet.")
print("Frozen model unchanged.")

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path("/content/ML_Project/ML_Project-main")
OUT = ROOT / "external_validation_v4/external_GSE130970"

clinical = pd.read_csv(
    OUT / "GSE130970_complete_metadata_audit.csv"
)

fields = {
    "inflammation": "characteristics_ch1_4",
    "ballooning": "characteristics_ch1_5",
    "steatosis": "characteristics_ch1_6",
    "nas": "characteristics_ch1_7",
    "fibrosis": "characteristics_ch1_8"
}

for name, column in fields.items():
    clinical[name] = pd.to_numeric(
        clinical[column].astype(str).str.extract(
            r":\s*(\d+)", expand=False
        ),
        errors="coerce"
    )

assert clinical[list(fields)].notna().all().all()
assert len(clinical) == 78

# Flag possible histologically normal samples.
clinical["candidate_normal"] = (
    (clinical["steatosis"] == 0)
    & (clinical["ballooning"] == 0)
    & (clinical["inflammation"] == 0)
)

columns = [
    "GSM_ID",
    "title_1",
    "steatosis",
    "ballooning",
    "inflammation",
    "nas",
    "fibrosis"
]

print("POTENTIAL HISTOLOGICALLY NORMAL SAMPLES")
candidates = clinical.loc[
    clinical["candidate_normal"], columns
]

display(candidates)

print("\nNumber of candidates:", len(candidates))

print("\nALL EIGHT STEATOSIS-GRADE-0 SAMPLES")
display(
    clinical.loc[
        clinical["steatosis"] == 0,
        columns
    ]
)

# Check consistency of the recorded NAS components.
clinical["calculated_nas"] = (
    clinical["steatosis"]
    + clinical["ballooning"]
    + clinical["inflammation"]
)

mismatch = clinical.loc[
    clinical["calculated_nas"] != clinical["nas"]
]

print("\nNAS COMPONENT MISMATCHES:", len(mismatch))

if not mismatch.empty:
    display(mismatch[
        columns + ["calculated_nas"]
    ])

clinical.to_csv(
    OUT / "GSE130970_histology_audit.csv",
    index=False
)

print("\nAudit saved. No clinical labels assigned.")

In [ ]:
import requests
import pandas as pd
import gzip
from pathlib import Path

ROOT = Path("/content/ML_Project/ML_Project-main")
OUT = ROOT / "external_validation_v4/external_GSE130970"
OUT.mkdir(parents=True, exist_ok=True)

filename = (
    "GSE130970_all_sample_salmon_tximport_"
    "counts_entrez_gene_ID.csv.gz"
)

url = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/"
    "GSE130nnn/GSE130970/suppl/" + filename
)

path = OUT / filename

if not path.exists() or path.stat().st_size == 0:
    print("Downloading gene-level count matrix...")
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)

print("Download complete.")
print("File size (MB):", round(path.stat().st_size / 1e6, 2))

print("\nFIRST FIVE ROWS")
preview = pd.read_csv(
    path,
    compression="gzip",
    nrows=5
)
display(preview)

print("\nCOLUMN NAMES")
print(preview.columns[:12].tolist())

print("\nCOLUMN COUNT")
print(len(preview.columns))

print("\nFIRST COLUMN EXAMPLES")
print(preview.iloc[:, 0].tolist())

print("\nMetadata audit remains unchanged.")
print("No clinical labels assigned.")
print("Frozen model unchanged.")

### Original cell 191 — incomplete, disabled

**Reason:** unterminated string literal (detected at line 156) (source line 156). This cell is preserved for review, not execution.

```python
import pandas as pd
import numpy as np
import joblib
import requests
from pathlib import Path

ROOT = Path("/content/ML_Project/ML_Project-main")
DATA = ROOT / "external_validation_v4/external_GSE130970"
MODEL = ROOT / "external_validation_v4/frozen_candidate_v1/binary_candidate_v1.joblib"

counts_path = DATA / (
    "GSE130970_all_sample_salmon_tximport_"
    "counts_entrez_gene_ID.csv.gz"
)

metadata_path = DATA / "GSE130970_complete_metadata_audit.csv"

# 1. Load source files
counts = pd.read_csv(
    counts_path,
    compression="gzip",
    dtype={"entrez_id": str}
)

metadata = pd.read_csv(metadata_path)

assert "entrez_id" in counts.columns
assert len(metadata) == 78
assert metadata["GSM_ID"].is_unique
assert metadata["title_1"].is_unique

expected_columns = metadata["title_1"].astype(str).tolist()
observed_columns = counts.columns.drop("entrez_id").tolist()

assert len(observed_columns) == 78
assert len(set(observed_columns)) == 78
assert set(expected_columns) == set(observed_columns), (
    "Count matrix sample names do not match metadata."
)

# Reorder columns using verified metadata order.
counts = counts[
    ["entrez_id"] + expected_columns
].copy()

counts["entrez_id"] = counts["entrez_id"].str.strip()

assert counts["entrez_id"].notna().all()
assert counts["entrez_id"].is_unique

values = counts[expected_columns].to_numpy(dtype=float)

assert np.isfinite(values).all(), "Non-finite count values detected"
assert (values >= 0).all(), "Negative count values detected"

print("SOURCE MATRIX")
print("Genes:", len(counts))
print("Samples:", len(expected_columns))
print("Sample alignment: PASSED")
print("Duplicate Entrez IDs: 0")

# 2. Download official human Entrez-to-symbol mapping
url = (
    "https://ftp.ncbi.nlm.nih.gov/gene/DATA/"
    "GENE_INFO/Mammalia/Homo_sapiens.gene_info.gz"
)

mapping_file = DATA / "Homo_sapiens.gene_info.gz"

if not mapping_file.exists() or mapping_file.stat().st_size == 0:
    print("\nDownloading NCBI human gene mapping...")

    with requests.get(url, stream=True, timeout=180) as response:
        response.raise_for_status()

        with open(mapping_file, "wb") as f:
            for chunk in response.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)

gene_info = pd.read_csv(
    mapping_file,
    sep="\t",
    compression="gzip",
    dtype=str,
    low_memory=False
)

gene_info.columns = gene_info.columns.str.lstrip("#")

assert "GeneID" in gene_info.columns
assert "Symbol" in gene_info.columns

gene_info = gene_info[
    ["GeneID", "Symbol"]
].drop_duplicates(subset=["GeneID"])

gene_info = gene_info.rename(
    columns={
        "GeneID": "entrez_id",
        "Symbol": "gene_symbol"
    }
)

# 3. Create verified mapping
mapped = counts[["entrez_id"]].merge(
    gene_info,
    on="entrez_id",
    how="left",
    validate="one_to_one"
)

valid = (
    mapped["gene_symbol"].notna()
    & mapped["gene_symbol"].ne("")
    & mapped["gene_symbol"].ne("-")
)

usable = mapped.loc[valid].copy()

artifact = joblib.load(MODEL)

selected_genes = set(artifact["selected_genes"])
reference_genes = set(artifact["common_genes"])
available_genes = set(usable["gene_symbol"])

# 4. Report mapping and coverage
print("\nENTREZ-TO-SYMBOL MAPPING")
print("=" * 55)

print("Source Entrez IDs:", len(mapped))
print("Successfully mapped IDs:", len(usable))
print("Unmapped IDs:", len(mapped) - len(usable))
print("Unique gene symbols:", len(available_genes))

duplicate_symbols = usable["gene_symbol"].duplicated().sum()
print("Additional rows sharing gene symbols:", duplicate_symbols)

print("\nFROZEN MODEL COMPATIBILITY")
print("=" * 55)

print(
    "Model genes available:",
    len(selected_genes & available_genes),
    "/",
    len(selected_genes)
)

print(
    "Reference-panel genes available:",
    len(reference_genes & available_genes),
    "/",
    len(reference_genes)
)

print("\nM
```


In [ ]:
import pandas as pd
import numpy as np
import joblib
import requests
from pathlib import Path

ROOT = Path("/content/ML_Project/ML_Project-main")
DATA = ROOT / "external_validation_v4/external_GSE130970"

MODEL = (
    ROOT / "external_validation_v4"
    / "frozen_candidate_v1"
    / "binary_candidate_v1.joblib"
)

COUNTS = DATA / (
    "GSE130970_all_sample_salmon_tximport_"
    "counts_entrez_gene_ID.csv.gz"
)

META = DATA / "GSE130970_complete_metadata_audit.csv"

# STEP 1: Load expression data and sample metadata
counts = pd.read_csv(COUNTS, dtype={"entrez_id": str})
metadata = pd.read_csv(META)

samples = metadata["title_1"].astype(str).tolist()

assert len(samples) == 78
assert len(set(samples)) == 78
assert set(samples) == set(counts.columns) - {"entrez_id"}

counts = counts[["entrez_id"] + samples].copy()
counts["entrez_id"] = counts["entrez_id"].str.strip()

assert counts["entrez_id"].is_unique

values = counts[samples].to_numpy(dtype=float)

assert np.isfinite(values).all()
assert (values >= 0).all()

print("SOURCE MATRIX")
print("Genes:", len(counts))
print("Samples:", len(samples))
print("Sample alignment: PASSED")

# STEP 2: Download official NCBI gene mapping
mapping_file = DATA / "Homo_sapiens.gene_info.gz"

url = (
    "https://ftp.ncbi.nlm.nih.gov/gene/DATA/"
    "GENE_INFO/Mammalia/Homo_sapiens.gene_info.gz"
)

if not mapping_file.exists() or mapping_file.stat().st_size == 0:
    print("\nDownloading human gene mapping...")

    response = requests.get(url, timeout=180)
    response.raise_for_status()
    mapping_file.write_bytes(response.content)

gene_info = pd.read_csv(
    mapping_file,
    sep="\t",
    dtype=str,
    low_memory=False
)

gene_info.columns = gene_info.columns.str.lstrip("#")

gene_info = gene_info[["GeneID", "Symbol"]]
gene_info = gene_info.drop_duplicates("GeneID")

gene_info = gene_info.rename(columns={
    "GeneID": "entrez_id",
    "Symbol": "gene_symbol"
})

# STEP 3: Map Entrez IDs to gene symbols
mapping = counts[["entrez_id"]].merge(
    gene_info,
    on="entrez_id",
    how="left",
    validate="one_to_one"
)

usable = mapping[
    mapping["gene_symbol"].notna()
    & mapping["gene_symbol"].ne("")
    & mapping["gene_symbol"].ne("-")
].copy()

# STEP 4: Check frozen model coverage
artifact = joblib.load(MODEL)

model_genes = set(artifact["selected_genes"])
reference_genes = set(artifact["common_genes"])
available_genes = set(usable["gene_symbol"])

print("\nENTREZ-TO-SYMBOL MAPPING")
print("Source Entrez IDs:", len(mapping))
print("Mapped IDs:", len(usable))
print("Unmapped IDs:", len(mapping) - len(usable))
print("Unique gene symbols:", len(available_genes))
print(
    "Duplicate gene symbols:",
    usable["gene_symbol"].duplicated().sum()
)

print("\nFROZEN MODEL COMPATIBILITY")
print(
    "Model genes:",
    len(model_genes & available_genes),
    "/",
    len(model_genes)
)
print(
    "Reference genes:",
    len(reference_genes & available_genes),
    "/",
    len(reference_genes)
)

print("\nMISSING MODEL GENES")
print(sorted(model_genes - available_genes))

print("\nEXAMPLE GENE MAPPINGS")
display(usable.head(10))

# STEP 5: Save the mapping audit
mapping.to_csv(
    DATA / "GSE130970_entrez_symbol_mapping_audit.csv",
    index=False
)

print("\nAUDIT COMPLETED SUCCESSFULLY")
print("Frozen model unchanged.")
print("No predictions generated.")

In [ ]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from math import ceil

ROOT = Path("/content/ML_Project/ML_Project-main")
DATA = ROOT / "external_validation_v4/external_GSE130970"

COUNTS = DATA / (
    "GSE130970_all_sample_salmon_tximport_"
    "counts_entrez_gene_ID.csv.gz"
)

MAPPING = DATA / "GSE130970_entrez_symbol_mapping_audit.csv"

MODEL = ROOT / (
    "external_validation_v4/"
    "frozen_candidate_v1/binary_candidate_v1.joblib"
)

OUT = DATA / "preprocessing_audit"
OUT.mkdir(parents=True, exist_ok=True)

# 1. Load the verified source data
counts = pd.read_csv(COUNTS, dtype={"entrez_id": str})
mapping = pd.read_csv(MAPPING, dtype=str)
artifact = joblib.load(MODEL)

metadata = pd.read_csv(
    DATA / "GSE130970_complete_metadata_audit.csv"
)

samples = metadata["title_1"].astype(str).tolist()

assert len(samples) == 78
assert set(samples) == set(counts.columns) - {"entrez_id"}
assert counts["entrez_id"].is_unique
assert mapping["entrez_id"].is_unique

counts = counts[["entrez_id"] + samples]

# 2. Join Entrez identifiers with gene symbols
combined = counts.merge(
    mapping[["entrez_id", "gene_symbol"]],
    on="entrez_id",
    how="left",
    validate="one_to_one"
)

combined = combined.dropna(subset=["gene_symbol"])

combined = combined[
    ~combined["gene_symbol"].isin(["", "-"])
]

# Check numeric expression values
values = combined[samples].to_numpy(dtype=float)

assert np.isfinite(values).all()
assert (values >= 0).all()

# 3. Aggregate mapped records by gene symbol
gene_counts = combined.groupby("gene_symbol")[
    samples
].sum()

print("Gene-level expression:", gene_counts.shape)

# 4. CPM normalization
library_sizes = gene_counts.sum(axis=0)

assert (library_sizes > 0).all()

cpm = gene_counts.div(library_sizes, axis=1) * 1e6

# 5. Expression filtering:
# CPM >= 1 in at least 20% of samples.
minimum_samples = ceil(0.20 * len(samples))

keep = (cpm >= 1).sum(axis=1) >= minimum_samples

filtered_cpm = cpm.loc[keep]

print("Minimum expressing samples:", minimum_samples)
print("Genes retained after CPM filter:", len(filtered_cpm))

# 6. Check the frozen reference panel
reference = list(artifact["common_genes"])
selected = list(artifact["selected_genes"])

available_before = set(gene_counts.index)
available_after = set(filtered_cpm.index)

missing_before = sorted(set(reference) - available_before)
missing_after = sorted(set(reference) - available_after)

selected_missing_before = sorted(
    set(selected) - available_before
)

selected_missing_after = sorted(
    set(selected) - available_after
)

print("\nGENE COVERAGE AUDIT")
print("=" * 55)

print("Reference panel expected:", len(reference))
print("Reference genes before filtering:",
      len(set(reference) & available_before))
print("Reference genes after filtering:",
      len(set(reference) & available_after))

print("Missing reference genes before filtering:",
      len(missing_before))
print("Missing reference genes after filtering:",
      len(missing_after))

print("\nSELECTED MODEL GENE AUDIT")
print("Selected genes expected:", len(selected))
print("Missing before filtering:", selected_missing_before)
print("Missing after filtering:", selected_missing_after)

# 7. Save results without modifying the classifier
pd.DataFrame({
    "gene": missing_before
}).to_csv(
    OUT / "missing_reference_before_filter.csv",
    index=False
)

pd.DataFrame({
    "gene": missing_after
}).to_csv(
    OUT / "missing_reference_after_filter.csv",
    index=False
)

pd.DataFrame({
    "gene": selected,
    "available_before_filter": [
        g in available_before for g in selected
    ],
    "available_after_filter": [
        g in available_after for g in selected
    ]
}).to_csv(
    OUT / "selected_gene_coverage.csv",
    index=False
)

filtered_cpm.to_csv(
    OUT / "GSE130970_filtered_CPM.csv.gz",
    compression="gzip"
)

print("\nPreprocessing audit saved:", OUT)
print("Clinical labels unchanged.")
print("No model predictions generated.")

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

ROOT = Path("/content/ML_Project/ML_Project-main")
DATA = ROOT / "external_validation_v4/external_GSE130970"
PREP = DATA / "preprocessing_audit"

artifact = joblib.load(
    ROOT / "external_validation_v4/frozen_candidate_v1/binary_candidate_v1.joblib"
)

reference = list(artifact["common_genes"])
selected = list(artifact["selected_genes"])

# 1. Load previously filtered CPM expression
cpm = pd.read_csv(
    PREP / "GSE130970_filtered_CPM.csv.gz",
    index_col=0
)

# 2. Check gene coverage
available = [g for g in reference if g in cpm.index]
missing = [g for g in reference if g not in cpm.index]

assert len(available) == 10005, "Unexpected reference gene coverage"
assert all(g in available for g in selected), "Required model gene missing"
assert cpm.index.is_unique

# 3. Reproduce within-sample percentile ranking approximately.
# log2(CPM + 1) is monotonic, so it does not change rank order.
ranked = cpm.loc[available].rank(
    axis=0,
    method="average",
    pct=True
)

X_external = ranked.loc[selected].T

assert X_external.shape == (78, 50)
assert np.isfinite(X_external.to_numpy()).all()

# 4. Load the previously audited histology metadata
clinical = pd.read_csv(
    DATA / "GSE130970_histology_audit.csv"
)

assert len(clinical) == 78
assert clinical["GSM_ID"].is_unique
assert clinical["title_1"].is_unique

# 5. Assign conservative PROVISIONAL categories
strict_normal = (
    (clinical["steatosis"] == 0)
    & (clinical["ballooning"] == 0)
    & (clinical["inflammation"] == 0)
    & (clinical["fibrosis"] == 0)
)

steatosis_present = clinical["steatosis"] > 0

clinical["provisional_group"] = np.select(
    [strict_normal, steatosis_present],
    ["strict_normal_candidate", "steatosis_present"],
    default="requires_clinical_review"
)

print("PROVISIONAL HISTOLOGY GROUPS")
print(clinical["provisional_group"].value_counts())

print("\nSAMPLES REQUIRING CLINICAL REVIEW")
display(
    clinical.loc[
        clinical["provisional_group"] == "requires_clinical_review",
        [
            "GSM_ID", "title_1", "steatosis",
            "ballooning", "inflammation",
            "nas", "fibrosis"
        ]
    ]
)

# 6. Verify that the external expression samples match metadata
expected_titles = clinical["title_1"].astype(str).tolist()

assert set(X_external.index) == set(expected_titles)
X_external = X_external.loc[expected_titles]

# 7. Save approximate-input features and provisional metadata
X_external.to_csv(
    PREP / "GSE130970_approximate_model_inputs.csv.gz",
    compression="gzip"
)

clinical.to_csv(
    PREP / "GSE130970_provisional_histology_groups.csv",
    index=False
)

pd.DataFrame({"gene": missing}).to_csv(
    PREP / "GSE130970_missing_147_reference_genes.csv",
    index=False
)

print("\nMODEL INPUT PREPARATION")
print("Input matrix:", X_external.shape)
print("Reference genes used:", len(available))
print("Reference genes missing:", len(missing))
print("Selected genes available:", len(selected))

print("\nFILES SAVED SUCCESSFULLY")
print("No predictions generated.")
print("No clinical labels finalized.")
print("Frozen classifier unchanged.")

In [ ]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from scipy.stats import binomtest

ROOT = Path("/content/ML_Project/ML_Project-main")
DATA = ROOT / "external_validation_v4/external_GSE130970"
PREP = DATA / "preprocessing_audit"
OUT = DATA / "strict_histology_exploratory_evaluation"
OUT.mkdir(parents=True, exist_ok=True)

model_path = (
    ROOT / "external_validation_v4"
    / "frozen_candidate_v1"
    / "binary_candidate_v1.joblib"
)

artifact = joblib.load(model_path)
model = artifact["model"]
selected = list(artifact["selected_genes"])

X = pd.read_csv(
    PREP / "GSE130970_approximate_model_inputs.csv.gz",
    index_col=0
)

clinical = pd.read_csv(
    PREP / "GSE130970_provisional_histology_groups.csv"
)

assert X.shape == (78, 50)
assert len(clinical) == 78
assert set(X.index) == set(clinical["title_1"])
assert set(selected) == set(X.columns)
assert list(model.classes_) == [0, 1]

clinical = clinical.set_index("title_1").loc[X.index].copy()

expected_counts = {
    "strict_normal_candidate": 4,
    "steatosis_present": 70,
    "requires_clinical_review": 4
}

assert clinical["provisional_group"].value_counts().to_dict() == expected_counts

# Fixed provisional label definitions; not selected using predictions.
labels = {
    "strict_normal_candidate": 0,
    "steatosis_present": 1
}

clinical["evaluation_label"] = (
    clinical["provisional_group"].map(labels)
)

# Produce unchanged-model predictions for all 78.
proba = model.predict_proba(X[selected])[:, 1]
pred = (proba >= 0.5).astype(int)

results = clinical.reset_index()[
    ["title_1", "GSM_ID", "provisional_group", "evaluation_label"]
].copy()

results["prob_disease"] = proba
results["predicted_label"] = pred

# Exclude clinically ambiguous cases from performance calculations.
evaluated = results.dropna(subset=["evaluation_label"]).copy()

y = evaluated["evaluation_label"].astype(int).to_numpy()
p = evaluated["prob_disease"].to_numpy()
yhat = evaluated["predicted_label"].to_numpy()

assert len(evaluated) == 74
assert (y == 0).sum() == 4
assert (y == 1).sum() == 70

tn, fp, fn, tp = confusion_matrix(
    y, yhat, labels=[0, 1]
).ravel()

def exact_ci(successes, total):
    ci = binomtest(
        int(successes), int(total)
    ).proportion_ci(
        confidence_level=0.95,
        method="exact"
    )
    return round(ci.low, 4), round(ci.high, 4)

metrics = pd.DataFrame([{
    "evaluation": "Strict-histology exploratory subset",
    "samples_evaluated": 74,
    "samples_excluded": 4,
    "reference_genes_available": 10005,
    "reference_genes_expected": 10152,
    "accuracy": accuracy_score(y, yhat),
    "balanced_accuracy": balanced_accuracy_score(y, yhat),
    "macro_f1": f1_score(y, yhat, average="macro"),
    "roc_auc": roc_auc_score(y, p),
    "specificity": tn / (tn + fp),
    "sensitivity": tp / (tp + fn),
    "TN": int(tn),
    "FP": int(fp),
    "FN": int(fn),
    "TP": int(tp)
}])

intervals = pd.DataFrame([
    {
        "metric": "specificity",
        "correct": int(tn),
        "total": int(tn + fp),
        "exact_95_CI": str(exact_ci(tn, tn + fp))
    },
    {
        "metric": "sensitivity",
        "correct": int(tp),
        "total": int(tp + fn),
        "exact_95_CI": str(exact_ci(tp, tp + fn))
    }
])

print("GSE130970 STRICT-HISTOLOGY EXPLORATORY EVALUATION")
display(metrics.T)

print("\nEXACT CONFIDENCE INTERVALS")
display(intervals)

print("\nEXCLUDED SAMPLES — PREDICTIONS FOR AUDIT ONLY")
display(
    results.loc[
        results["evaluation_label"].isna(),
        ["GSM_ID", "provisional_group",
         "prob_disease", "predicted_label"]
    ].round(4)
)

results.to_csv(
    OUT / "all_78_predictions_with_provisional_labels.csv",
    index=False
)

metrics.to_csv(
    OUT / "strict_histology_metrics.csv",
    index=False
)

intervals.to_csv(
    OUT / "exact_confidence_intervals.csv",
    index=False
)

print("\nResults saved to:", OUT)
print("Frozen model unchanged.")
print("Threshold unchanged at 0.5.")
print("No model retraining performed.")

In [ ]:
results = clinical.copy()
results["title_1"] = X.index.to_numpy()

results = results[
    ["title_1", "GSM_ID", "provisional_group", "evaluation_label"]
].reset_index(drop=True)

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from scipy.stats import binomtest

# ==============================================
# STEP 1: FILE PATHS
# ==============================================

ROOT = Path("/content/ML_Project/ML_Project-main")

DATA = ROOT / "external_validation_v4/external_GSE130970"
PREP = DATA / "preprocessing_audit"

OUT = DATA / "strict_histology_exploratory_evaluation"
OUT.mkdir(parents=True, exist_ok=True)

MODEL_FILE = (
    ROOT / "external_validation_v4"
    / "frozen_candidate_v1"
    / "binary_candidate_v1.joblib"
)

X_FILE = PREP / "GSE130970_approximate_model_inputs.csv.gz"

CLINICAL_FILE = (
    PREP / "GSE130970_provisional_histology_groups.csv"
)

# ==============================================
# STEP 2: CHECK FILES
# ==============================================

for file in [MODEL_FILE, X_FILE, CLINICAL_FILE]:
    if not file.exists():
        raise FileNotFoundError(f"Missing file: {file}")

print("All required files found.")

# ==============================================
# STEP 3: LOAD MODEL AND DATA
# ==============================================

artifact = joblib.load(MODEL_FILE)
model = artifact["model"]

genes = list(artifact["selected_genes"])

X = pd.read_csv(X_FILE, index_col=0)
clinical = pd.read_csv(CLINICAL_FILE)

X.index = X.index.astype(str)
clinical["title_1"] = clinical["title_1"].astype(str)

assert X.index.is_unique
assert clinical["title_1"].is_unique
assert len(X) == 78
assert len(clinical) == 78
assert all(g in X.columns for g in genes)
assert set(X.index) == set(clinical["title_1"])
assert list(model.classes_) == [0, 1]

print("Model and data loaded successfully.")

# ==============================================
# STEP 4: ALIGN METADATA WITH EXPRESSION
# ==============================================

clinical = clinical.set_index("title_1", drop=True)

clinical = clinical.loc[X.index].copy()

print("Metadata alignment successful.")

# ==============================================
# STEP 5: ASSIGN PROVISIONAL LABELS
# ==============================================

label_mapping = {
    "strict_normal_candidate": 0,
    "steatosis_present": 1,
    "requires_clinical_review": np.nan
}

clinical["evaluation_label"] = (
    clinical["provisional_group"].map(label_mapping)
)

print("\nClinical groups:")
print(clinical["provisional_group"].value_counts())

assert (
    clinical["provisional_group"].value_counts().to_dict()
    == {
        "steatosis_present": 70,
        "strict_normal_candidate": 4,
        "requires_clinical_review": 4
    }
)

# ==============================================
# STEP 6: RUN FROZEN MODEL
# ==============================================

probabilities = model.predict_proba(
    X.loc[:, genes]
)[:, 1]

predictions = (probabilities >= 0.5).astype(int)

print("\nFrozen model predictions generated successfully.")

# ==============================================
# STEP 7: CREATE RESULTS TABLE
# ==============================================

# Build explicitly to avoid pandas index-name issues.
results = pd.DataFrame({
    "title_1": X.index.to_numpy(),
    "GSM_ID": clinical["GSM_ID"].to_numpy(),
    "provisional_group": clinical["provisional_group"].to_numpy(),
    "evaluation_label": clinical["evaluation_label"].to_numpy(),
    "prob_disease": probabilities,
    "predicted_label": predictions
})

print("Results table created successfully.")

# ==============================================
# STEP 8: EXCLUDE AMBIGUOUS SAMPLES FROM METRICS
# ==============================================

evaluated = results.dropna(
    subset=["evaluation_label"]
).copy()

y_true = evaluated["evaluation_label"].astype(int).to_numpy()
y_pred = evaluated["predicted_label"].to_numpy()
y_prob = evaluated["prob_disease"].to_numpy()

assert len(evaluated) == 74
assert sum(y_true == 0) == 4
assert sum(y_true == 1) == 70

# ==============================================
# STEP 9: CALCULATE METRICS
# ==============================================

tn, fp, fn, tp = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
).ravel()

metrics = {
    "Samples evaluated": len(evaluated),
    "Samples excluded": 4,
    "Accuracy": accuracy_score(y_true, y_pred),
    "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
    "Macro F1": f1_score(y_true, y_pred, average="macro"),
    "ROC-AUC": roc_auc_score(y_true, y_prob),
    "Sensitivity": tp / (tp + fn),
    "Specificity": tn / (tn + fp),
    "True negatives": int(tn),
    "False positives": int(fp),
    "False negatives": int(fn),
    "True positives": int(tp)
}

print("\nGSE130970 EXTERNAL EVALUATION RESULTS")
print("=" * 55)

for key, value in metrics.items():
    print(f"{key}: {value}")

# ==============================================
# STEP 10: EXACT 95% CONFIDENCE INTERVALS
# ==============================================

def calculate_ci(correct, total):
    ci = binomtest(
        int(correct),
        int(total)
    ).proportion_ci(
        confidence_level=0.95,
        method="exact"
    )
    return (ci.low, ci.high)

specificity_ci = calculate_ci(tn, tn + fp)
sensitivity_ci = calculate_ci(tp, tp + fn)

print("\nEXACT 95% CONFIDENCE INTERVALS")
print("=" * 55)

print("Specificity CI:", specificity_ci)
print("Sensitivity CI:", sensitivity_ci)

# ==============================================
# STEP 11: REVIEW EXCLUDED SAMPLES
# ==============================================

excluded = results[
    results["evaluation_label"].isna()
].copy()

print("\nFOUR CLINICALLY AMBIGUOUS SAMPLES")

display(excluded)

# ==============================================
# STEP 12: SAVE EVERYTHING
# ==============================================

results.to_csv

## 08 — V3 pipeline packaging and integrity checks


In [ ]:
from google.colab import files
from pathlib import Path
import subprocess
import sys

# Verify the existing project.
root = Path("/content/ML_Project/ML_Project-main")

assert root.exists(), (
    "Project folder missing. Restore ML_Project-main first."
)

print("STEP 1: Upload final_masld_validation.py")
uploaded = files.upload()

script = Path("/content/final_masld_validation.py")

assert script.exists(), (
    "Please upload final_masld_validation.py"
)

print("\nSTEP 2: Running all validation checks...")

subprocess.run(
    [sys.executable, str(script)],
    check=True
)

print("\nSTEP 3: Preparing final results...")

output = (
    root / "external_validation_v4"
    / "FINAL_EXTERNAL_VALIDATION"
)

zip_file = output / "FINAL_VALIDATION_PACKAGE.zip"

assert zip_file.exists(), (
    "Final ZIP not found. Check the error output above."
)

print("\nSUCCESS — ALL CHECKS COMPLETED")
print("Results folder:", output)

# Download results to your computer.
files.download(str(zip_file))

In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import files

# STEP 1: Locate your uploaded script automatically
filename = "final_masld_validation.py"

possible_paths = [
    Path.cwd() / filename,
    Path("/content") / filename,
    Path("/content/ML_Project/ML_Project-main") / filename
]

script = next(
    (p for p in possible_paths if p.is_file()),
    None
)

if script is None:
    raise FileNotFoundError(
        "Cannot find final_masld_validation.py. "
        "Check the Colab Files panel."
    )

print("Script found:", script)

# STEP 2: Verify the project
root = Path("/content/ML_Project/ML_Project-main")

if not root.is_dir():
    raise FileNotFoundError(
        f"Project folder not found: {root}"
    )

print("Project folder found.")

# STEP 3: Run the final validation script
print("\nRunning final MASLD validation...\n")

subprocess.run(
    [sys.executable, str(script)],
    cwd=str(root),
    check=True
)

# STEP 4: Locate the final ZIP
output = (
    root / "external_validation_v4"
    / "FINAL_EXTERNAL_VALIDATION"
)

zip_file = output / "FINAL_VALIDATION_PACKAGE.zip"

if not zip_file.is_file():
    raise FileNotFoundError(
        f"Final ZIP not found: {zip_file}"
    )

# STEP 5: Download the results
print("\nVALIDATION COMPLETED SUCCESSFULLY")
print("ZIP:", zip_file)

files.download(str(zip_file))

In [ ]:
from pathlib import Path
import zipfile

root = Path("/content/ML_Project/ML_Project-main")

print("PROJECT EXISTS:", root.exists())
print("CURRENT DIRECTORY:", Path.cwd())

print("\nLOOKING FOR UPLOADED ZIP")

found = []

for folder in [Path.cwd(), Path("/content"), root]:
    if folder.exists():
        found.extend(folder.glob("masld_remaining_tasks*.zip"))

found = list(set(found))

for p in found:
    print("Found:", p)
    print("Valid ZIP:", zipfile.is_zipfile(p))

    if zipfile.is_zipfile(p):
        with zipfile.ZipFile(p) as z:
            print("Files inside ZIP:")
            for name in z.namelist():
                print("  ", name)

print("\nTOTAL MATCHING ZIP FILES:", len(found))

print("\nCHECKING FINAL OUTPUT FOLDER")

output = root / "external_validation_v4/FINAL_EXTERNAL_VALIDATION"

print("Output folder exists:", output.exists())

if output.exists():
    for p in output.iterdir():
        print("  ", p.name)

print("\nDIAGNOSTIC COMPLETED")

In [ ]:
from pathlib import Path
import zipfile
import subprocess
import sys

ROOT = Path("/content/ML_Project/ML_Project-main")
SOURCE = ROOT / "external_validation_v3"
WORK = ROOT / "external_validation_v4/final_completion_scripts"

WORK.mkdir(parents=True, exist_ok=True)

# Find the existing uploaded ZIP
zip_files = sorted(
    SOURCE.glob("masld_remaining_tasks*.zip"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

if not zip_files:
    raise FileNotFoundError("Uploaded ZIP not found.")

zip_path = zip_files[0]
print("Using ZIP:", zip_path, flush=True)

# Extract the scripts
with zipfile.ZipFile(zip_path) as z:
    for name in [
        "masld_final_preprocessing_audit.py",
        "build_masld_research_report.py"
    ]:
        z.extract(name, WORK)

print("Scripts extracted successfully.", flush=True)

# Run scripts separately and show their output
scripts = [
    "masld_final_preprocessing_audit.py",
    "build_masld_research_report.py"
]

for name in scripts:
    script = WORK / name

    print("\n" + "=" * 60, flush=True)
    print("RUNNING:", name, flush=True)
    print("=" * 60, flush=True)

    result = subprocess.run(
        [sys.executable, "-u", str(script)],
        cwd=str(ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    print(result.stdout, flush=True)
    print("Exit code:", result.returncode, flush=True)

    if result.returncode != 0:
        print("\nSTOPPED: The error is shown above.")
        break
else:
    print("\nBOTH SCRIPTS COMPLETED SUCCESSFULLY.")

    OUTPUT = ROOT / "external_validation_v4/FINAL_EXTERNAL_VALIDATION"
    FINDINGS = OUTPUT / "PREPROCESSING_AUDIT/PREPROCESSING_FINDINGS.md"
    REPORT = OUTPUT / "COMPREHENSIVE_RESEARCH_REPORT_DRAFT.md"

    print("Preprocessing audit exists:", FINDINGS.is_file())
    print("Research report exists:", REPORT.is_file())

    if FINDINGS.is_file():
        print("\nPREPROCESSING AUDIT FINDINGS:")
        print(FINDINGS.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path

root = Path("/content/ML_Project/ML_Project-main")
v3 = root / "external_validation_v3"

print("V3 PYTHON SCRIPTS")
print("=" * 50)

for p in sorted(v3.rglob("*.py")):
    if len(p.relative_to(v3).parts) <= 3:
        print(p.relative_to(root))

print("\nV3 MANIFESTS")
print("=" * 50)

for p in sorted(v3.rglob("*.json")):
    if any(word in p.name.lower() for word in [
        "manifest", "preprocess", "config"
    ]):
        print(p.relative_to(root))

In [ ]:
from pathlib import Path
import ast

ROOT = Path("/content/ML_Project/ML_Project-main")
SCRIPT = ROOT / "external_validation_v3/run_v3_pipeline.py"

assert SCRIPT.is_file(), "V3 pipeline script not found."

source = SCRIPT.read_text(encoding="utf-8")
tree = ast.parse(source)

keywords = [
    "prepare", "preprocess", "normalize",
    "rank", "filter", "common", "cpm",
    "expression", "gene"
]

sections = []

for node in ast.walk(tree):
    if isinstance(
        node,
        (ast.FunctionDef, ast.AsyncFunctionDef)
    ):
        if any(k in node.name.lower() for k in keywords):
            code = ast.get_source_segment(source, node)

            if code:
                sections.append(
                    f"\n{'=' * 70}\n"
                    f"FUNCTION: {node.name}\n"
                    f"{'=' * 70}\n"
                    f"{code}\n"
                )

print("Relevant functions found:", len(sections))

OUTPUT = ROOT / "external_validation_v3/V3_PREPROCESSING_SOURCE.txt"

OUTPUT.write_text(
    "\n".join(sections),
    encoding="utf-8"
)

print("Saved:", OUTPUT)

print("\nPREPROCESSING KEYWORD MATCHES")

for number, line in enumerate(source.splitlines(), 1):
    if any(
        keyword in line.lower()
        for keyword in [
            "rank(", "rank_genes", "cpm",
            "log2(", "common_genes",
            "filter_genes", "percentile"
        ]
    ):
        print(f"Line {number}: {line[:180]}")

print("\nExtraction completed.")

In [ ]:
from google.colab import files

files.download(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3/V3_PREPROCESSING_SOURCE.txt"
)

In [ ]:
from pathlib import Path

root = Path("/content/ML_Project/ML_Project-main")

source_file = root / "external_validation_v3/run_v3_pipeline.py"
output_file = root / "external_validation_v3/V3_PREPROCESSING_SOURCE.txt"

print("V3 source exists:", source_file.is_file())
print("Extracted file exists:", output_file.is_file())

if output_file.is_file():
    print("Extracted file size:", output_file.stat().st_size, "bytes")

    content = output_file.read_text(encoding="utf-8")

    if content.strip():
        print("\nEXTRACTED PREPROCESSING CONTENT:")
        print(content[:15000])
    else:
        print("The extracted file is empty.")

elif source_file.is_file():
    print("\nShowing relevant lines directly from the original V3 script:")

    keywords = [
        "preprocess", "normalize", "rank",
        "filter", "cpm", "common_genes",
        "log2", "percentile", "import"
    ]

    for line_number, line in enumerate(
        source_file.read_text(encoding="utf-8").splitlines(),
        1
    ):
        if any(k in line.lower() for k in keywords):
            print(f"{line_number}: {line[:200]}")

else:
    print("Original V3 script was not found.")

In [ ]:
from pathlib import Path
import ast
import zipfile

ROOT = Path("/content/ML_Project/ML_Project-main")
V3 = ROOT / "external_validation_v3"
SOURCE = V3 / "run_v3_pipeline.py"

assert SOURCE.is_file(), "V3 source script not found."

code = SOURCE.read_text(encoding="utf-8")

print("V3 SOURCE FILE CHECK")
print("=" * 50)
print("File:", SOURCE.name)
print("Size:", SOURCE.stat().st_size, "bytes")
print("Number of lines:", len(code.splitlines()))

# Check whether the script is valid Python.
try:
    tree = ast.parse(code)
    print("Python syntax: VALID")
except SyntaxError as e:
    print("Python syntax issue:", e)
    tree = None

# Display potentially relevant lines.
keywords = [
    "preprocess",
    "normalize",
    "rank",
    "filter",
    "cpm",
    "common_gene",
    "log2",
    "percentile",
    "subprocess",
    "import"
]

print("\nRELEVANT SOURCE LINES")
print("=" * 50)

for number, line in enumerate(code.splitlines(), 1):
    if any(word in line.lower() for word in keywords):
        print(f"{number}: {line[:220]}")

# Create a package with the original V3 script and
# nearby project Python source files.
output = V3 / "V3_SOURCE_REVIEW_PACKAGE.zip"

source_files = [SOURCE]

# Include nearby Python scripts for tracing imports.
for folder in [V3, ROOT]:
    for path in folder.glob("*.py"):
        if path not in source_files:
            source_files.append(path)

with zipfile.ZipFile(
    output, "w", zipfile.ZIP_DEFLATED
) as z:
    for path in source_files:
        z.write(path, arcname=str(path.relative_to(ROOT)))

print("\nSOURCE REVIEW PACKAGE CREATED")
print("ZIP:", output)
print("Files included:", len(source_files))
print("ZIP size:", output.stat().st_size, "bytes")

In [ ]:
from google.colab import files

files.download(
    "/content/ML_Project/ML_Project-main/"
    "external_validation_v3/V3_SOURCE_REVIEW_PACKAGE.zip"
)

## 09 — Model recovery, V2/V3 reconstruction and V4 investigation


In [ ]:
from pathlib import Path

root = Path("/content/ML_Project/ML_Project-main")
v4 = root / "external_validation_v4"

for pattern in ["*loso*summary*.csv", "*ablation*results*.csv",
                "*refinement*results*.csv", "*selected_genes*.csv"]:
    for file in v4.rglob(pattern):
        print(file.relative_to(root))

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

for base in ['/content/ML_Project', '/content/drive/MyDrive']:
    folder = Path(base)
    if folder.exists():
        for pattern in ['*.joblib', '*.pkl']:
            for file in folder.rglob(pattern):
                print(file)

In [ ]:

from pathlib import Path
import joblib

base = Path("/content/drive/MyDrive/MASLD_Validation_Backup")

files = [
    base / "v3_full_results/models/final_binary_full.joblib",
    base / "v3_full_results/models/final_binary_quick.joblib",
    base / "v3_quick_final/models/final_binary_quick.joblib",
    base / "baseline_v2/models/final_binary.joblib",
]

for path in files:
    print("\n" + "=" * 65)
    print("FILE:", path)
    if not path.exists():
        print("NOT FOUND")
        continue

    try:
        model = joblib.load(path)
        print("TYPE:", type(model).__name__)

        if hasattr(model, "steps"):
            print("PIPELINE STEPS:", [
                (name, type(step).__name__)
                for name, step in model.steps
            ])
            estimator = model.steps[-1][1]
        else:
            estimator = model

        print("ESTIMATOR:", type(estimator).__name__)
        print("N FEATURES:", getattr(
            estimator, "n_features_in_", "Not available"
        ))
        print("C:", getattr(estimator, "C", "Not applicable"))
        print("CLASSES:", getattr(
            estimator, "classes_", "Not available"
        ))

        if hasattr(model, "feature_names_in_"):
            print("FEATURE NAMES:", list(
                model.feature_names_in_
            )[:10])

    except Exception as e:
        print("ERROR:", repr(e))


In [ ]:

from pathlib import Path
import joblib

base = Path("/content/drive/MyDrive/MASLD_Validation_Backup")

files = [
    base / "v3_full_results/models/final_binary_full.joblib",
    base / "v3_full_results/models/final_binary_quick.joblib",
    base / "v3_quick_final/models/final_binary_quick.joblib",
    base / "baseline_v2/models/final_binary.joblib"
]

for path in files:
    print("\n" + "=" * 75)
    print("FILE:", path.name)
    print("LOCATION:", path.parent)

    if not path.exists():
        print("File not found")
        continue

    data = joblib.load(path)

    print("\nDICTIONARY KEYS:")
    for key, value in data.items():
        print(f"  {key}: {type(value).__name__}")

        if hasattr(value, "steps"):
            print("    PIPELINE STEPS:")
            for name, step in value.steps:
                print(f"      {name}: {type(step).__name__}")

        if hasattr(value, "n_features_in_"):
            print("    N FEATURES:", value.n_features_in_)

        if hasattr(value, "C"):
            print("    C:", value.C)


In [ ]:

from pathlib import Path

root = Path("/content/drive/MyDrive")

keywords = [
    "v4", "frozen", "logistic", "final",
    "manifest", "classifier", "model", "50gene"
]

extensions = {".joblib", ".pkl", ".pickle", ".json"}

matches = []

for path in root.rglob("*"):
    if not path.is_file():
        continue

    name = path.name.lower()
    if path.suffix.lower() in extensions and any(
        word in name or word in str(path.parent).lower()
        for word in keywords
    ):
        matches.append(path)

print("FOUND:", len(matches), "candidate files")

for path in matches:
    print(path)


In [ ]:

from pathlib import Path
from sklearn.linear_model import LogisticRegression
import joblib

root = Path("/content/drive/MyDrive")

for path in root.rglob("*"):
    if not path.is_file() or path.suffix.lower() not in {
        ".joblib", ".pkl", ".pickle"
    }:
        continue

    try:
        obj = joblib.load(path)

        if isinstance(obj, dict):
            model = obj.get("model", obj.get("pipeline"))
        else:
            model = obj

        if model is None:
            continue

        if hasattr(model, "steps"):
            classifiers = [
                step for _, step in model.steps
                if isinstance(step, LogisticRegression)
            ]
        elif isinstance(model, LogisticRegression):
            classifiers = [model]
        else:
            classifiers = []

        for clf in classifiers:
            print("\nPOSSIBLE LOGISTIC MODEL")
            print("FILE:", path)
            print("C:", clf.C)
            print("SOLVER:", clf.solver)
            print("MAX ITER:", clf.max_iter)
            print(
                "FEATURES:",
                getattr(model, "n_features_in_", "Unknown")
            )

    except Exception:
        continue


In [ ]:

from pathlib import Path

root = Path("/content/drive/MyDrive")

keywords = [
    "v4", "final", "frozen", "rank",
    "harmon", "training", "manifest",
    "selected", "feature", "external",
    "validation", "development"
]

extensions = {
    ".csv", ".parquet", ".feather",
    ".npy", ".npz", ".json",
    ".py", ".ipynb"
}

matches = []

for path in root.rglob("*"):
    if not path.is_file():
        continue

    name = path.name.lower()

    if path.suffix.lower() in extensions:
        if any(word in name for word in keywords):
            matches.append(path)

print("FOUND:", len(matches), "candidate files\n")

for path in sorted(matches):
    print(path)


In [ ]:

from pathlib import Path

root = Path("/content/drive/MyDrive/MASLD_Validation_Backup")

folders = [
    root / "baseline_v2/gene_level",
    root / "v3_experiments/data",
    root / "v3_full_results",
    root / "common_gene_ranking"
]

for folder in folders:
    print("\n" + "=" * 75)
    print("FOLDER:", folder)

    if not folder.exists():
        print("NOT FOUND")
        continue

    files = sorted(p for p in folder.rglob("*") if p.is_file())

    print("TOTAL FILES:", len(files))

    for file in files[:100]:
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"{size_mb:8.2f} MB  {file.relative_to(folder)}")


In [ ]:

import pandas as pd
from pathlib import Path

root = Path(
    "/content/drive/MyDrive/MASLD_Validation_Backup/"
    "baseline_v2/gene_level"
)

for gse in ["GSE48452", "GSE126848", "GSE135251"]:
    X = pd.read_csv(
        root / f"{gse}_common_rank.csv.gz",
        index_col=0
    )
    y = pd.read_csv(root / f"{gse}_labels.csv")

    print(f"\n{gse}")
    print("Expression shape:", X.shape)
    print("First sample IDs:", X.index[:3].tolist())
    print("First gene names:", X.columns[:5].tolist())
    print("Label columns:", y.columns.tolist())
    print(y.head(3).to_string(index=False))


In [ ]:

from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount("/content/drive")

root = Path(
    "/content/drive/MyDrive/MASLD_Validation_Backup/"
    "baseline_v2/gene_level"
)

datasets = ["GSE48452", "GSE126848", "GSE135251"]

all_X = []
all_y = []

for gse in datasets:
    X = pd.read_csv(
        root / f"{gse}_common_rank.csv.gz",
        index_col=0
    ).T

    labels = pd.read_csv(
        root / f"{gse}_labels.csv"
    ).set_index("Unnamed: 0")

    assert X.index.is_unique, f"Duplicate samples: {gse}"
    assert labels.index.is_unique, f"Duplicate labels: {gse}"
    assert set(X.index) == set(labels.index), (
        f"Sample mismatch: {gse}"
    )

    y = labels.loc[X.index, "label_binary"].astype(int)

    assert y.isin([0, 1]).all()
    assert X.columns.is_unique

    print(f"\n{gse}")
    print("Samples x genes:", X.shape)
    print("Healthy:", (y == 0).sum())
    print("Disease:", (y == 1).sum())
    print("Missing expression values:", X.isna().sum().sum())

    all_X.append(X)
    all_y.append(y)

X_all = pd.concat(all_X)
y_all = pd.concat(all_y)

assert X_all.shape == (346, 10152)
assert X_all.columns.is_unique
assert not X_all.isna().any().any()

print("\nFINAL DEVELOPMENT DATA")
print("Shape:", X_all.shape)
print("Healthy:", (y_all == 0).sum())
print("Disease:", (y_all == 1).sum())
print("Gene names:", X_all.columns[:10].tolist())

# Check for the original final-model manifest
manifest_files = list(
    Path("/content/drive/MyDrive").rglob("FINAL_manifest.json")
)

print("\nFinal model manifests found:")
for p in manifest_files:
    print(p)

if not manifest_files:
    print("Upload FINAL_manifest.json from your supervisor ZIP.")


In [ ]:

from pathlib import Path
import hashlib

EXPECTED_SHA = (
    "74df2a653c9f23e7c9035cd335638685abf8f61e19bd707227d8a681cc66ff25"
)

roots = [
    Path("/content/ML_Project"),
    Path("/content/drive/MyDrive"),
]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

found = []

for root in roots:
    if not root.exists():
        continue

    for path in root.rglob("binary_candidate_v1.joblib"):
        digest = sha256_file(path)
        match = digest == EXPECTED_SHA

        print("\nFILE:", path)
        print("SHA256:", digest)
        print("EXACT MATCH:", match)

        found.append((path, match))

print("\nTOTAL CANDIDATES:", len(found))
print("VERIFIED ORIGINALS:", sum(match for _, match in found))

if not found:
    print("Original frozen model not found in the searched folders.")


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

root = Path("/content/drive/MyDrive/MASLD_Validation_Backup")
datasets = ["GSE48452", "GSE126848", "GSE135251"]

# Exact ordered genes from your uploaded FINAL_manifest.json
selected_genes = [
    "VIL1", "P4HA1", "CYP7A1", "SPP1", "IL32",
    "CHI3L1", "FNDC5", "CXCL10", "CYP4F22", "FAT1",
    "CXCL9", "TP53I3", "ME1", "ISM1", "NIPAL1",
    "IGF1", "SLC16A1", "FABP4", "MCM6", "TYMS",
    "SQLE", "PLSCR4", "LGALS4", "PRKCE", "THY1",
    "ZFP1", "SOCS2", "MVP", "PPM1K", "FSTL3",
    "APOL3", "PDE11A", "MCM4", "GAS6", "COL1A1",
    "RUNDC3B", "LRRC31", "FCAMR", "MAP1LC3A",
    "DNAJC12", "AKR1B10", "AQP1", "RPS6KA1",
    "ANGPT2", "DTNA", "MVD", "TNFRSF12A",
    "TREH", "SLC19A1", "EGFLAM"
]

assert len(selected_genes) == 50
assert len(set(selected_genes)) == 50

rows = []

for gse in datasets:
    v2_path = (
        root / "baseline_v2/gene_level"
        / f"{gse}_common_rank.csv.gz"
    )
    v3_path = (
        root / "v3_experiments/data"
        / f"{gse}_common_rank.csv.gz"
    )

    v2 = pd.read_csv(v2_path, index_col=0)
    v3 = pd.read_csv(v3_path, index_col=0)

    common_rows = v2.index.intersection(v3.index)
    common_cols = v2.columns.intersection(v3.columns)

    a = v2.loc[common_rows, common_cols].to_numpy()
    b = v3.loc[common_rows, common_cols].to_numpy()

    rows.append({
        "Dataset": gse,
        "V2 shape": str(v2.shape),
        "V3 shape": str(v3.shape),
        "Matching genes": len(common_rows),
        "Matching samples": len(common_cols),
        "V2 selected genes present": sum(
            g in v2.index for g in selected_genes
        ),
        "V3 selected genes present": sum(
            g in v3.index for g in selected_genes
        ),
        "Maximum absolute difference": float(
            np.max(np.abs(a - b))
        ) if a.size else np.nan,
        "All shared values identical": bool(
            np.array_equal(a, b)
        ) if a.size else False
    })

comparison = pd.DataFrame(rows)
display(comparison)


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import json
import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

root = Path("/content/drive/MyDrive/MASLD_Validation_Backup")

output = Path(
    "/content/drive/MyDrive/MASLD_Project_Supervisor/"
    "04_Trained_Model"
)
output.mkdir(parents=True, exist_ok=True)

datasets = ["GSE48452", "GSE126848", "GSE135251"]

assert len(selected_genes) == 50
assert len(set(selected_genes)) == 50

def load_development(version):
    folder = (
        root / "baseline_v2/gene_level"
        if version == "V2"
        else root / "v3_experiments/data"
    )

    matrices = []
    labels_list = []

    for gse in datasets:
        X = pd.read_csv(
            folder / f"{gse}_common_rank.csv.gz",
            index_col=0
        ).T

        labels = pd.read_csv(
            root / "baseline_v2/gene_level"
            / f"{gse}_labels.csv"
        ).set_index("Unnamed: 0")

        assert set(X.index) == set(labels.index)
        assert all(g in X.columns for g in selected_genes)

        X = X.loc[:, selected_genes].astype(float)
        y = labels.loc[X.index, "label_binary"].astype(int)

        matrices.append(X)
        labels_list.append(y)

    X_all = pd.concat(matrices)
    y_all = pd.concat(labels_list)

    assert X_all.shape == (346, 50)
    assert int((y_all == 0).sum()) == 77
    assert int((y_all == 1).sum()) == 269
    assert np.isfinite(X_all.to_numpy()).all()

    return X_all, y_all

for version in ["V2", "V3"]:
    X, y = load_development(version)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=0.1,
            solver="lbfgs",
            max_iter=3000,
            class_weight="balanced"
        ))
    ])

    model.fit(X, y)

    artifact = {
        "model": model,
        "selected_genes": list(selected_genes),
        "class_mapping": {"Healthy": 0, "Disease": 1},
        "threshold": 0.5,
        "development_datasets": datasets,
        "n_training_samples": len(X),
        "training_version": version,
        "provenance": "RECONSTRUCTED - NOT ORIGINAL FROZEN MODEL",
        "training_utc": datetime.now(timezone.utc).isoformat(),
        "sklearn_version": sklearn.__version__
    }

    filename = (
        f"binary_50gene_RECONSTRUCTED_{version}.joblib"
    )

    save_path = output / filename
    joblib.dump(artifact, save_path)

    print("\nVERSION:", version)
    print("Saved:", save_path)
    print("Training accuracy:", model.score(X, y))
    print("Coefficients:", model.named_steps[
        "classifier"
    ].coef_.shape)
    print("Genes:", len(artifact["selected_genes"]))

print("\nBoth reconstructed candidates have been saved.")


In [ ]:

from pathlib import Path

root = Path("/content/drive/MyDrive")

keywords = [
    "GSE89632",
    "GSE130970",
    "external_validation",
    "frozen_candidate"
]

extensions = {
    ".csv", ".gz", ".tsv", ".txt",
    ".parquet", ".feather", ".npy",
    ".npz", ".pkl", ".joblib",
    ".json", ".py", ".ipynb"
}

matches = []

for path in root.rglob("*"):
    if not path.is_file():
        continue

    path_text = str(path).lower()

    if (
        path.suffix.lower() in extensions
        and any(k.lower() in path_text for k in keywords)
    ):
        matches.append(path)

print("TOTAL MATCHING FILES:", len(matches))

for path in sorted(matches):
    print(
        f"{path.stat().st_size / 1024**2:7.2f} MB",
        path
    )


In [ ]:

from pathlib import Path

roots = [
    Path("/content/ML_Project"),
    Path("/content/external_validation_v4"),
    Path("/content")
]

extensions = {
    ".csv", ".gz", ".tsv", ".txt", ".parquet",
    ".npy", ".npz", ".pkl", ".joblib", ".json"
}

seen = set()
matches = []

for root in roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if str(path) in seen:
            continue

        seen.add(str(path))

        if any(
            term in str(path).lower()
            for term in [
                "gse89632",
                "gse130970",
                "external_validation_v4",
                "frozen_candidate"
            ]
        ):
            if path.suffix.lower() in extensions:
                matches.append(path)

print("FILES FOUND:", len(matches))

for path in sorted(matches):
    print(
        f"{path.stat().st_size / 1024**2:.2f} MB",
        path
    )


## 10 — External GEO data and artifact collection


In [ ]:

from google.colab import drive
from pathlib import Path
import GEOparse
import pandas as pd

drive.mount("/content/drive")

base = Path(
    "/content/drive/MyDrive/MASLD_Project_Supervisor/"
    "03_Input_Data/External"
)
base.mkdir(parents=True, exist_ok=True)

for accession in ["GSE89632", "GSE130970"]:
    print("\n" + "=" * 65)
    print("DOWNLOADING GEO:", accession)

    target = base / accession
    target.mkdir(parents=True, exist_ok=True)

    gse = GEOparse.get_GEO(
        geo=accession,
        destdir=str(target),
        silent=False
    )

    print("Samples:", len(gse.gsms))
    print("Platforms:", list(gse.gpls.keys()))

    sample_rows = []

    for gsm_id, gsm in gse.gsms.items():
        sample_rows.append({
            "sample_id": gsm_id,
            "title": gsm.metadata.get("title", [""])[0],
            "source": gsm.metadata.get(
                "source_name_ch1", [""]
            )[0],
            "characteristics": " | ".join(
                gsm.metadata.get("characteristics_ch1", [])
            )
        })

    metadata = pd.DataFrame(sample_rows)

    metadata.to_csv(
        target / f"{accession}_raw_metadata.csv",
        index=False
    )

    print("Metadata saved:", len(metadata), "samples")

    if gse.gsms:
        first_gsm = next(iter(gse.gsms.values()))
        print(
            "First sample expression table:",
            first_gsm.table.shape
        )

print("\nGEO retrieval completed.")


In [ ]:

%pip install -q GEOparse


In [ ]:

import GEOparse

print("GEOparse installed successfully!")


In [ ]:
from google.colab import drive
from pathlib import Path
import GEOparse
import pandas as pd

In [ ]:

import GEOparse
from pathlib import Path
import pandas as pd

base = Path(
    "/content/drive/MyDrive/MASLD_Project_Supervisor/"
    "03_Input_Data/External"
)
base.mkdir(parents=True, exist_ok=True)

for accession in ["GSE89632", "GSE130970"]:
    print("\nDownloading:", accession)

    folder = base / accession
    folder.mkdir(parents=True, exist_ok=True)

    gse = GEOparse.get_GEO(
        geo=accession,
        destdir=str(folder),
        silent=False
    )

    print("GEO:", accession)
    print("Number of samples:", len(gse.gsms))
    print("Platforms:", list(gse.gpls.keys()))

    rows = []
    for gsm_id, gsm in gse.gsms.items():
        rows.append({
            "sample_id": gsm_id,
            "title": gsm.metadata.get("title", [""])[0],
            "characteristics": " | ".join(
                gsm.metadata.get("characteristics_ch1", [])
            )
        })

    pd.DataFrame(rows).to_csv(
        folder / f"{accession}_raw_metadata.csv",
        index=False
    )

    print("Metadata saved successfully.")

print("\nBoth GEO download attempts completed.")


In [ ]:

from pathlib import Path

base = Path(
    "/content/drive/MyDrive/MASLD_Project_Supervisor/"
    "03_Input_Data/External"
)

for accession in ["GSE89632", "GSE130970"]:
    folder = base / accession

    print("\n" + "=" * 60)
    print("DATASET:", accession)

    if not folder.exists():
        print("Folder not found.")
        continue

    for file in sorted(folder.rglob("*")):
        if file.is_file():
            size_mb = file.stat().st_size / (1024 ** 2)
            print(f"{size_mb:.2f} MB — {file.name}")


In [ ]:

import GEOparse
from pathlib import Path
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

base = Path(
    "/content/drive/MyDrive/MASLD_Project_Supervisor/"
    "03_Input_Data/External"
)

# PART A: GSE89632 microarray expression
print("=" * 65)
print("GSE89632 MICROARRAY EXPRESSION")

soft_file = base / "GSE89632/GSE89632_family.soft.gz"

gse = GEOparse.get_GEO(
    filepath=str(soft_file),
    silent=True
)

first_id = next(iter(gse.gsms))
first_sample = gse.gsms[first_id]

print("Samples:", len(gse.gsms))
print("First sample:", first_id)
print("Expression table shape:", first_sample.table.shape)
print("Expression columns:", first_sample.table.columns.tolist())
print("\nFirst five rows:")
print(first_sample.table.head().to_string(index=False))

print("\nPlatform annotation:")
for name, platform in gse.gpls.items():
    print("Platform:", name)
    print("Annotation columns:", platform.table.columns.tolist())
    print(platform.table.head(3).to_string(index=False))

# PART B: GSE130970 RNA-seq supplementary files
print("\n" + "=" * 65)
print("GSE130970 RNA-SEQ SUPPLEMENTARY FILES")

url = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/"
    "GSE130nnn/GSE130970/suppl/"
)

try:
    response = requests.get(url, timeout=40)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.endswith("/") or href.startswith("?"):
            continue
        links.append(urljoin(url, href))

    print("Supplementary files found:", len(links))

    for link in links:
        print(link)

except Exception as exc:
    print("Directory listing unavailable:", exc)
    print("Open this directory manually:")
    print(url)


In [ ]:

from pathlib import Path
import requests

folder = Path(
    "/content/drive/MyDrive/MASLD_Project_Supervisor/"
    "03_Input_Data/External/GSE130970"
)
folder.mkdir(parents=True, exist_ok=True)

base_url = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/"
    "GSE130nnn/GSE130970/suppl/"
)

filenames = [
    "GSE130970_all_sample_salmon_tximport_TPM_entrez_gene_ID.csv.gz",
    "GSE130970_all_sample_salmon_tximport_counts_entrez_gene_ID.csv.gz"
]

for filename in filenames:
    destination = folder / filename

    if destination.exists() and destination.stat().st_size > 0:
        print("Already downloaded:", destination.name)
        continue

    print("Downloading:", filename)

    response = requests.get(
        base_url + filename,
        stream=True,
        timeout=180
    )
    response.raise_for_status()

    with open(destination, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    print(
        "Saved:",
        destination.name,
        round(destination.stat().st_size / 1024**2, 2),
        "MB"
    )

print("\nDownload complete.")


In [ ]:

from pathlib import Path
import pandas as pd
import GEOparse

base = Path(
    "/content/drive/MyDrive/MASLD_Project_Supervisor/"
    "03_Input_Data/External"
)

# GSE89632: inspect probe-to-gene mapping
print("=" * 65)
print("GSE89632 MICROARRAY")

soft_path = base / "GSE89632/GSE89632_family.soft.gz"

gse = GEOparse.get_GEO(
    filepath=str(soft_path),
    silent=True
)

platform = next(iter(gse.gpls.values())).table

annotation = platform[["ID", "Symbol"]].copy()
annotation["Symbol"] = annotation["Symbol"].fillna("").astype(str).str.strip()

print("Samples:", len(gse.gsms))
print("Total platform probes:", len(annotation))
print("Probes with symbols:", (annotation["Symbol"] != "").sum())
print("Unique nonempty symbols:",
      annotation.loc[annotation["Symbol"] != "", "Symbol"].nunique())

print("\nExample annotation:")
print(annotation.head(10).to_string(index=False))

# GSE130970: inspect original RNA-seq matrices
print("\n" + "=" * 65)
print("GSE130970 RNA-SEQ")

folder = base / "GSE130970"

for suffix in ["TPM", "counts"]:
    filename = (
        "GSE130970_all_sample_salmon_tximport_"
        f"{suffix}_entrez_gene_ID.csv.gz"
    )
    path = folder / filename

    if not path.exists():
        print("MISSING:", filename)
        continue

    df = pd.read_csv(path, index_col=0)

    print(f"\n{suffix} matrix")
    print("Shape:", df.shape)
    print("First row identifiers:", df.index[:5].tolist())
    print("First column identifiers:", df.columns[:5].tolist())
    print("Missing values:", int(df.isna().sum().sum()))
    print(df.iloc[:3, :4].to_string())

# Check external metadata columns
print("\n" + "=" * 65)
print("EXTERNAL SAMPLE METADATA")

for accession in ["GSE89632", "GSE130970"]:
    path = base / accession / f"{accession}_raw_metadata.csv"

    metadata = pd.read_csv(path)

    print("\n", accession)
    print("Rows:", len(metadata))
    print("Columns:", metadata.columns.tolist())
    print(metadata.head(5).to_string(index=False))


In [ ]:
# Run in Google Colab after mounting Google Drive. This COPIES files; it does not train models.
# !pip install -q pandas   # only if missing
from google.colab import drive, files
from pathlib import Path
import shutil, zipfile, hashlib, json

drive.mount('/content/drive')
my = Path('/content/drive/MyDrive')
base = my/'MASLD_Project_Supervisor'
backup = my/'MASLD_Validation_Backup'
out = Path('/content/MASLD_GDrive_Research_Inputs_And_Reconstructed_Models')
out.mkdir(parents=True, exist_ok=True)
copy_log=[]

def copy_one(source, destination_relative):
    target=out/destination_relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if source.is_file():
        shutil.copy2(source, target)
        h=hashlib.sha256()
        with target.open('rb') as f:
            for part in iter(lambda:f.read(1024*1024), b''):
                h.update(part)
        copy_log.append({'destination':str(destination_relative),'source':str(source),'sha256':h.hexdigest(),'bytes':target.stat().st_size})
        print('COPIED', destination_relative)
    else:
        print('MISSING (not copied):',source)

# The two fitted reconstructed classifiers, NOT the original frozen V4 file.
for version in ('V2','V3'):
    n=f'binary_50gene_RECONSTRUCTED_{version}.joblib'
    copy_one(base/'04_Trained_Model'/n, Path('04_Trained_Model')/n)

for accession in ('GSE48452','GSE126848','GSE135251'):
    v2=backup/'baseline_v2/gene_level'
    for suffix in ('_common_rank.csv.gz','_labels.csv','_metadata_harmonized.csv'):
        n=accession+suffix
        copy_one(v2/n,Path('03_Input_Data/Development_V2')/n)
    n=accession+'_common_rank.csv.gz'
    copy_one(backup/'v3_experiments/data'/n,Path('03_Input_Data/Development_V3')/n)
copy_one(backup/'baseline_v2/gene_level/common_genes.txt',Path('03_Input_Data/Development_V2/common_genes.txt'))
for accession in ('GSE89632','GSE130970'):
    folder=base/'03_Input_Data/External'/accession
    if folder.exists():
        for p in sorted(folder.iterdir()):
            if p.is_file() and p.suffix.lower() in ('.gz','.csv','.txt','.tsv'):
                copy_one(p,Path('03_Input_Data/External')/accession/p.name)

(out/'COPY_PROVENANCE.json').write_text(json.dumps(copy_log,indent=2))
zip_path=Path('/content/MASLD_GDrive_Research_Inputs_And_Reconstructed_Models.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in out.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(out.parent))
print('READY:',zip_path,'size MB:',round(zip_path.stat().st_size/1024**2,2))
files.download(str(zip_path))


In [ ]:

from google.colab import files

files.download(
    "/content/MASLD_GDrive_Research_Inputs_And_Reconstructed_Models.zip"
)


## 11 — Reconstructed-model inspection and V4 candidate


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os
import joblib

folder = "/content/drive/MyDrive/MASLD_Project_Supervisor/04_Trained_Model"

files = [
    "binary_50gene_RECONSTRUCTED_V2.joblib",
    "binary_50gene_RECONSTRUCTED_V3.joblib"
]

for filename in files:
    path = os.path.join(folder, filename)
    print("\nMODEL:", filename)

    if not os.path.exists(path):
        print("File not found")
        continue

    artifact = joblib.load(path)

    if isinstance(artifact, dict):
        print("Saved keys:", list(artifact.keys()))

        model = artifact.get("model")
        genes = artifact.get("selected_genes")

        print("Model:", model)
        print("Number of genes:", len(genes) if genes is not None else "Unknown")
    else:
        print("Saved object:", artifact)


In [ ]:

import os
import joblib

folder = "/content/drive/MyDrive/MASLD_Project_Supervisor/04_Trained_Model"

v2 = joblib.load(os.path.join(
    folder, "binary_50gene_RECONSTRUCTED_V2.joblib"
))

v3 = joblib.load(os.path.join(
    folder, "binary_50gene_RECONSTRUCTED_V3.joblib"
))

print("=== MODEL COMPARISON ===")

print("\nV2 training samples:",
      v2.get("n_training_samples"))

print("V3 training samples:",
      v3.get("n_training_samples"))

print("\nV2 threshold:", v2.get("threshold"))
print("V3 threshold:", v3.get("threshold"))

print("\nV2 class mapping:", v2.get("class_mapping"))
print("V3 class mapping:", v3.get("class_mapping"))

print("\nSame selected genes in same order:",
      list(v2["selected_genes"]) ==
      list(v3["selected_genes"]))

print("\nV2 provenance:", v2.get("provenance"))
print("V3 provenance:", v3.get("provenance"))

print("\nV2 training version:", v2.get("training_version"))
print("V3 training version:", v3.get("training_version"))


In [ ]:

# ============================================================
# MASLD PROJECT — RECONSTRUCTED V4 MODEL CREATION
# Creates a documented replacement candidate from V3.
# DOES NOT recover or reproduce the missing original V4.
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
import csv
import copy
import hashlib
import joblib
import sklearn
from datetime import datetime, timezone

# ------------------------------------------------------------
# STEP 1 — FOLDER LOCATIONS
# ------------------------------------------------------------

BASE = "/content/drive/MyDrive/MASLD_Project_Supervisor"
MODEL_DIR = os.path.join(BASE, "04_Trained_Model")

V2_PATH = os.path.join(
    MODEL_DIR, "binary_50gene_RECONSTRUCTED_V2.joblib"
)

V3_PATH = os.path.join(
    MODEL_DIR, "binary_50gene_RECONSTRUCTED_V3.joblib"
)

V4_DIR = os.path.join(
    MODEL_DIR, "V4_RECONSTRUCTED"
)

os.makedirs(V4_DIR, exist_ok=True)

V4_PATH = os.path.join(
    V4_DIR, "MASLD_50gene_V4_RECONSTRUCTED_CANDIDATE.joblib"
)

MANIFEST_PATH = os.path.join(
    V4_DIR, "V4_RECONSTRUCTED_manifest.json"
)

GENES_PATH = os.path.join(
    V4_DIR, "V4_selected_50_genes.csv"
)

# ------------------------------------------------------------
# STEP 2 — VERIFY INPUT FILES
# ------------------------------------------------------------

print("=" * 60)
print("MASLD — RECONSTRUCTED V4 MODEL CREATION")
print("=" * 60)

for path in [V2_PATH, V3_PATH]:
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Required model file is missing:\n{path}"
        )

print("\nBoth reconstructed model files found.")

# IMPORTANT:
# Only load joblib files that you created or trust.

v2 = joblib.load(V2_PATH)
v3 = joblib.load(V3_PATH)

# ------------------------------------------------------------
# STEP 3 — VERIFY MODEL STRUCTURE
# ------------------------------------------------------------

required_keys = [
    "model",
    "selected_genes",
    "class_mapping",
    "threshold",
    "n_training_samples"
]

for name, artifact in [("V2", v2), ("V3", v3)]:

    if not isinstance(artifact, dict):
        raise TypeError(f"{name} is not a saved artifact dictionary.")

    missing = [
        key for key in required_keys
        if key not in artifact
    ]

    if missing:
        raise ValueError(
            f"{name} is missing required keys: {missing}"
        )

genes_v2 = list(v2["selected_genes"])
genes_v3 = list(v3["selected_genes"])

if len(genes_v3) != 50:
    raise ValueError(
        f"Expected 50 genes, found {len(genes_v3)}."
    )

if len(set(genes_v3)) != 50:
    raise ValueError("Duplicate genes detected in V3.")

same_genes = (genes_v2 == genes_v3)

print("\nMODEL COMPARISON")
print("-" * 60)

print("V2 training samples:", v2["n_training_samples"])
print("V3 training samples:", v3["n_training_samples"])

print("V2 threshold:", v2["threshold"])
print("V3 threshold:", v3["threshold"])

print("Same genes in the same order:", same_genes)

print("\nV2 training version:",
      v2.get("training_version"))

print("V3 training version:",
      v3.get("training_version"))

if not same_genes:
    raise ValueError(
        "V2 and V3 gene lists differ. "
        "Manual investigation is required before proceeding."
    )

if int(v3["n_training_samples"]) != 346:
    raise ValueError(
        "Unexpected training sample count. "
        "Expected 346 samples."
    )

if float(v3["threshold"]) != 0.5:
    raise ValueError(
        "Unexpected classification threshold."
    )

pipeline = v3["model"]

if not hasattr(pipeline, "named_steps"):
    raise TypeError("V3 model is not a scikit-learn pipeline.")

if "scaler" not in pipeline.named_steps:
    raise ValueError("StandardScaler step not found.")

if "classifier" not in pipeline.named_steps:
    raise ValueError("Classifier step not found.")

scaler = pipeline.named_steps["scaler"]
classifier = pipeline.named_steps["classifier"]

if not isinstance(scaler, sklearn.preprocessing.StandardScaler):
    raise TypeError("Unexpected scaler type.")

if not isinstance(
    classifier, sklearn.linear_model.LogisticRegression
):
    raise TypeError("Unexpected classifier type.")

if getattr(classifier, "n_features_in_", None) != 50:
    raise ValueError(
        "Fitted classifier does not expect exactly 50 features."
    )

if abs(classifier.C - 0.1) > 1e-12:
    raise ValueError("Unexpected logistic regression C value.")

if classifier.class_weight != "balanced":
    raise ValueError("Unexpected class weighting.")

if sorted(classifier.classes_.tolist()) != [0, 1]:
    raise ValueError(
        "Expected binary model classes [0, 1]."
    )

if not hasattr(classifier, "coef_"):
    raise ValueError("Classifier does not appear to be fitted.")

print("\nV3 model verification: PASSED")
print("Algorithm: Logistic Regression")
print("Input genes:", len(genes_v3))
print("C parameter:", classifier.C)
print("Class weights:", classifier.class_weight)
print("Threshold:", v3["threshold"])

# ------------------------------------------------------------
# STEP 4 — CREATE RECONSTRUCTED V4 CANDIDATE
# ------------------------------------------------------------
# This preserves the existing fitted V3 parameters.
# It does not retrain the original model or verify its
# equivalence to the missing original V4.

v4 = copy.deepcopy(v3)

created_utc = datetime.now(timezone.utc).isoformat()

v4["training_version"] = "V4_RECONSTRUCTED_CANDIDATE"

v4["provenance"] = {
    "source_model": os.path.basename(V3_PATH),
    "source_training_version": v3.get("training_version"),
    "original_v4_model_recovered": False,
    "original_v4_equivalence_verified": False,
    "external_validation_completed_for_this_artifact": False,
    "new_fit_performed": False,
    "method": "Preserved fitted V3 pipeline as a "
              "separately named V4 reconstruction candidate",
    "note": (
        "This artifact is not the original frozen V4 model. "
        "Previously archived external validation metrics must "
        "not be attributed to this replacement unless they are "
        "independently reproduced and verified."
    )
}

v4["v4_candidate_created_utc"] = created_utc

# ------------------------------------------------------------
# STEP 5 — SAVE RECONSTRUCTED V4 MODEL
# ------------------------------------------------------------

joblib.dump(v4, V4_PATH, compress=3)

# Verify saved file can be loaded successfully
check = joblib.load(V4_PATH)

assert len(check["selected_genes"]) == 50
assert check["training_version"] == "V4_RECONSTRUCTED_CANDIDATE"

# Check that fitted parameters are unchanged
import numpy as np

np.testing.assert_array_equal(
    check["model"].named_steps["classifier"].coef_,
    classifier.coef_
)

np.testing.assert_array_equal(
    check["model"].named_steps["classifier"].intercept_,
    classifier.intercept_
)

# ------------------------------------------------------------
# STEP 6 — SAVE THE 50 SELECTED GENES
# ------------------------------------------------------------

with open(GENES_PATH, "w", newline="") as f:

    writer = csv.writer(f)
    writer.writerow(["feature_order", "gene_symbol"])

    for i, gene in enumerate(genes_v3, start=1):
        writer.writerow([i, gene])

# ------------------------------------------------------------
# STEP 7 — SAVE MODEL INFORMATION
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()

manifest = {
    "project": "MASLD Transcriptomic Machine Learning",
    "artifact_name": "V4_RECONSTRUCTED_CANDIDATE",
    "created_utc": created_utc,
    "model_filename": os.path.basename(V4_PATH),
    "sha256": sha256_file(V4_PATH),
    "model_type": "StandardScaler + LogisticRegression",
    "number_of_genes": 50,
    "number_of_training_samples": v4["n_training_samples"],
    "selected_genes": genes_v3,
    "class_mapping": v4["class_mapping"],
    "threshold": v4["threshold"],
    "classifier_parameters": {
        "C": classifier.C,
        "class_weight": classifier.class_weight,
        "max_iter": classifier.max_iter,
        "solver": classifier.solver
    },
    "source_model": os.path.basename(V3_PATH),
    "source_provenance": v3.get("provenance"),
    "original_v4_recovered": False,
    "external_validation_status": "NOT YET PERFORMED",
    "original_reported_external_metrics_transferable": False,
    "sklearn_version": sklearn.__version__
}

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=4, default=str)

# ------------------------------------------------------------
# STEP 8 — FINAL RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SUCCESS — RECONSTRUCTED V4 CANDIDATE SAVED")
print("=" * 60)

print("\nV4 MODEL:")
print(V4_PATH)

print("\n50-GENE LIST:")
print(GENES_PATH)

print("\nMODEL MANIFEST:")
print(MANIFEST_PATH)

print("\nNumber of selected genes:", len(genes_v3))

print("Model:", check["model"])

print("\nIMPORTANT SCIENTIFIC STATUS:")
print("Original frozen V4 recovered: NO")
print("Reconstructed V4 candidate created: YES")
print("Fitted V3 parameters preserved: YES")
print("New external validation performed: NO")

print("\nNEXT REQUIRED STEP:")
print(
    "Evaluate this saved candidate on independently "
    "preprocessed and correctly labeled GSE89632 "
    "and GSE130970 samples."
)

print("\nDONE.")
